# Figure 3I: transcription factor expression along nine tree segments

Read the accepted tree and statistics saved by `04_Trajectory_dot_plot-2.ipynb`. The original Early/Late grouping, observed-expression means, curated rows and display scale are retained. This file draws only panel I; it does not rebuild the tree or expression models.

Replace each `ROOT` placeholder with the required input, output, font or temporary path before running. The accepted H5 contains the original AnimalTFDB reference table. No encoded data block is stored in this code.


In [ ]:
#1. =========== Input data and output paths
from pathlib import Path
import ast, hashlib, io, json, os, pickle, shutil, subprocess, sys, tempfile, warnings
import h5py
import numpy as np
import pandas as pd

# Replace each ROOT placeholder with the required file or directory path before running.
ROOT = Path(".")
# Set the file path. Uses tree_results-1.pkl generated by the accepted grafted-tree analysis in 04_tree-1.ipynb.
BUNDLE = ROOT
# Set the file path. Uses Trajectory dot plot_results-2.h5 generated by 04_Trajectory_dot_plot-2.ipynb.
RESULT = ROOT
# Set the figure output directory.
FIGURE_DIR = ROOT
# Set the temporary parent directory. Generated run directories are managed automatically.
SCRATCH = ROOT
NAME_STEM = "Fig3I_Trajectory_dot_plot"
PANEL = "I"
EXPECTED_R_SOURCE_SHA256 = "879df4707195749dfb2aa79d77d65cc754bc6ba25c3bfee9c4ac45091c39ded7"
sys.dont_write_bytecode = True
os.environ["PYTHONDONTWRITEBYTECODE"] = "1"
# Set the Matplotlib cache directory.
os.environ["MPLCONFIGDIR"] = str(ROOT)

# AnimalTFDB mouse TSV archived by TOME on 2021-10-16.
# Original columns: Species, Symbol, Ensembl, Family, Protein, Entrez ID.
# The accepted result already stores these exact TSV bytes; no additional TF file is needed.
with h5py.File(RESULT, "r") as _reference:
    TF_BYTES = _reference["reference/tf_snapshot_tsv"][:].tobytes()
assert hashlib.sha256(TF_BYTES).hexdigest() == "c09a702a433584eb8db30024b3712feaa69672dbdedf7a68489a1e719f884dd3"
TF_SET = set(pd.read_csv(io.BytesIO(TF_BYTES), sep="\t").Symbol.astype(str))
assert len(TF_SET) == 1623


In [ ]:
# 2. =========== Tree and expression contracts
from pathlib import Path

import hashlib

import json

import pickle

import sys

import warnings

import h5py

import numpy as np

import pandas as pd

from scipy import sparse

sys.dont_write_bytecode = True

N_CELLS, N_GENES = 7355, 11467

SEGMENTS = ['CVG', 'VGN', 'T', 'AC', 'Ib', 'Ic', 'AB', 'Ia', 'II']

SEGMENT_EDGES = {'CVG': (159, 154), 'VGN': (154, 142), 'T': (154, 35),
                 'AC': (35, 91), 'Ib': (91, 106), 'Ic': (91, 81),
                 'AB': (35, 87), 'Ia': (87, 60), 'II': (87, 38)}

EXPECTED_COUNTS = {'CVG': 120, 'VGN': 175, 'T': 2037, 'AC': 904, 'Ib': 693,
                   'Ic': 226, 'AB': 1705, 'Ia': 967, 'II': 528}

FIXED_PATHS = {'VGN': ['CVG', 'VGN'], 'Ib': ['CVG', 'T', 'AC', 'Ib'],
               'Ic': ['CVG', 'T', 'AC', 'Ic'], 'Ia': ['CVG', 'T', 'AB', 'Ia'],
               'II': ['CVG', 'T', 'AB', 'II']}

RNA_SHA256 = '8625e64dfb80313661f2258c1363e63ebe3e6f5446c4e2da3c920e46137e1e2c'

def sequence_hash(value):
    text = json.dumps([str(v) for v in value], ensure_ascii=False, separators=(',', ':'))
    return hashlib.sha256(text.encode('utf-8')).hexdigest()


In [ ]:
# 3. =========== Read and verify accepted statistics
import hashlib

import io

import json

import os

from pathlib import Path

import pickle

import subprocess

import tempfile

import warnings

import h5py

import numpy as np

import pandas as pd

RESULT_SCHEMA = 'Trajectory_dot_plot_v2_9segments'

TREE_CORE_SCHEMA = 'CVG-nine-natural-segments-core-v1'

def runtime_file_hash(path, algorithm='sha256'):
    value = hashlib.new(algorithm)
    with Path(path).open('rb') as stream:
        for block in iter(lambda: stream.read(4 * 1024 * 1024), b''):
            value.update(block)
    return value.hexdigest()

def canonical_tree_value(value):
    if isinstance(value, pd.DataFrame):
        return {'kind': 'DataFrame', 'index': canonical_tree_value(value.index),
                'columns': canonical_tree_value(value.columns),
                'series': [canonical_tree_value(value.iloc[:, i]) for i in range(value.shape[1])]}
    if isinstance(value, pd.Series):
        result = {'kind': 'Series', 'name': canonical_tree_value(value.name), 'dtype': str(value.dtype)}
        if isinstance(value.dtype, pd.CategoricalDtype):
            result.update(categories=canonical_tree_value(value.cat.categories),
                          ordered=bool(value.cat.ordered), codes=canonical_tree_value(value.cat.codes.to_numpy()))
        else:
            result['values'] = canonical_tree_value(value.to_numpy())
        return result
    if isinstance(value, pd.Index):
        return {'kind': 'Index', 'name': canonical_tree_value(value.name), 'dtype': str(value.dtype),
                'values': canonical_tree_value(value.to_numpy())}
    if isinstance(value, np.ndarray):
        result = {'kind': 'ndarray', 'dtype': value.dtype.str, 'shape': list(value.shape)}
        if value.dtype.hasobject:
            result['values'] = [canonical_tree_value(v) for v in value.ravel(order='C')]
        else:
            result['bytes_sha256'] = hashlib.sha256(np.ascontiguousarray(value).tobytes()).hexdigest()
        return result
    if isinstance(value, np.generic):
        return {'kind': 'numpy-scalar', 'dtype': value.dtype.str, 'bytes': value.tobytes().hex()}
    if isinstance(value, dict):
        pairs = [[canonical_tree_value(k), canonical_tree_value(v)] for k, v in value.items()]
        pairs.sort(key=lambda item: json.dumps(item[0], sort_keys=True, ensure_ascii=False, separators=(',', ':')))
        return {'kind': 'dict', 'items': pairs}
    if isinstance(value, (list, tuple)):
        return {'kind': type(value).__name__, 'items': [canonical_tree_value(v) for v in value]}
    if isinstance(value, float):
        return {'kind': 'float', 'hex': value.hex()}
    if value is None or isinstance(value, (str, int, bool)):
        return {'kind': type(value).__name__, 'value': value}
    raise TypeError('Unsupported fixed-tree value: ' + type(value).__name__)

def fixed_tree_core_hash(tree):
    projection_keys = ('root', 'edges', 'edge_id', 'alpha', 't', 'root_distance', 'parent', 'residual')
    core = dict(obs=tree['obs'], F=tree['model']['F'], B=tree['model']['B'],
                parameters=tree['model']['parameters'],
                projection={key: tree['projection'][key] for key in projection_keys},
                segments=[{key: segment[key] for key in ('id', 'start', 'end', 'path', 'edge_ids', 'role', 'label')}
                          for segment in tree['segments']],
                natural_branches=[{key: branch[key] for key in ('id', 'start', 'end', 'path', 'edge_ids')}
                                  for branch in tree['publication_display']['branches']],
                local_candidate_calls=tree['local_candidate_calls'],
                projection_component=tree['projection_component'])
    payload = dict(schema=TREE_CORE_SCHEMA, core=canonical_tree_value(core))
    text = json.dumps(payload, sort_keys=True, ensure_ascii=False, separators=(',', ':')).encode('utf-8')
    return hashlib.sha256(text).hexdigest()

def read_fixed_tree():
    with warnings.catch_warnings(), Path(BUNDLE).open('rb') as stream:
        warnings.simplefilter('ignore', DeprecationWarning)
        tree = pickle.load(stream)
    assert len(tree['obs']) == N_CELLS and tree['projection']['root'] == 159
    assert tree['model']['B'].shape == (160, 160)
    return tree

def read_table(group):
    columns = json.loads(group.attrs['columns'])
    values = {column: (group[column].asstr()[:] if h5py.check_string_dtype(group[column].dtype)
                       else group[column][:]) for column in columns}
    return pd.DataFrame(values, index=pd.Index(group['_index'].asstr()[:], name=group.attrs['index_name'] or None))

def requested_genes():
    names = set()
    for panel in (SELECTION_H, SELECTION_I):
        assert set(panel) == set(SEGMENTS)
        for segment in SEGMENTS:
            assert set(panel[segment]) == {'up', 'down'}
            names.update(panel[segment]['up'])
            names.update(panel[segment]['down'])
    assert names, 'No displayed genes were requested.'
    return sorted(names)

def fixed_grouping(cells):
    indices, extremes, counts, midpoints, limits = [], [], [], [], []
    names = []
    for segment in SEGMENTS:
        positions = np.flatnonzero(cells.seg.eq(segment).to_numpy())
        t = cells.iloc[positions].t.to_numpy(dtype=float)
        assert len(positions) == EXPECTED_COUNTS[segment] and np.isfinite(t).all()
        low, high = float(t.min()), float(t.max())
        midpoint = (low + high) / 2
        early, late = positions[t <= midpoint], positions[t > midpoint]
        assert len(early) > 0 and len(late) > 0
        indices.extend([early, late])
        counts.extend([len(early), len(late)])
        extremes.append([int(positions[t.argmin()]), int(positions[t.argmax()])])
        midpoints.append(midpoint)
        limits.append([low, high])
        names.extend([segment + '_early', segment + '_late'])
    return dict(indices=indices, extremes=np.asarray(extremes, dtype=np.int64),
                counts=np.asarray(counts, dtype=np.int64), names=np.asarray(names, dtype=str),
                midpoints=np.asarray(midpoints), limits=np.asarray(limits))

def automatic_orders(symbols, changes):
    up, down = {}, {}
    symbols = np.asarray(symbols, dtype=str)
    for column, segment in enumerate(SEGMENTS):
        delta = changes[:, column]
        positive, negative = np.flatnonzero(delta > 0), np.flatnonzero(delta < 0)
        up[segment] = positive[np.lexsort((symbols[positive], -delta[positive]))]
        down[segment] = negative[np.lexsort((symbols[negative], delta[negative]))]
    return up, down

def validate_association(stats, genes, fitmeta):
    assert len(stats) == len(genes) == N_GENES
    assert np.array_equal(stats.gene.astype(str), genes.gene_symbol.astype(str))
    assert np.array_equal(stats.gene_id.astype(str), genes.gene_id.astype(str))
    assert np.array_equal(stats.gene_index.to_numpy(), np.arange(N_GENES))
    assert pd.api.types.is_bool_dtype(stats.signi)
    numeric = ['p_val', 'p_adj', 'A', 'df_num', 'df_den', 'null_deviance', 'segment_deviance']
    assert np.isfinite(stats[numeric].to_numpy(dtype=float)).all()
    assert stats.p_val.between(0, 1).all() and stats.p_adj.between(0, 1).all()
    assert (stats.A >= 0).all() and (stats.df_num > 0).all() and (stats.df_den > 0).all()
    corrected = np.minimum(stats.p_val.to_numpy() * N_GENES, 1.0)
    np.testing.assert_allclose(stats.p_adj, corrected, rtol=1e-11, atol=1e-300)
    expected = (stats.p_adj < 1e-4) & (stats.A > 1)
    assert np.array_equal(stats.signi.to_numpy(), expected.to_numpy())
    assert np.array_equal(stats.st.to_numpy(), expected.astype(int).to_numpy())
    assert fitmeta['status'] == 'complete'
    a, fit, tree = fitmeta['association'], fitmeta['fit'], fitmeta['fixed_tree']
    assert a['n_genes'] == a['family_size'] == N_GENES and a['n_map'] == 1
    assert a['correction'] == 'bonferroni' and a['spline_df'] == 5
    assert a['adjusted_p_strictly_less_than'] == 1e-4
    assert a['amplitude_strictly_greater_than'] == 1 and a['stability_strictly_greater_than'] == .8
    assert a['significant_genes'] == int(expected.sum()) and a['failed_genes'] == 0
    assert fit['gamma'] == 1.5 and fit['successful_genes'] == int(expected.sum()) and fit['failed_genes'] == 0
    assert tree['n_cells'] == N_CELLS and tree['root'] == 159
    assert {key: tuple(value) for key, value in tree['edges'].items()} == SEGMENT_EDGES
    assert tree['paths'] == FIXED_PATHS and tree['segment_counts'] == EXPECTED_COUNTS
    assert stats.loc[stats.signi, 'fit_status'].eq('ok').all()

def validate_tree_cells(tree, cells):
    assert cells.index.equals(tree['obs'].index) and cells.index.is_unique
    for column in tree['obs']:
        a, b = cells[column], tree['obs'][column]
        if pd.api.types.is_numeric_dtype(b.dtype):
            np.testing.assert_array_equal(a.to_numpy(), b.to_numpy())
        else:
            np.testing.assert_array_equal(a.astype(object).fillna('').astype(str),
                                          b.astype(object).fillna('').astype(str))
    np.testing.assert_array_equal(cells.t.to_numpy(), tree['projection']['t'])
    names = {tuple(pair): name for name, pair in SEGMENT_EDGES.items()}
    edge_names = np.full(len(tree['projection']['edges']), '', dtype=object)
    covered = np.zeros(len(edge_names), dtype=int)
    for branch in tree['publication_display']['branches']:
        name = names[(int(branch['start']), int(branch['end']))]
        edge_names[branch['edge_ids']] = name
        covered[branch['edge_ids']] += 1
    assert np.all(covered == 1)
    np.testing.assert_array_equal(cells.seg.to_numpy(), edge_names[tree['projection']['edge_id']])
    assert cells.seg.value_counts().to_dict() == EXPECTED_COUNTS
    if 'candidate_call' in cells:
        expected = tree['local_candidate_calls'].reindex(cells.index).fillna('Not assessed in local 478')
        np.testing.assert_array_equal(cells.candidate_call.astype(str), expected.astype(str))

def validate_compact_data(data):
    cells, genes, stats, candidate = (data[key] for key in ('cells', 'genes', 'stats', 'candidate'))
    validate_association(stats, genes, data['fitmeta'])
    assert len(cells) == N_CELLS and cells.seg.value_counts().to_dict() == EXPECTED_COUNTS
    requested = list(data['requested'])
    assert len(requested) == len(set(requested)) and len(requested) > 0
    assert data['expression'].shape == data['fitted'].shape == (N_CELLS, len(requested))
    assert np.isfinite(data['expression']).all() and np.all(data['expression'] >= 0)
    assert np.isfinite(data['fitted']).all()
    accepted = stats.loc[stats.signi]
    count = len(accepted)
    assert np.array_equal(candidate['gene_symbols'], accepted.gene.astype(str))
    assert np.array_equal(candidate['gene_ids'], accepted.gene_id.astype(str))
    assert np.array_equal(candidate['gene_index'], accepted.gene_index)
    assert np.array_equal(candidate['is_tf'], np.isin(candidate['gene_symbols'], list(TF_SET)))
    assert set(requested).issubset(set(candidate['gene_symbols']))
    for key in ('fitted_start', 'fitted_end', 'fitted_change'):
        assert candidate[key].shape == (count, len(SEGMENTS)) and np.isfinite(candidate[key]).all()
    np.testing.assert_array_equal(candidate['fitted_change'], candidate['fitted_end'] - candidate['fitted_start'])
    assert candidate['mean_expression'].shape == (count, 2 * len(SEGMENTS))
    assert np.isfinite(candidate['mean_expression']).all() and np.all(candidate['mean_expression'] >= 0)
    assert candidate['fit_amplitude'].shape == (count,) and np.isfinite(candidate['fit_amplitude']).all()
    assert np.all(candidate['fit_amplitude'] >= 0)
    assert np.array_equal(candidate['segment_order'], SEGMENTS)
    grouping = fixed_grouping(cells)
    for key, reference in [('group_names', 'names'), ('group_cell_counts', 'counts'),
                           ('extreme_cell_index', 'extremes'), ('segment_midpoints', 'midpoints'),
                           ('segment_t_limits', 'limits')]:
        np.testing.assert_array_equal(candidate[key], grouping[reference])
    up, down = automatic_orders(candidate['gene_symbols'], candidate['fitted_change'])
    for direction, expected in [('up_order', up), ('down_order', down)]:
        for segment in SEGMENTS:
            np.testing.assert_array_equal(candidate[direction][segment], expected[segment])
    columns = pd.Index(candidate['gene_symbols']).get_indexer(requested)
    for segment_index, (first, last) in enumerate(grouping['extremes']):
        np.testing.assert_array_equal(data['fitted'][first], candidate['fitted_start'][columns, segment_index])
        np.testing.assert_array_equal(data['fitted'][last], candidate['fitted_end'][columns, segment_index])
    np.testing.assert_array_equal(np.ptp(data['fitted'], axis=0), candidate['fit_amplitude'][columns])
    expected_means = np.asarray([data['expression'][group].mean(axis=0) for group in grouping['indices']]).T
    np.testing.assert_allclose(expected_means, candidate['mean_expression'][columns], rtol=0, atol=1e-14)

def read_candidate(group):
    result = dict(group.attrs)
    for key, value in group.items():
        if isinstance(value, h5py.Group):
            result[key] = {segment: value[segment][:] for segment in SEGMENTS}
        elif h5py.check_string_dtype(value.dtype):
            result[key] = value.asstr()[:]
        else:
            result[key] = value[:]
    return result

def load_results():
    with h5py.File(RESULT, 'r') as store:
        assert store.attrs['schema'] == RESULT_SCHEMA and bool(store.attrs['complete'])
        assert store.attrs['cells'] == N_CELLS and store.attrs['common_genes'] == N_GENES
        assert store.attrs['natural_segments'] == 9 and store.attrs['root_to_tip_paths'] == 5
        assert store.attrs['tree_core_schema'] == TREE_CORE_SCHEMA
        assert store.attrs['tf_sha256'] == hashlib.sha256(TF_BYTES).hexdigest()
        assert store['reference/tf_snapshot_tsv'][:].tobytes() == TF_BYTES
        data = {key: read_table(store[key]) for key in ('cells', 'genes', 'association')}
        data['stats'] = data.pop('association').reset_index(drop=True)
        data['requested'] = list(store['selected/gene_symbols'].asstr()[:])
        expected_indices = pd.Index(data['genes'].gene_symbol).get_indexer(data['requested'])
        np.testing.assert_array_equal(store['selected/gene_index'][:], expected_indices)
        data['expression'], data['fitted'] = store['selected/expression'][:], store['selected/fitted'][:]
        assert np.all(store['selected/fit_ok'][:] == 1)
        for key in ('library_total', 'common_axis_total'):
            np.testing.assert_array_equal(store['normalization/' + key][:], data['cells'][key])
        for key in ('input', 'fit', 'plot'):
            data[key + 'meta'] = json.loads(store['meta/' + key].asstr()[()])
        data['candidate'] = read_candidate(store['candidate'])
        assert store.attrs['stored_fitted_genes'] == len(data['requested'])
        assert store.attrs['significant_genes'] == len(data['candidate']['gene_symbols'])
        assert store.attrs['significant_tf_genes'] == int(data['candidate']['is_tf'].sum())
        assert store.attrs['tree_core_sha256'] == data['inputmeta']['tree_core_sha256']
        stored_core, stored_container = str(store.attrs['tree_core_sha256']), str(store.attrs['tree_sha256'])
    validate_compact_data(data)
    tree = read_fixed_tree()
    assert fixed_tree_core_hash(tree) == stored_core, 'Accepted model, projection, annotations or natural branches changed.'
    validate_tree_cells(tree, data['cells'])
    assert data['inputmeta']['tree_core_schema'] == TREE_CORE_SCHEMA
    assert sequence_hash(data['cells'].index) == data['inputmeta']['cell_ids_sha256']
    assert sequence_hash(data['genes'].gene_id) == data['inputmeta']['gene_ids_sha256']
    assert sequence_hash(data['genes'].gene_symbol) == data['inputmeta']['gene_symbols_sha256']
    assert data['inputmeta']['source_h5ad_sha256'] == RNA_SHA256
    expected_r_hash = EXPECTED_R_SOURCE_SHA256
    assert data['fitmeta']['R_source_sha256_lf'] == expected_r_hash, 'Embedded R engine changed; rebuild explicitly.'
    assert np.isfinite(data['cells'].library_total).all() and (data['cells'].library_total > 0).all()
    assert (data['cells'].library_total >= data['cells'].common_axis_total).all()
    if runtime_file_hash(BUNDLE) != stored_container:
        print('Tree container metadata changed; the explicitly frozen analysis core remains identical.')
    missing = set(requested_genes()) - set(data['requested'])
    assert not missing, 'Displayed genes missing from compact per-cell cache; use REBUILD_ANALYSIS=True: ' + ', '.join(sorted(missing))
    return data


In [ ]:
# 4. =========== Curated gene rows and evidence
SELECTION_H = {'CVG': {'up': ['Isl1', 'Shox2', 'Elavl3', 'Fgf10'], 'down': ['Sox2', 'Id2', 'Ptn', 'Ccnd2']}, 'VGN': {'up': ['Tbx3', 'Unc5c', 'Nrg1', 'Nav2'], 'down': ['Bdnf', 'Gas1', 'Dact1', 'Fn1']}, 'T': {'up': ['Epha5', 'Ntrk3', 'Ntm', 'Stmn2'], 'down': ['Neurod1', 'Insm1', 'Mdk', 'Ccnd1']}, 'AC': {'up': ['Mgat4c', 'Adcy1', 'Cacna2d3', 'Aldh1a3'], 'down': ['Dio3', 'Sobp', 'Calb1', 'Igfbp2']}, 'Ib': {'up': ['Calb2', 'Igfbpl1', 'Nrcam', 'Dmxl2'], 'down': ['Lypd1', 'Ntng1', 'Bhlhe22', 'Rbfox3']}, 'Ic': {'up': ['Gata3', 'Igfbpl1', 'Gas1', 'Fxyd6'], 'down': ['Nefh', 'Slc17a6', 'Rgs10', 'Rbfox3']}, 'AB': {'up': ['Islr2', 'Unc5d', 'Tmem132e', 'Cntn5'], 'down': ['Lypd1', 'Ebf1', 'Kcna1', 'Sema3a']}, 'Ia': {'up': ['Calb2', 'Cntn5', 'Cntnap5a', 'Nefh'], 'down': ['Cacng5', 'Tmem132e', 'Tgfb2', 'Cplx2']}, 'II': {'up': ['Tac1', 'Th', 'Gpr22', 'Ngfr'], 'down': ['Sema3e', 'Fgf10', 'Kcnip4', 'Epha4']}}

SELECTION_I = {'CVG': {'up': ['Isl1', 'Shox2', 'Insm1', 'Nhlh2'], 'down': ['Sox2', 'Id2', 'Hmga2', 'Rest']}, 'VGN': {'up': ['Tbx3', 'Ebf3', 'Klf7', 'Zfp536'], 'down': ['Sox2', 'Gata3', 'Tbx2', 'E2f1']}, 'T': {'up': ['Gata3', 'Myt1l', 'Thra', 'Esrrg'], 'down': ['Neurod1', 'Insm1', 'Hes6', 'Zbtb18']}, 'AC': {'up': ['Mafb', 'Prox1', 'Thrb', 'Casz1'], 'down': ['Id2', 'Etv4', 'Sox11', 'Irx1']}, 'Ib': {'up': ['Rxrg', 'Prrx2', 'Pou4f2', 'Meis3'], 'down': ['Ebf1', 'Bhlhe22', 'Shox2', 'Bcl11a']}, 'Ic': {'up': ['Id1', 'Maf', 'Tshz2', 'Jarid2'], 'down': ['Bcl11a', 'Bhlhe22', 'Mycn', 'Jun']}, 'AB': {'up': ['Pbx1', 'Smad9', 'Tshz3', 'Zfhx3'], 'down': ['Etv4', 'Ebf1', 'Sox11', 'Zeb2']}, 'Ia': {'up': ['Prrx2', 'Prox1', 'Thra'], 'down': ['Gata3', 'Nhlh2', 'Arid1a', 'Zfp536', 'Thrb']}, 'II': {'up': ['Etv4', 'Prrx2', 'Cbfb', 'Hey1'], 'down': ['Esrrg', 'Zfhx4', 'Casz1', 'Mef2c']}}

RED_I = {'Neurod1', 'Maf', 'Gata3', 'Isl1', 'Etv4', 'Mafb', 'Rxrg', 'Sox2', 'Prox1', 'Shox2', 'Id1', 'Tbx3', 'Insm1'}

NAMES = {'CVG': 'Undiff.\nCVG', 'VGN': 'CVG to\nVGN', 'T': 'CVG to\nUns.', 'AC': 'Ib/Ic\nshared', 'Ib': 'Ib-related\nend', 'Ic': 'Ic-related\nend', 'AB': 'Ia/II\nshared', 'Ia': 'Ia-related\nend', 'II': 'II-related\nend'}

EDGES = SEGMENT_EDGES

EVIDENCE = {'Isl1': {'evidence_scope': 'Mouse embryonic CVG; genetic and chromatin evidence', 'biological_role': 'CVG neuronal differentiation, migration, axon pathfinding and survival', 'primary_sources': 'https://pmc.ncbi.nlm.nih.gov/articles/PMC10883659/; https://pmc.ncbi.nlm.nih.gov/articles/PMC4051200/', 'uniprot_url': '', 'red_recommended': True, 'highlight_basis': 'Established early CVG regulatory reference', 'limitations': 'Does not establish SGN/VGN branch choice or current trajectory causality', 'selection_context': 'H/CVG/up: rank=3, fitted_delta=0.97447, observed_delta=1.5697, local_fraction=0.4563; I/CVG/up: rank=2, fitted_delta=0.97447, observed_delta=1.5697, local_fraction=0.4563', 'selection_reason': 'Significant whole-tree association; fitted and observed local directions agree; selected to display cvg neuronal differentiation, migration, axon pathfinding and survival. The stated protein mechanism motivates follow-up, not a causal claim from the dot plot.', 'direction_interpretation': 'Current fixed-tree local expression direction; pooled age/branch expression in literature is not automatically comparable', 'review_date': '2026-10-22', 'regulatory_class': 'Source-catalogue transcription factor/regulator', 'trajectory_label_scope': 'Undiff. CVG labels the early mixed common segment; it does not demonstrate uncommitted/bipotent cells'}, 'Shox2': {'evidence_scope': 'Mouse E9.5-E13.5 CVG and later SGN/VGN; expression and lineage tracing', 'biological_role': 'Early otic neuronal-lineage and differentiation-state reference', 'primary_sources': 'https://doi.org/10.1016/j.celrep.2022.110542; https://pmc.ncbi.nlm.nih.gov/articles/PMC7299348/', 'uniprot_url': '', 'red_recommended': True, 'highlight_basis': 'Established CVG/neuronal-state reference', 'limitations': 'Expressed in both SGN and VGN; not a VGN-specific fate determinant', 'selection_context': 'H/CVG/up: rank=1, fitted_delta=1.1962, observed_delta=1.1296, local_fraction=0.4345; I/CVG/up: rank=1, fitted_delta=1.1962, observed_delta=1.1296, local_fraction=0.4345; I/Ib/down: rank=6, fitted_delta=-0.36049, observed_delta=-0.36405, local_fraction=0.131', 'selection_reason': 'Significant whole-tree association; fitted and observed local directions agree; selected to display early otic neuronal-lineage and differentiation-state reference. The stated protein mechanism motivates follow-up, not a causal claim from the dot plot.', 'direction_interpretation': 'Current fixed-tree local expression direction; pooled age/branch expression in literature is not automatically comparable', 'review_date': '2026-10-22', 'regulatory_class': 'Source-catalogue transcription factor/regulator', 'trajectory_label_scope': 'Undiff. CVG labels the early mixed common segment; it does not demonstrate uncommitted/bipotent cells'}, 'Elavl3': {'evidence_scope': 'Developing mouse SGN; pan-neuronal expression', 'biological_role': 'Neuronal RNA-binding protein and neuronal-state reference', 'primary_sources': 'https://www.nature.com/articles/s41467-022-31580-1', 'uniprot_url': 'https://www.uniprot.org/uniprotkb/Q60900/entry', 'red_recommended': False, 'highlight_basis': 'Pan-neuronal RNA-binding reference, retained black; not a specific otic fate marker', 'limitations': 'No gene-specific CVG/VGN fate perturbation established here', 'selection_context': 'H/CVG/up: rank=2, fitted_delta=1.0542, observed_delta=1.2349, local_fraction=0.4365', 'selection_reason': 'Significant whole-tree association; fitted and observed local directions agree; selected to display neuronal rna-binding protein and neuronal-state reference. The stated protein mechanism motivates follow-up, not a causal claim from the dot plot.', 'direction_interpretation': 'Current fixed-tree local expression direction; pooled age/branch expression in literature is not automatically comparable', 'review_date': '2026-10-22', 'regulatory_class': 'H-panel gene; molecular role stated separately', 'trajectory_label_scope': 'Undiff. CVG labels the early mixed common segment; it does not demonstrate uncommitted/bipotent cells'}, 'Fgf10': {'evidence_scope': 'Mouse E10.5-E11.5 otic/CVG expression; CVG chromatin evidence', 'biological_role': 'Otic neuronal-lineage reference and ISL1/POU4F1-regulated CVG effector', 'primary_sources': 'https://pmc.ncbi.nlm.nih.gov/articles/PMC10592509/; https://pmc.ncbi.nlm.nih.gov/articles/PMC10883659/', 'uniprot_url': '', 'red_recommended': True, 'highlight_basis': 'Early otic/CVG lineage reference', 'limitations': 'Also functions in epithelial development; current CVG RNA change is not a causal experiment', 'selection_context': 'H/CVG/up: rank=66, fitted_delta=0.3323, observed_delta=0.84563, local_fraction=0.2019; H/II/down: rank=2, fitted_delta=-1.1835, observed_delta=-0.65285, local_fraction=0.7191', 'selection_reason': 'Significant whole-tree association; fitted and observed local directions agree; selected to display otic neuronal-lineage reference and isl1/pou4f1-regulated cvg effector. The stated protein mechanism motivates follow-up, not a causal claim from the dot plot.', 'direction_interpretation': 'Current fixed-tree local expression direction; pooled age/branch expression in literature is not automatically comparable', 'review_date': '2026-10-22', 'regulatory_class': 'H-panel gene; molecular role stated separately', 'trajectory_label_scope': 'Undiff. CVG labels the early mixed common segment; it does not demonstrate uncommitted/bipotent cells'}, 'Sox2': {'evidence_scope': 'Mouse early otic progenitors; expression and timed genetic studies', 'biological_role': 'Neurosensory competence and early progenitor-state reference', 'primary_sources': 'https://pmc.ncbi.nlm.nih.gov/articles/PMC5481345/; https://doi.org/10.1016/j.celrep.2022.110542; https://www.sciencedirect.com/science/article/pii/S0012160619303367', 'uniprot_url': '', 'red_recommended': True, 'highlight_basis': 'Established early progenitor-state regulator', 'limitations': 'Requirement depends on timing/model; not proof of uncommitted or bipotent CVG cells', 'selection_context': 'H/CVG/down: rank=11, fitted_delta=-0.55365, observed_delta=-0.43515, local_fraction=0.3096; I/CVG/down: rank=3, fitted_delta=-0.55365, observed_delta=-0.43515, local_fraction=0.3096; I/VGN/down: rank=5, fitted_delta=-0.53563, observed_delta=-0.52554, local_fraction=0.2996', 'selection_reason': 'Significant whole-tree association; fitted and observed local directions agree; selected to display neurosensory competence and early progenitor-state reference. The stated protein mechanism motivates follow-up, not a causal claim from the dot plot.', 'direction_interpretation': 'Current fixed-tree local expression direction; pooled age/branch expression in literature is not automatically comparable', 'review_date': '2026-10-22', 'regulatory_class': 'Source-catalogue transcription factor/regulator', 'trajectory_label_scope': 'Undiff. CVG labels the early mixed common segment; it does not demonstrate uncommitted/bipotent cells; CVG to VGN denotes the VGN-candidate-enriched arm, not validated VGN identity for every cell'}, 'Id2': {'evidence_scope': 'Mouse E14.5/E16.5 spiral ganglion expression', 'biological_role': 'Inhibitor of DNA-binding/bHLH regulation; early-state candidate', 'primary_sources': 'https://pmc.ncbi.nlm.nih.gov/articles/PMC6674393/', 'uniprot_url': '', 'red_recommended': False, 'highlight_basis': 'Expression plus plausible differentiation-regulatory mechanism', 'limitations': 'Functional manipulation in the cited study mainly concerns hair cells; not CVG fate validation', 'selection_context': 'H/CVG/down: rank=24, fitted_delta=-0.45555, observed_delta=-0.82041, local_fraction=0.1877; I/CVG/down: rank=6, fitted_delta=-0.45555, observed_delta=-0.82041, local_fraction=0.1877; I/AC/down: rank=2, fitted_delta=-1.1126, observed_delta=-0.33111, local_fraction=0.4585', 'selection_reason': 'Significant whole-tree association; fitted and observed local directions agree; selected to display inhibitor of dna-binding/bhlh regulation; early-state candidate. The stated protein mechanism motivates follow-up, not a causal claim from the dot plot.', 'direction_interpretation': 'Current fixed-tree local expression direction; pooled age/branch expression in literature is not automatically comparable', 'review_date': '2026-10-22', 'regulatory_class': 'Non-DNA-binding HLH inhibitor of bHLH transcription factors', 'trajectory_label_scope': 'Undiff. CVG labels the early mixed common segment; it does not demonstrate uncommitted/bipotent cells'}, 'Ptn': {'evidence_scope': 'Postnatal mouse cochlea; SGN culture functional evidence', 'biological_role': 'Secreted trophic factor promoting SGN number and neurite extension in vitro', 'primary_sources': 'https://pubmed.ncbi.nlm.nih.gov/31515588/', 'uniprot_url': '', 'red_recommended': False, 'highlight_basis': 'Direct neuronal trophic mechanism', 'limitations': 'No matching embryonic CVG time-direction result or SGN/VGN identity specificity', 'selection_context': 'H/CVG/down: rank=4, fitted_delta=-0.68108, observed_delta=-0.80753, local_fraction=0.3447', 'selection_reason': 'Significant whole-tree association; fitted and observed local directions agree; selected to display secreted trophic factor promoting sgn number and neurite extension in vitro. The stated protein mechanism motivates follow-up, not a causal claim from the dot plot.', 'direction_interpretation': 'Current fixed-tree local expression direction; pooled age/branch expression in literature is not automatically comparable', 'review_date': '2026-10-22', 'regulatory_class': 'H-panel gene; molecular role stated separately', 'trajectory_label_scope': 'Undiff. CVG labels the early mixed common segment; it does not demonstrate uncommitted/bipotent cells'}, 'Ccnd2': {'evidence_scope': 'General mouse cell-cycle protein annotation', 'biological_role': 'Cyclin-D2/CDK regulation of G1/S progression', 'primary_sources': '', 'uniprot_url': 'https://www.uniprot.org/uniprotkb/P30280/entry', 'red_recommended': False, 'highlight_basis': 'Cell-cycle-state candidate', 'limitations': 'Not a CVG identity marker; phase/composition can alter transcript abundance', 'selection_context': 'H/CVG/down: rank=2, fitted_delta=-0.77792, observed_delta=-1.152, local_fraction=0.4289', 'selection_reason': 'Significant whole-tree association; fitted and observed local directions agree; selected to display cyclin-d2/cdk regulation of g1/s progression. The stated protein mechanism motivates follow-up, not a causal claim from the dot plot.', 'direction_interpretation': 'Current fixed-tree local expression direction; pooled age/branch expression in literature is not automatically comparable', 'review_date': '2026-10-22', 'regulatory_class': 'H-panel gene; molecular role stated separately', 'trajectory_label_scope': 'Undiff. CVG labels the early mixed common segment; it does not demonstrate uncommitted/bipotent cells'}, 'Tbx3': {'evidence_scope': 'Mouse early VG vs SG; expression profiling and in situ hybridization', 'biological_role': 'Vestibular-enriched transcription-factor reference', 'primary_sources': 'https://pmc.ncbi.nlm.nih.gov/articles/PMC3167573/; https://shield.hms.harvard.edu/references/Lu_Goodrich.pdf', 'uniprot_url': '', 'red_recommended': True, 'highlight_basis': 'Direct VG-enriched in situ reference, Fig.7C/C-prime', 'limitations': 'Enrichment does not establish a causal VGN fate function or validate the whole candidate arm', 'selection_context': 'H/VGN/up: rank=13, fitted_delta=0.96558, observed_delta=0.90982, local_fraction=0.6014; I/VGN/up: rank=1, fitted_delta=0.96558, observed_delta=0.90982, local_fraction=0.6014', 'selection_reason': 'Significant whole-tree association; fitted and observed local directions agree; selected to display vestibular-enriched transcription-factor reference. The stated protein mechanism motivates follow-up, not a causal claim from the dot plot.', 'direction_interpretation': 'Current fixed-tree local expression direction; pooled age/branch expression in literature is not automatically comparable', 'review_date': '2026-10-22', 'regulatory_class': 'Source-catalogue transcription factor/regulator', 'trajectory_label_scope': 'CVG to VGN denotes the VGN-candidate-enriched arm, not validated VGN identity for every cell'}, 'Unc5c': {'evidence_scope': 'Mouse embryonic VG/SG in situ and limited mutant analysis', 'biological_role': 'UNC5-family guidance receptor candidate', 'primary_sources': 'https://ijdb.ehu.eus/article/pdf/072273tm', 'uniprot_url': '', 'red_recommended': False, 'highlight_basis': 'Mechanistically interpretable candidate; no automatic red for any ear association', 'limitations': 'Small mutant sample lacked obvious ear morphology defect; no demonstrated VGN-specific fate or guidance mechanism', 'selection_context': 'H/VGN/up: rank=3, fitted_delta=1.267, observed_delta=0.66818, local_fraction=0.7001', 'selection_reason': 'Significant whole-tree association; fitted and observed local directions agree; selected to display unc5-family guidance receptor candidate. The stated protein mechanism motivates follow-up, not a causal claim from the dot plot.', 'direction_interpretation': 'Current fixed-tree local expression direction; pooled age/branch expression in literature is not automatically comparable', 'review_date': '2026-10-22', 'regulatory_class': 'H-panel gene; molecular role stated separately', 'trajectory_label_scope': 'CVG to VGN denotes the VGN-candidate-enriched arm, not validated VGN identity for every cell'}, 'Nrg1': {'evidence_scope': 'Mouse embryonic SGN deletion and adult cochlear neuron-support-cell signaling; mouse postnatal vestibular supporting-cell signaling/synapse experiments', 'biological_role': 'ERBB ligand; neuronal survival and neuron-glia/support-cell communication', 'primary_sources': 'https://pmc.ncbi.nlm.nih.gov/articles/PMC2172983/; https://pmc.ncbi.nlm.nih.gov/articles/PMC6729966/; https://pmc.ncbi.nlm.nih.gov/articles/PMC2947909/', 'uniprot_url': 'https://www.uniprot.org/uniprotkb/Q6DR98/entry', 'red_recommended': False, 'highlight_basis': 'Mechanistically interpretable candidate; no automatic red for any ear association', 'limitations': 'Known SGN and isoform-specific functions do not prove VGN specificity; transcript assay does not resolve all NRG1 isoforms Vestibular erbB/BDNF synapse evidence is postnatal and mainly supporting-cell perturbation, not embryonic VGN fate choice.', 'selection_context': 'H/VGN/up: rank=12, fitted_delta=0.99085, observed_delta=0.72307, local_fraction=0.5177', 'selection_reason': 'Significant whole-tree association; fitted and observed local directions agree; selected to display erbb ligand; neuronal survival and neuron-glia/support-cell communication. The stated protein mechanism motivates follow-up, not a causal claim from the dot plot.', 'direction_interpretation': 'Current fixed-tree local expression direction; pooled age/branch expression in literature is not automatically comparable', 'review_date': '2026-10-22', 'regulatory_class': 'H-panel gene; molecular role stated separately', 'trajectory_label_scope': 'CVG to VGN denotes the VGN-candidate-enriched arm, not validated VGN identity for every cell'}, 'Nav2': {'evidence_scope': 'Mouse/general neuronal development; curated protein annotation', 'biological_role': 'Cytoskeletal organization, neuronal migration and neurite-extension candidate', 'primary_sources': '', 'uniprot_url': 'https://www.uniprot.org/uniprotkb/V9GX91/entry', 'red_recommended': False, 'highlight_basis': 'Mechanistic neuronal-growth candidate', 'limitations': 'Several protein functions are annotated by similarity; no direct VGN-specific function established', 'selection_context': 'H/VGN/up: rank=16, fitted_delta=0.926, observed_delta=0.79238, local_fraction=0.6015', 'selection_reason': 'Significant whole-tree association; fitted and observed local directions agree; selected to display cytoskeletal organization, neuronal migration and neurite-extension candidate. The stated protein mechanism motivates follow-up, not a causal claim from the dot plot.', 'direction_interpretation': 'Current fixed-tree local expression direction; pooled age/branch expression in literature is not automatically comparable', 'review_date': '2026-10-22', 'regulatory_class': 'H-panel gene; molecular role stated separately', 'trajectory_label_scope': 'CVG to VGN denotes the VGN-candidate-enriched arm, not validated VGN identity for every cell'}, 'Bdnf': {'evidence_scope': 'Mouse embryonic/postnatal CVG; culture and knockout', 'biological_role': 'Neurotrophin supporting vestibular-neuron survival and target innervation', 'primary_sources': 'https://pubmed.ncbi.nlm.nih.gov/8139657/; https://pubmed.ncbi.nlm.nih.gov/8010414/; https://pmc.ncbi.nlm.nih.gov/articles/PMC2711896/', 'uniprot_url': '', 'red_recommended': True, 'highlight_basis': 'Established inner-ear neuronal trophic reference', 'limitations': 'Target-derived and exogenous trophic functions do not require VGN-intrinsic Bdnf RNA to rise', 'selection_context': 'H/VGN/down: rank=1, fitted_delta=-1.0581, observed_delta=-0.91282, local_fraction=0.496', 'selection_reason': 'Significant whole-tree association; fitted and observed local directions agree; selected to display neurotrophin supporting vestibular-neuron survival and target innervation. The stated protein mechanism motivates follow-up, not a causal claim from the dot plot.', 'direction_interpretation': 'Current fixed-tree local expression direction; pooled age/branch expression in literature is not automatically comparable', 'review_date': '2026-10-22', 'regulatory_class': 'H-panel gene; molecular role stated separately', 'trajectory_label_scope': 'CVG to VGN denotes the VGN-candidate-enriched arm, not validated VGN identity for every cell'}, 'Gas1': {'evidence_scope': 'Mouse curated protein and SHH-receptor annotation', 'biological_role': 'Growth-arrest/SHH-coreceptor signaling candidate', 'primary_sources': '', 'uniprot_url': 'https://www.uniprot.org/uniprotkb/Q01721/entry; https://www.uniprot.org/uniprotkb/Q62226/entry', 'red_recommended': False, 'highlight_basis': 'Specific signaling mechanism', 'limitations': 'No direct VGN-neuron perturbation or developmental direction established', 'selection_context': 'H/VGN/down: rank=23, fitted_delta=-0.65892, observed_delta=-0.66122, local_fraction=0.3123; H/Ic/up: rank=3, fitted_delta=0.51806, observed_delta=0.89157, local_fraction=0.2455', 'selection_reason': 'Significant whole-tree association; fitted and observed local directions agree; selected to display growth-arrest/shh-coreceptor signaling candidate. The stated protein mechanism motivates follow-up, not a causal claim from the dot plot.', 'direction_interpretation': 'Current fixed-tree local expression direction; pooled age/branch expression in literature is not automatically comparable', 'review_date': '2026-10-22', 'regulatory_class': 'H-panel gene; molecular role stated separately', 'trajectory_label_scope': 'CVG to VGN denotes the VGN-candidate-enriched arm, not validated VGN identity for every cell'}, 'Dact1': {'evidence_scope': 'Mouse forebrain neuron knockout/culture; curated protein function', 'biological_role': 'Dishevelled/Wnt-associated scaffold; dendrite and postsynaptic development', 'primary_sources': 'https://pmc.ncbi.nlm.nih.gov/articles/PMC2848693/; https://pmc.ncbi.nlm.nih.gov/articles/PMC3691262/', 'uniprot_url': 'https://www.uniprot.org/uniprotkb/Q8R4A3/entry', 'red_recommended': False, 'highlight_basis': 'Mechanistically interpretable candidate; no automatic red for any ear association', 'limitations': 'No direct VGN development function or matching temporal direction established', 'selection_context': 'H/VGN/down: rank=4, fitted_delta=-0.89451, observed_delta=-0.6547, local_fraction=0.4836', 'selection_reason': 'Significant whole-tree association; fitted and observed local directions agree; selected to display dishevelled/wnt-associated scaffold; dendrite and postsynaptic development. The stated protein mechanism motivates follow-up, not a causal claim from the dot plot.', 'direction_interpretation': 'Current fixed-tree local expression direction; pooled age/branch expression in literature is not automatically comparable', 'review_date': '2026-10-22', 'regulatory_class': 'H-panel gene; molecular role stated separately', 'trajectory_label_scope': 'CVG to VGN denotes the VGN-candidate-enriched arm, not validated VGN identity for every cell'}, 'Fn1': {'evidence_scope': 'Mixed spiral-ganglion culture/substrate experiments; broad ECM biology', 'biological_role': 'Extracellular-matrix adhesion and neurite-growth context', 'primary_sources': 'https://pmc.ncbi.nlm.nih.gov/articles/PMC2855887/', 'uniprot_url': '', 'red_recommended': False, 'highlight_basis': 'Extracellular-matrix context', 'limitations': 'Culture substrate use does not prove endogenous VGN neuronal Fn1 expression/function', 'selection_context': 'H/VGN/down: rank=21, fitted_delta=-0.66448, observed_delta=-0.71401, local_fraction=0.3591', 'selection_reason': 'Significant whole-tree association; fitted and observed local directions agree; selected to display extracellular-matrix adhesion and neurite-growth context. The stated protein mechanism motivates follow-up, not a causal claim from the dot plot.', 'direction_interpretation': 'Current fixed-tree local expression direction; pooled age/branch expression in literature is not automatically comparable', 'review_date': '2026-10-22', 'regulatory_class': 'H-panel gene; molecular role stated separately', 'trajectory_label_scope': 'CVG to VGN denotes the VGN-candidate-enriched arm, not validated VGN identity for every cell'}, 'Epha5': {'evidence_scope': 'Mouse early differentiating SGN expression and CVG regulatory targets', 'biological_role': 'Auditory-lineage expression reference; Eph receptor / axon-interaction candidate', 'primary_sources': 'https://doi.org/10.1016/j.celrep.2022.110542; https://pmc.ncbi.nlm.nih.gov/articles/PMC3167573/; https://pmc.ncbi.nlm.nih.gov/articles/PMC10883659/', 'uniprot_url': '', 'red_recommended': True, 'highlight_basis': 'Established marker/regulatory reference; evidence class stated separately', 'limitations': 'Do not confuse EphA5 receptor with ephrin-A5 ligand; current T rise does not prove branch commitment', 'selection_context': 'H/T/up: rank=12, fitted_delta=2.3302, observed_delta=1.1101, local_fraction=0.6888', 'selection_reason': 'Significant whole-tree association; fitted and observed local directions agree; selected to display auditory-lineage expression reference; eph receptor / axon-interaction candidate. The stated protein mechanism motivates follow-up, not a causal claim from the dot plot.', 'direction_interpretation': 'Current fixed-tree local expression direction; pooled age/branch expression in literature is not automatically comparable', 'review_date': '2026-10-22', 'regulatory_class': 'H-panel gene; molecular role stated separately', 'trajectory_label_scope': 'Fixed project SGN annotation/tree; local expression association'}, 'Ntrk3': {'evidence_scope': 'Mouse inner-ear sensory neurons; receptor-mutant experiments', 'biological_role': 'TrkC neurotrophin receptor supporting target innervation', 'primary_sources': 'https://pmc.ncbi.nlm.nih.gov/articles/PMC155354/', 'uniprot_url': '', 'red_recommended': True, 'highlight_basis': 'Established neuronal development/trophic reference', 'limitations': 'Not a unique SGN subtype marker; expression and receptor requirement are different evidence', 'selection_context': 'H/T/up: rank=25, fitted_delta=2.0165, observed_delta=1.4463, local_fraction=0.7957', 'selection_reason': 'Significant whole-tree association; fitted and observed local directions agree; selected to display trkc neurotrophin receptor supporting target innervation. The stated protein mechanism motivates follow-up, not a causal claim from the dot plot.', 'direction_interpretation': 'Current fixed-tree local expression direction; pooled age/branch expression in literature is not automatically comparable', 'review_date': '2026-10-22', 'regulatory_class': 'H-panel gene; molecular role stated separately', 'trajectory_label_scope': 'Fixed project SGN annotation/tree; local expression association'}, 'Ntm': {'evidence_scope': 'General mouse protein annotation', 'biological_role': 'Neural cell-adhesion molecule; neuronal connectivity candidate', 'primary_sources': '', 'uniprot_url': 'https://www.uniprot.org/uniprotkb/Q99PJ0/entry', 'red_recommended': False, 'highlight_basis': 'Mechanistically interpretable neuronal candidate', 'limitations': 'No direct early CVG/SGN/VGN-specific causal study identified', 'selection_context': 'H/T/up: rank=6, fitted_delta=2.4952, observed_delta=1.7337, local_fraction=0.8951', 'selection_reason': 'Significant whole-tree association; fitted and observed local directions agree; selected to display neural cell-adhesion molecule; neuronal connectivity candidate. The stated protein mechanism motivates follow-up, not a causal claim from the dot plot.', 'direction_interpretation': 'Current fixed-tree local expression direction; pooled age/branch expression in literature is not automatically comparable', 'review_date': '2026-10-22', 'regulatory_class': 'H-panel gene; molecular role stated separately', 'trajectory_label_scope': 'Fixed project SGN annotation/tree; local expression association'}, 'Stmn2': {'evidence_scope': 'Mouse sensory/motor neurons; microtubule and axon-outgrowth experiments', 'biological_role': 'Neuronal microtubule dynamics and axon-growth candidate', 'primary_sources': 'https://pmc.ncbi.nlm.nih.gov/articles/PMC9327139/', 'uniprot_url': 'https://www.uniprot.org/uniprotkb/P55821/entry', 'red_recommended': False, 'highlight_basis': 'Mechanistic neuronal-growth candidate', 'limitations': 'No VGN-specific developmental perturbation identified; not automatically red for neuronal expression', 'selection_context': 'H/T/up: rank=4, fitted_delta=3.0316, observed_delta=1.5681, local_fraction=0.8475', 'selection_reason': 'Significant whole-tree association; fitted and observed local directions agree; selected to display neuronal microtubule dynamics and axon-growth candidate. The stated protein mechanism motivates follow-up, not a causal claim from the dot plot.', 'direction_interpretation': 'Current fixed-tree local expression direction; pooled age/branch expression in literature is not automatically comparable', 'review_date': '2026-10-22', 'regulatory_class': 'H-panel gene; molecular role stated separately', 'trajectory_label_scope': 'Fixed project SGN annotation/tree; local expression association'}, 'Neurod1': {'evidence_scope': 'Mouse early CVG genetics and embryonic SGN diversification', 'biological_role': 'Neuronal differentiation/survival and early Ic specification regulator', 'primary_sources': 'https://pmc.ncbi.nlm.nih.gov/articles/PMC8894106/; https://www.nature.com/articles/s41467-022-31580-1', 'uniprot_url': '', 'red_recommended': True, 'highlight_basis': 'Established early neuronal/developmental regulator', 'limitations': 'RNA may fall while a later target-specific regulon remains active', 'selection_context': 'H/T/down: rank=30, fitted_delta=-2.2467, observed_delta=-0.96397, local_fraction=0.6791; I/T/down: rank=4, fitted_delta=-2.2467, observed_delta=-0.96397, local_fraction=0.6791', 'selection_reason': 'Significant whole-tree association; fitted and observed local directions agree; selected to display neuronal differentiation/survival and early ic specification regulator. The stated protein mechanism motivates follow-up, not a causal claim from the dot plot.', 'direction_interpretation': 'Current fixed-tree local expression direction; pooled age/branch expression in literature is not automatically comparable', 'review_date': '2026-10-22', 'regulatory_class': 'Source-catalogue transcription factor/regulator', 'trajectory_label_scope': 'Fixed project SGN annotation/tree; local expression association'}, 'Insm1': {'evidence_scope': 'Mouse delaminated otic progenitors; lineage tracing and knockout', 'biological_role': 'Promotes neurogenic proliferation and production of CVG neurons', 'primary_sources': 'https://pmc.ncbi.nlm.nih.gov/articles/PMC4679445/; https://pmc.ncbi.nlm.nih.gov/articles/PMC10592509/', 'uniprot_url': '', 'red_recommended': True, 'highlight_basis': 'Established early neurogenic-state regulator', 'limitations': 'Declining RNA later does not imply that its earlier function was inhibitory', 'selection_context': 'H/T/down: rank=86, fitted_delta=-1.5356, observed_delta=-0.99959, local_fraction=0.6119; I/CVG/up: rank=10, fitted_delta=0.34781, observed_delta=0.51246, local_fraction=0.1386; I/T/down: rank=11, fitted_delta=-1.5356, observed_delta=-0.99959, local_fraction=0.6119', 'selection_reason': 'Significant whole-tree association; fitted and observed local directions agree; selected to display promotes neurogenic proliferation and production of cvg neurons. The stated protein mechanism motivates follow-up, not a causal claim from the dot plot.', 'direction_interpretation': 'Current fixed-tree local expression direction; pooled age/branch expression in literature is not automatically comparable', 'review_date': '2026-10-22', 'regulatory_class': 'Source-catalogue transcription factor/regulator', 'trajectory_label_scope': 'Undiff. CVG labels the early mixed common segment; it does not demonstrate uncommitted/bipotent cells'}, 'Mdk': {'evidence_scope': 'General neuronal growth-factor evidence', 'biological_role': 'Retinoic-acid-responsive secreted growth/differentiation-factor candidate', 'primary_sources': 'https://www.sciencedirect.com/science/article/pii/S0012160683712509', 'uniprot_url': 'https://www.uniprot.org/uniprotkb/P12025/entry', 'red_recommended': False, 'highlight_basis': 'Mechanistic growth-factor candidate', 'limitations': 'Original neurite-growth evidence concerns CNS; no matched CVG fate-direction validation', 'selection_context': 'H/T/down: rank=17, fitted_delta=-2.5342, observed_delta=-2.0076, local_fraction=0.8235', 'selection_reason': 'Significant whole-tree association; fitted and observed local directions agree; selected to display retinoic-acid-responsive secreted growth/differentiation-factor candidate. The stated protein mechanism motivates follow-up, not a causal claim from the dot plot.', 'direction_interpretation': 'Current fixed-tree local expression direction; pooled age/branch expression in literature is not automatically comparable', 'review_date': '2026-10-22', 'regulatory_class': 'H-panel gene; molecular role stated separately', 'trajectory_label_scope': 'Fixed project SGN annotation/tree; local expression association'}, 'Ccnd1': {'evidence_scope': 'General mouse cell-cycle protein annotation', 'biological_role': 'Cyclin-D1/CDK control of G1/S progression', 'primary_sources': 'https://www.informatics.jax.org/sequence/P25322', 'uniprot_url': 'https://www.uniprot.org/uniprotkb/P25322/entry', 'red_recommended': False, 'highlight_basis': 'Cell-cycle-state candidate', 'limitations': 'Does not establish a specific SGN/VGN fate function', 'selection_context': 'H/T/down: rank=15, fitted_delta=-2.574, observed_delta=-1.5854, local_fraction=0.8167', 'selection_reason': 'Significant whole-tree association; fitted and observed local directions agree; selected to display cyclin-d1/cdk control of g1/s progression. The stated protein mechanism motivates follow-up, not a causal claim from the dot plot.', 'direction_interpretation': 'Current fixed-tree local expression direction; pooled age/branch expression in literature is not automatically comparable', 'review_date': '2026-10-22', 'regulatory_class': 'H-panel gene; molecular role stated separately', 'trajectory_label_scope': 'Fixed project SGN annotation/tree; local expression association'}, 'Mgat4c': {'evidence_scope': 'Mouse E16.5 emerging-Ic RNAscope expression', 'biological_role': 'Glycosyltransferase-family protein; early Ic-state expression reference', 'primary_sources': 'https://www.nature.com/articles/s41467-022-31580-1; https://pmc.ncbi.nlm.nih.gov/articles/PMC3220513/', 'uniprot_url': '', 'red_recommended': True, 'highlight_basis': 'Established marker/regulatory reference; evidence class stated separately', 'limitations': 'Marker evidence does not establish enzymatic control of Ic fate or validate new AC topology Broader glycosylation experiments are in HEK293 cells; enzyme substrate activity is not established by its SGN marker expression.', 'selection_context': 'H/AC/up: rank=953, fitted_delta=0.30325, observed_delta=0.19166, local_fraction=0.3742', 'selection_reason': 'Significant whole-tree association; fitted and observed local directions agree; selected to display glycosyltransferase-family protein; early ic-state expression reference. The stated protein mechanism motivates follow-up, not a causal claim from the dot plot. Effect size and study/source sensitivity must be read alongside rank and significance.', 'direction_interpretation': 'Current fixed-tree local expression direction; pooled age/branch expression in literature is not automatically comparable', 'review_date': '2026-10-22', 'regulatory_class': 'H-panel gene; molecular role stated separately', 'trajectory_label_scope': 'Fixed project SGN annotation/tree; local expression association'}, 'Adcy1': {'evidence_scope': 'General mouse curated protein function', 'biological_role': 'Calcium/calmodulin-sensitive cAMP synthesis; signaling/maturation candidate', 'primary_sources': '', 'uniprot_url': 'https://www.uniprot.org/uniprotkb/O88444/entry', 'red_recommended': False, 'highlight_basis': 'Mechanistically interpretable candidate; no automatic red for any ear association', 'limitations': 'No matching embryonic SGN-segment functional or temporal validation established here', 'selection_context': 'H/AC/up: rank=10, fitted_delta=1.258, observed_delta=0.88351, local_fraction=0.7152', 'selection_reason': 'Significant whole-tree association; fitted and observed local directions agree; selected to display calcium/calmodulin-sensitive camp synthesis; signaling/maturation candidate. The stated protein mechanism motivates follow-up, not a causal claim from the dot plot.', 'direction_interpretation': 'Current fixed-tree local expression direction; pooled age/branch expression in literature is not automatically comparable', 'review_date': '2026-10-22', 'regulatory_class': 'H-panel gene; molecular role stated separately', 'trajectory_label_scope': 'Fixed project SGN annotation/tree; local expression association'}, 'Cacna2d3': {'evidence_scope': 'Mouse SGN/auditory brainstem expression, knockout and electrophysiology', 'biological_role': 'Calcium-channel auxiliary subunit controlling auditory-nerve synapse structure/transmission', 'primary_sources': 'https://pmc.ncbi.nlm.nih.gov/articles/PMC6608152/', 'uniprot_url': '', 'red_recommended': False, 'highlight_basis': 'Mechanistically interpretable candidate; no automatic red for any ear association', 'limitations': 'Auditory transmission function does not validate this AC developmental direction', 'selection_context': 'H/AC/up: rank=64, fitted_delta=0.80938, observed_delta=0.79093, local_fraction=0.4787', 'selection_reason': 'Significant whole-tree association; fitted and observed local directions agree; selected to display calcium-channel auxiliary subunit controlling auditory-nerve synapse structure/transmission. The stated protein mechanism motivates follow-up, not a causal claim from the dot plot.', 'direction_interpretation': 'Current fixed-tree local expression direction; pooled age/branch expression in literature is not automatically comparable', 'review_date': '2026-10-22', 'regulatory_class': 'H-panel gene; molecular role stated separately', 'trajectory_label_scope': 'Fixed project SGN annotation/tree; local expression association'}, 'Aldh1a3': {'evidence_scope': 'Mouse embryonic ganglion expression and curated enzyme annotation', 'biological_role': 'Retinaldehyde oxidation and retinoic-acid synthesis candidate', 'primary_sources': 'https://pmc.ncbi.nlm.nih.gov/articles/PMC3167573/', 'uniprot_url': 'https://www.uniprot.org/uniprotkb/Q9JHW9/entry', 'red_recommended': False, 'highlight_basis': 'Mechanistically interpretable candidate; no automatic red for any ear association', 'limitations': 'Early VG enrichment does not make this AC change VGN-specific or establish its direction at the current ages', 'selection_context': 'H/AC/up: rank=2, fitted_delta=2.519, observed_delta=1.6725, local_fraction=0.8409', 'selection_reason': 'Significant whole-tree association; fitted and observed local directions agree; selected to display retinaldehyde oxidation and retinoic-acid synthesis candidate. The stated protein mechanism motivates follow-up, not a causal claim from the dot plot.', 'direction_interpretation': 'Current fixed-tree local expression direction; pooled age/branch expression in literature is not automatically comparable', 'review_date': '2026-10-22', 'regulatory_class': 'H-panel gene; molecular role stated separately', 'trajectory_label_scope': 'Fixed project SGN annotation/tree; local expression association'}, 'Dio3': {'evidence_scope': 'Mouse embryonic SG/epithelial expression and cochlear knockout', 'biological_role': 'Thyroid-hormone inactivation; limits premature cochlear differentiation', 'primary_sources': 'https://pmc.ncbi.nlm.nih.gov/articles/PMC2659284/', 'uniprot_url': '', 'red_recommended': False, 'highlight_basis': 'Mechanistically interpretable candidate; no automatic red for any ear association', 'limitations': 'Whole-cochlea/epithelial phenotype does not establish SGN-autonomous AC differentiation', 'selection_context': 'H/AC/down: rank=22, fitted_delta=-0.67509, observed_delta=-0.55755, local_fraction=0.9044', 'selection_reason': 'Significant whole-tree association; fitted and observed local directions agree; selected to display thyroid-hormone inactivation; limits premature cochlear differentiation. The stated protein mechanism motivates follow-up, not a causal claim from the dot plot.', 'direction_interpretation': 'Current fixed-tree local expression direction; pooled age/branch expression in literature is not automatically comparable', 'review_date': '2026-10-22', 'regulatory_class': 'H-panel gene; molecular role stated separately', 'trajectory_label_scope': 'Fixed project SGN annotation/tree; local expression association'}, 'Sobp': {'evidence_scope': 'Mouse Jackson-circler genetic cochlear phenotype and acoustic ganglion expression', 'biological_role': 'Nuclear zinc-finger/cofactor protein in cochlear growth and patterning', 'primary_sources': 'https://www.jneurosci.org/content/28/26/6633', 'uniprot_url': '', 'red_recommended': False, 'highlight_basis': 'Mechanistically interpretable candidate; no automatic red for any ear association', 'limitations': 'Main phenotypes concern cochlear duct/organ-of-Corti; no isolated SGN AC direction validated', 'selection_context': 'H/AC/down: rank=12, fitted_delta=-0.82095, observed_delta=-0.46054, local_fraction=0.7256', 'selection_reason': 'Significant whole-tree association; fitted and observed local directions agree; selected to display nuclear zinc-finger/cofactor protein in cochlear growth and patterning. The stated protein mechanism motivates follow-up, not a causal claim from the dot plot.', 'direction_interpretation': 'Current fixed-tree local expression direction; pooled age/branch expression in literature is not automatically comparable', 'review_date': '2026-10-22', 'regulatory_class': 'H-panel gene; molecular role stated separately', 'trajectory_label_scope': 'Fixed project SGN annotation/tree; local expression association'}, 'Calb1': {'evidence_scope': 'Mouse SGN subtype expression and tissue validation', 'biological_role': 'Calcium-binding protein; combinatorial SGN subtype reference', 'primary_sources': 'https://pmc.ncbi.nlm.nih.gov/articles/PMC6150604/; https://www.nature.com/articles/s41467-022-31580-1', 'uniprot_url': '', 'red_recommended': True, 'highlight_basis': 'Established marker/regulatory reference; evidence class stated separately', 'limitations': 'Calb1 is not a unique Ib identity gene and embryonic/postnatal labels are age dependent', 'selection_context': 'H/AC/down: rank=13, fitted_delta=-0.81134, observed_delta=-0.61845, local_fraction=0.3531', 'selection_reason': 'Significant whole-tree association; fitted and observed local directions agree; selected to display calcium-binding protein; combinatorial sgn subtype reference. The stated protein mechanism motivates follow-up, not a causal claim from the dot plot.', 'direction_interpretation': 'Current fixed-tree local expression direction; pooled age/branch expression in literature is not automatically comparable', 'review_date': '2026-10-22', 'regulatory_class': 'H-panel gene; molecular role stated separately', 'trajectory_label_scope': 'Fixed project SGN annotation/tree; local expression association'}, 'Igfbp2': {'evidence_scope': 'Mouse SGN developmental branch expression', 'biological_role': 'IGF-binding secreted signaling modulator', 'primary_sources': 'https://pmc.ncbi.nlm.nih.gov/articles/PMC9860252/', 'uniprot_url': '', 'red_recommended': False, 'highlight_basis': 'Mechanistically interpretable candidate; no automatic red for any ear association', 'limitations': 'Current AC decline is not evidence of inhibitory fate control', 'selection_context': 'H/AC/down: rank=1, fitted_delta=-1.525, observed_delta=-0.79101, local_fraction=0.6426', 'selection_reason': 'Significant whole-tree association; fitted and observed local directions agree; selected to display igf-binding secreted signaling modulator. The stated protein mechanism motivates follow-up, not a causal claim from the dot plot.', 'direction_interpretation': 'Current fixed-tree local expression direction; pooled age/branch expression in literature is not automatically comparable', 'review_date': '2026-10-22', 'regulatory_class': 'H-panel gene; molecular role stated separately', 'trajectory_label_scope': 'Fixed project SGN annotation/tree; local expression association'}, 'Calb2': {'evidence_scope': 'Mouse SGN subtype expression and genetic/tissue validation', 'biological_role': 'Calretinin calcium-binding protein; Ia-high / Ib-intermediate expression reference', 'primary_sources': 'https://pmc.ncbi.nlm.nih.gov/articles/PMC6150604/; https://pmc.ncbi.nlm.nih.gov/articles/PMC10202259/', 'uniprot_url': '', 'red_recommended': True, 'highlight_basis': 'Established marker/regulatory reference; evidence class stated separately', 'limitations': 'Early Calb2 is broader than mature Ia; present Ib increase is not a unique Ib marker claim', 'selection_context': 'H/Ib/up: rank=1, fitted_delta=1.5947, observed_delta=1.1481, local_fraction=0.6787; H/Ia/up: rank=1, fitted_delta=0.52166, observed_delta=0.49227, local_fraction=0.222', 'selection_reason': 'Significant whole-tree association; fitted and observed local directions agree; selected to display calretinin calcium-binding protein; ia-high / ib-intermediate expression reference. The stated protein mechanism motivates follow-up, not a causal claim from the dot plot.', 'direction_interpretation': 'Current fixed-tree local expression direction; pooled age/branch expression in literature is not automatically comparable', 'review_date': '2026-10-22', 'regulatory_class': 'H-panel gene; molecular role stated separately', 'trajectory_label_scope': 'Fixed project SGN annotation/tree; local expression association'}, 'Igfbpl1': {'evidence_scope': 'Mouse embryonic SGN subtype expression, original Fig1', 'biological_role': 'IGF-related neuronal-growth protein; immature SGN expression reference', 'primary_sources': 'https://www.nature.com/articles/s41467-022-31580-1', 'uniprot_url': '', 'red_recommended': True, 'highlight_basis': 'Established marker/regulatory reference; evidence class stated separately', 'limitations': 'Not exclusive to Type II: embryonic Ia/Ib/II and current Ic contexts can express it', 'selection_context': 'H/Ib/up: rank=2, fitted_delta=1.4074, observed_delta=0.95207, local_fraction=0.5308; H/Ic/up: rank=1, fitted_delta=0.68789, observed_delta=0.55901, local_fraction=0.2594', 'selection_reason': 'Significant whole-tree association; fitted and observed local directions agree; selected to display igf-related neuronal-growth protein; immature sgn expression reference. The stated protein mechanism motivates follow-up, not a causal claim from the dot plot.', 'direction_interpretation': 'Current fixed-tree local expression direction; pooled age/branch expression in literature is not automatically comparable', 'review_date': '2026-10-22', 'regulatory_class': 'H-panel gene; molecular role stated separately', 'trajectory_label_scope': 'Fixed project SGN annotation/tree; local expression association'}, 'Nrcam': {'evidence_scope': 'Mouse embryonic/postnatal SGN protein localization and cochlear knockout', 'biological_role': 'Neuronal cell adhesion, cochlear afferent guidance and patterning', 'primary_sources': 'https://pmc.ncbi.nlm.nih.gov/articles/PMC6105381/', 'uniprot_url': '', 'red_recommended': False, 'highlight_basis': 'Mechanistically interpretable candidate; no automatic red for any ear association', 'limitations': 'Broad cochlear expression/whole knockout; not an Ib-specific fate mechanism', 'selection_context': 'H/Ib/up: rank=82, fitted_delta=0.60252, observed_delta=0.38985, local_fraction=0.6571', 'selection_reason': 'Significant whole-tree association; fitted and observed local directions agree; selected to display neuronal cell adhesion, cochlear afferent guidance and patterning. The stated protein mechanism motivates follow-up, not a causal claim from the dot plot.', 'direction_interpretation': 'Current fixed-tree local expression direction; pooled age/branch expression in literature is not automatically comparable', 'review_date': '2026-10-22', 'regulatory_class': 'H-panel gene; molecular role stated separately', 'trajectory_label_scope': 'Fixed project SGN annotation/tree; local expression association'}, 'Dmxl2': {'evidence_scope': 'Mouse auditory hair-cell conditional knockout', 'biological_role': 'Vesicle endocytosis/recycling regulator with auditory synapse relevance', 'primary_sources': 'https://pmc.ncbi.nlm.nih.gov/articles/PMC11411588/', 'uniprot_url': '', 'red_recommended': False, 'highlight_basis': 'Mechanistically interpretable candidate; no automatic red for any ear association', 'limitations': 'Functional experiments are hair-cell-specific; no direct Ib SGN mechanism claimed', 'selection_context': 'H/Ib/up: rank=177, fitted_delta=0.54579, observed_delta=0.39472, local_fraction=0.4586', 'selection_reason': 'Significant whole-tree association; fitted and observed local directions agree; selected to display vesicle endocytosis/recycling regulator with auditory synapse relevance. The stated protein mechanism motivates follow-up, not a causal claim from the dot plot.', 'direction_interpretation': 'Current fixed-tree local expression direction; pooled age/branch expression in literature is not automatically comparable', 'review_date': '2026-10-22', 'regulatory_class': 'H-panel gene; molecular role stated separately', 'trajectory_label_scope': 'Fixed project SGN annotation/tree; local expression association'}, 'Lypd1': {'evidence_scope': 'Mouse SGN embryonic/postnatal expression and Runx1 lineage genetics', 'biological_role': 'Ly6-domain cell-surface protein; Ib/Ic precursor and later Ic expression reference', 'primary_sources': 'https://pmc.ncbi.nlm.nih.gov/articles/PMC9860252/; https://pmc.ncbi.nlm.nih.gov/articles/PMC10202259/', 'uniprot_url': '', 'red_recommended': True, 'highlight_basis': 'Established marker/regulatory reference; evidence class stated separately', 'limitations': 'Early Lypd1-positive cells are broader than mature Ic; current Ib/AB declines support divergence without proving lineage ancestry', 'selection_context': 'H/Ib/down: rank=1, fitted_delta=-1.1849, observed_delta=-0.69473, local_fraction=0.3522; H/AB/down: rank=1, fitted_delta=-1.8522, observed_delta=-1.2854, local_fraction=0.5506', 'selection_reason': 'Significant whole-tree association; fitted and observed local directions agree; selected to display ly6-domain cell-surface protein; ib/ic precursor and later ic expression reference. The stated protein mechanism motivates follow-up, not a causal claim from the dot plot.', 'direction_interpretation': 'Current fixed-tree local expression direction; pooled age/branch expression in literature is not automatically comparable', 'review_date': '2026-10-22', 'regulatory_class': 'H-panel gene; molecular role stated separately', 'trajectory_label_scope': 'Fixed project SGN annotation/tree; local expression association'}, 'Ntng1': {'evidence_scope': 'Mouse developing SGN and SGN subtype expression', 'biological_role': 'Netrin-family neuronal adhesion and SGN identity/developmental reference', 'primary_sources': 'https://pmc.ncbi.nlm.nih.gov/articles/PMC3167573/; https://pmc.ncbi.nlm.nih.gov/articles/PMC9860252/; https://pmc.ncbi.nlm.nih.gov/articles/PMC6150604/', 'uniprot_url': '', 'red_recommended': True, 'highlight_basis': 'Established SGN developmental/identity reference', 'limitations': 'Not a unique embryonic Ic marker and not causal fate validation', 'selection_context': 'H/Ib/down: rank=8, fitted_delta=-0.68431, observed_delta=-0.35153, local_fraction=0.2658', 'selection_reason': 'Significant whole-tree association; fitted and observed local directions agree; selected to display netrin-family neuronal adhesion and sgn identity/developmental reference. The stated protein mechanism motivates follow-up, not a causal claim from the dot plot.', 'direction_interpretation': 'Current fixed-tree local expression direction; pooled age/branch expression in literature is not automatically comparable', 'review_date': '2026-10-22', 'regulatory_class': 'H-panel gene; molecular role stated separately', 'trajectory_label_scope': 'Fixed project SGN annotation/tree; local expression association'}, 'Bhlhe22': {'evidence_scope': 'Mouse developing SGN expression/module, original Fig4j', 'biological_role': 'Neuronal bHLH transcriptional regulator; Ic-early module association', 'primary_sources': 'https://www.nature.com/articles/s41467-022-31580-1', 'uniprot_url': '', 'red_recommended': False, 'highlight_basis': 'Mechanistically interpretable candidate; no automatic red for any ear association', 'limitations': 'Module membership is expression association, not targeted SGN functional validation', 'selection_context': 'H/Ib/down: rank=38, fitted_delta=-0.46885, observed_delta=-0.36337, local_fraction=0.2734; I/Ib/down: rank=4, fitted_delta=-0.46885, observed_delta=-0.36337, local_fraction=0.2734; I/Ic/down: rank=9, fitted_delta=-0.14169, observed_delta=-0.22083, local_fraction=0.08263', 'selection_reason': 'Significant whole-tree association; fitted and observed local directions agree; selected to display neuronal bhlh transcriptional regulator; ic-early module association. The stated protein mechanism motivates follow-up, not a causal claim from the dot plot.', 'direction_interpretation': 'Current fixed-tree local expression direction; pooled age/branch expression in literature is not automatically comparable', 'review_date': '2026-10-22', 'regulatory_class': 'Source-catalogue transcription factor/regulator', 'trajectory_label_scope': 'Fixed project SGN annotation/tree; local expression association'}, 'Rbfox3': {'evidence_scope': 'Mouse developing SGN Ic-early module', 'biological_role': 'Neuronal RNA-binding/splicing factor NeuN', 'primary_sources': 'https://www.nature.com/articles/s41467-022-31580-1', 'uniprot_url': '', 'red_recommended': True, 'highlight_basis': 'Established marker/regulatory reference; evidence class stated separately', 'limitations': 'Broad neuronal expression; not a DNA-binding transcription factor or Ic-specific marker', 'selection_context': 'H/Ib/down: rank=9, fitted_delta=-0.67799, observed_delta=-0.54146, local_fraction=0.4531; H/Ic/down: rank=2, fitted_delta=-0.57194, observed_delta=-0.35497, local_fraction=0.3822', 'selection_reason': 'Significant whole-tree association; fitted and observed local directions agree; selected to display neuronal rna-binding/splicing factor neun. The stated protein mechanism motivates follow-up, not a causal claim from the dot plot.', 'direction_interpretation': 'Current fixed-tree local expression direction; pooled age/branch expression in literature is not automatically comparable', 'review_date': '2026-10-22', 'regulatory_class': 'RNA-binding/splicing protein; H panel only', 'trajectory_label_scope': 'Fixed project SGN annotation/tree; local expression association'}, 'Gata3': {'evidence_scope': 'Mouse CVG/SGN/VGN expression and auditory-neuron genetics', 'biological_role': 'Early auditory developmental and SGN identity-regulatory reference', 'primary_sources': 'https://www.nature.com/articles/s41467-022-31580-1; https://pmc.ncbi.nlm.nih.gov/articles/PMC3167573/', 'uniprot_url': '', 'red_recommended': True, 'highlight_basis': 'Established auditory/CVG developmental regulator', 'limitations': 'Early VGN subset also expresses Gata3; gene expression alone is not SGN-specific at these ages Current Ic increase differs from the original 2022 Type-I decline; age/state windows and fixed topology are not directly matched.', 'selection_context': 'H/Ic/up: rank=2, fitted_delta=0.58221, observed_delta=0.93689, local_fraction=0.2406; I/VGN/down: rank=7, fitted_delta=-0.47238, observed_delta=-0.46814, local_fraction=0.1952; I/T/up: rank=4, fitted_delta=1.6678, observed_delta=0.93319, local_fraction=0.6892; I/Ia/down: rank=5, fitted_delta=-0.21716, observed_delta=-0.20211, local_fraction=0.08975', 'selection_reason': 'Significant whole-tree association; fitted and observed local directions agree; selected to display early auditory developmental and sgn identity-regulatory reference. The stated protein mechanism motivates follow-up, not a causal claim from the dot plot.', 'direction_interpretation': 'Current fixed-tree local expression direction; pooled age/branch expression in literature is not automatically comparable', 'review_date': '2026-10-22', 'regulatory_class': 'Source-catalogue transcription factor/regulator', 'trajectory_label_scope': 'CVG to VGN denotes the VGN-candidate-enriched arm, not validated VGN identity for every cell'}, 'Fxyd6': {'evidence_scope': 'SGN protein localization and biochemical ATPase regulation', 'biological_role': 'Na/K-ATPase regulatory protein; ionic-maturation candidate', 'primary_sources': 'https://doi.org/10.1074/jbc.M609872200; https://doi.org/10.1002/dvdy.21269', 'uniprot_url': '', 'red_recommended': False, 'highlight_basis': 'Mechanistically interpretable candidate; no automatic red for any ear association', 'limitations': 'No Ic-specific fate function established', 'selection_context': 'H/Ic/up: rank=4, fitted_delta=0.47858, observed_delta=0.90666, local_fraction=0.3007', 'selection_reason': 'Significant whole-tree association; fitted and observed local directions agree; selected to display na/k-atpase regulatory protein; ionic-maturation candidate. The stated protein mechanism motivates follow-up, not a causal claim from the dot plot.', 'direction_interpretation': 'Current fixed-tree local expression direction; pooled age/branch expression in literature is not automatically comparable', 'review_date': '2026-10-22', 'regulatory_class': 'H-panel gene; molecular role stated separately', 'trajectory_label_scope': 'Fixed project SGN annotation/tree; local expression association'}, 'Nefh': {'evidence_scope': 'Mouse SGN maturation/identity expression', 'biological_role': 'Neurofilament-heavy neuronal maturation-state reference', 'primary_sources': 'https://www.nature.com/articles/s41467-022-31580-1', 'uniprot_url': '', 'red_recommended': True, 'highlight_basis': 'Established neuronal maturation reference', 'limitations': 'Broad neuronal maturation protein; Ia rise is modest and Ic decline is segment specific; no fate-control claim', 'selection_context': 'H/Ic/down: rank=14, fitted_delta=-0.32781, observed_delta=-0.12629, local_fraction=0.1747; H/Ia/up: rank=11, fitted_delta=0.19958, observed_delta=0.31442, local_fraction=0.1064', 'selection_reason': 'Significant whole-tree association; fitted and observed local directions agree; selected to display neurofilament-heavy neuronal maturation-state reference. The stated protein mechanism motivates follow-up, not a causal claim from the dot plot. Effect size and study/source sensitivity must be read alongside rank and significance.', 'direction_interpretation': 'Current fixed-tree local expression direction; pooled age/branch expression in literature is not automatically comparable', 'review_date': '2026-10-22', 'regulatory_class': 'H-panel gene; molecular role stated separately', 'trajectory_label_scope': 'Fixed project SGN annotation/tree; local expression association'}, 'Slc17a6': {'evidence_scope': 'Purified mouse SGN transcript expression', 'biological_role': 'VGLUT2 vesicular glutamate transport; neurotransmitter-program candidate', 'primary_sources': 'https://elifesciences.org/articles/50491', 'uniprot_url': '', 'red_recommended': False, 'highlight_basis': 'Mechanistically interpretable candidate; no automatic red for any ear association', 'limitations': 'Neurotransmitter machinery rather than demonstrated Ic fate control', 'selection_context': 'H/Ic/down: rank=4, fitted_delta=-0.48018, observed_delta=-0.28299, local_fraction=0.257', 'selection_reason': 'Significant whole-tree association; fitted and observed local directions agree; selected to display vglut2 vesicular glutamate transport; neurotransmitter-program candidate. The stated protein mechanism motivates follow-up, not a causal claim from the dot plot.', 'direction_interpretation': 'Current fixed-tree local expression direction; pooled age/branch expression in literature is not automatically comparable', 'review_date': '2026-10-22', 'regulatory_class': 'H-panel gene; molecular role stated separately', 'trajectory_label_scope': 'Fixed project SGN annotation/tree; local expression association'}, 'Rgs10': {'evidence_scope': 'Biochemical human RGS10 and curated mouse ortholog', 'biological_role': 'Gi-family GTPase activation and attenuation of GPCR signaling', 'primary_sources': 'https://www.nature.com/articles/383175a0', 'uniprot_url': 'https://www.uniprot.org/uniprotkb/Q9CQE5/entry', 'red_recommended': False, 'highlight_basis': 'Mechanistically interpretable candidate; no automatic red for any ear association', 'limitations': 'No direct ear-development functional study established; potassium-channel modulation is mainly by-similarity annotation; do not call validated SGN excitability or Ic fate factor', 'selection_context': 'H/Ic/down: rank=1, fitted_delta=-0.61943, observed_delta=-0.39133, local_fraction=0.2974', 'selection_reason': 'Significant whole-tree association; fitted and observed local directions agree; selected to display gi-family gtpase activation and attenuation of gpcr signaling. The stated protein mechanism motivates follow-up, not a causal claim from the dot plot.', 'direction_interpretation': 'Current fixed-tree local expression direction; pooled age/branch expression in literature is not automatically comparable', 'review_date': '2026-10-22', 'regulatory_class': 'H-panel gene; molecular role stated separately', 'trajectory_label_scope': 'Fixed project SGN annotation/tree; local expression association'}, 'Islr2': {'evidence_scope': 'Mouse E16.5 SGN RNAscope and trajectory state', 'biological_role': 'Cell-surface LRR/Ig neuronal protein; non-Ic intermediate-state expression reference', 'primary_sources': 'https://www.nature.com/articles/s41467-022-31580-1', 'uniprot_url': '', 'red_recommended': True, 'highlight_basis': 'Established marker/regulatory reference; evidence class stated separately', 'limitations': 'Original Ia/Ib/II intermediate differs from the current fixed topology; no proof of common-fate potency', 'selection_context': 'H/AB/up: rank=19, fitted_delta=1.2746, observed_delta=1.0571, local_fraction=0.5518', 'selection_reason': 'Significant whole-tree association; fitted and observed local directions agree; selected to display cell-surface lrr/ig neuronal protein; non-ic intermediate-state expression reference. The stated protein mechanism motivates follow-up, not a causal claim from the dot plot.', 'direction_interpretation': 'Current fixed-tree local expression direction; pooled age/branch expression in literature is not automatically comparable', 'review_date': '2026-10-22', 'regulatory_class': 'H-panel gene; molecular role stated separately', 'trajectory_label_scope': 'Fixed project SGN annotation/tree; local expression association'}, 'Unc5d': {'evidence_scope': 'Mouse cortical neuronal gain/loss-of-function experiments', 'biological_role': 'FLRT-dependent repulsive signaling / neuronal migration receptor candidate', 'primary_sources': 'https://pmc.ncbi.nlm.nih.gov/articles/PMC3160250/; https://pmc.ncbi.nlm.nih.gov/articles/PMC4210639/', 'uniprot_url': '', 'red_recommended': False, 'highlight_basis': 'Mechanistically interpretable candidate; no automatic red for any ear association', 'limitations': 'Cortical functional mechanism does not validate SGN AB development; no direct ear role claimed', 'selection_context': 'H/AB/up: rank=31, fitted_delta=1.0913, observed_delta=0.68096, local_fraction=0.6028', 'selection_reason': 'Significant whole-tree association; fitted and observed local directions agree; selected to display flrt-dependent repulsive signaling / neuronal migration receptor candidate. The stated protein mechanism motivates follow-up, not a causal claim from the dot plot.', 'direction_interpretation': 'Current fixed-tree local expression direction; pooled age/branch expression in literature is not automatically comparable', 'review_date': '2026-10-22', 'regulatory_class': 'H-panel gene; molecular role stated separately', 'trajectory_label_scope': 'Fixed project SGN annotation/tree; local expression association'}, 'Tmem132e': {'evidence_scope': 'Curated mouse protein and orthologous hair-cell/hearing annotation', 'biological_role': 'Transmembrane protein; membrane-state/hearing-related candidate', 'primary_sources': '', 'uniprot_url': 'https://www.uniprot.org/uniprotkb/Q6IEE6/entry', 'red_recommended': False, 'highlight_basis': 'Mechanistically interpretable candidate; no automatic red for any ear association', 'limitations': 'Mouse function annotation is by similarity; no SGN AB/Ia-specific function established', 'selection_context': 'H/AB/up: rank=168, fitted_delta=0.64892, observed_delta=0.55682, local_fraction=0.4135; H/Ia/down: rank=9, fitted_delta=-0.27452, observed_delta=-0.15752, local_fraction=0.1749', 'selection_reason': 'Significant whole-tree association; fitted and observed local directions agree; selected to display transmembrane protein; membrane-state/hearing-related candidate. The stated protein mechanism motivates follow-up, not a causal claim from the dot plot.', 'direction_interpretation': 'Current fixed-tree local expression direction; pooled age/branch expression in literature is not automatically comparable', 'review_date': '2026-10-22', 'regulatory_class': 'H-panel gene; molecular role stated separately', 'trajectory_label_scope': 'Fixed project SGN annotation/tree; local expression association'}, 'Cntn5': {'evidence_scope': 'Curated contactin function; neural development context', 'biological_role': 'GPI-anchored neuronal cell-adhesion/connectivity candidate', 'primary_sources': 'https://www.informatics.jax.org/marker/MGI:3042287', 'uniprot_url': 'https://www.uniprot.org/uniprotkb/P68500/entry', 'red_recommended': False, 'highlight_basis': 'Mechanistically interpretable candidate; no automatic red for any ear association', 'limitations': 'Central auditory/contactin evidence must not be presented as an SGN-autonomous subtype mechanism', 'selection_context': 'H/AB/up: rank=15, fitted_delta=1.3186, observed_delta=0.28587, local_fraction=0.4419; H/Ia/up: rank=6, fitted_delta=0.29582, observed_delta=0.33105, local_fraction=0.09913', 'selection_reason': 'Significant whole-tree association; fitted and observed local directions agree; selected to display gpi-anchored neuronal cell-adhesion/connectivity candidate. The stated protein mechanism motivates follow-up, not a causal claim from the dot plot.', 'direction_interpretation': 'Current fixed-tree local expression direction; pooled age/branch expression in literature is not automatically comparable', 'review_date': '2026-10-22', 'regulatory_class': 'H-panel gene; molecular role stated separately', 'trajectory_label_scope': 'Fixed project SGN annotation/tree; local expression association'}, 'Ebf1': {'evidence_scope': 'Mouse early CVG/SG expression; broad inner-ear knockout', 'biological_role': 'Neuronal differentiation-associated transcription factor; SGN organization/innervation context', 'primary_sources': 'https://pmc.ncbi.nlm.nih.gov/articles/PMC10592509/; https://pmc.ncbi.nlm.nih.gov/articles/PMC10869149/', 'uniprot_url': '', 'red_recommended': False, 'highlight_basis': 'Direct expression and developmental context', 'limitations': 'Whole-ear/mesenchymal effects do not prove CVG-neuron-autonomous control', 'selection_context': 'H/AB/down: rank=17, fitted_delta=-0.95214, observed_delta=-0.51627, local_fraction=0.5125; I/Ib/down: rank=2, fitted_delta=-0.49277, observed_delta=-0.28875, local_fraction=0.2652; I/AB/down: rank=3, fitted_delta=-0.95214, observed_delta=-0.51627, local_fraction=0.5125', 'selection_reason': 'Significant whole-tree association; fitted and observed local directions agree; selected to display neuronal differentiation-associated transcription factor; sgn organization/innervation context. The stated protein mechanism motivates follow-up, not a causal claim from the dot plot.', 'direction_interpretation': 'Current fixed-tree local expression direction; pooled age/branch expression in literature is not automatically comparable', 'review_date': '2026-10-22', 'regulatory_class': 'Source-catalogue transcription factor/regulator', 'trajectory_label_scope': 'Fixed project SGN annotation/tree; local expression association'}, 'Kcna1': {'evidence_scope': 'Mouse SGN genetic/electrophysiological studies', 'biological_role': 'Kv1.1 potassium-channel excitability and timing control', 'primary_sources': 'https://pmc.ncbi.nlm.nih.gov/articles/PMC3798938/', 'uniprot_url': '', 'red_recommended': False, 'highlight_basis': 'Mechanistically interpretable candidate; no automatic red for any ear association', 'limitations': 'Mature neuronal physiology does not establish AB branch fate regulation', 'selection_context': 'H/AB/down: rank=101, fitted_delta=-0.35277, observed_delta=-0.2658, local_fraction=0.2287', 'selection_reason': 'Significant whole-tree association; fitted and observed local directions agree; selected to display kv1.1 potassium-channel excitability and timing control. The stated protein mechanism motivates follow-up, not a causal claim from the dot plot.', 'direction_interpretation': 'Current fixed-tree local expression direction; pooled age/branch expression in literature is not automatically comparable', 'review_date': '2026-10-22', 'regulatory_class': 'H-panel gene; molecular role stated separately', 'trajectory_label_scope': 'Fixed project SGN annotation/tree; local expression association'}, 'Sema3a': {'evidence_scope': 'Mouse cochlear guidance and synaptic-patterning experiments', 'biological_role': 'Secreted semaphorin guidance signal modulating cochlear innervation', 'primary_sources': 'https://pmc.ncbi.nlm.nih.gov/articles/PMC9812910/', 'uniprot_url': '', 'red_recommended': False, 'highlight_basis': 'Mechanistically interpretable candidate; no automatic red for any ear association', 'limitations': 'Extracellular source/cell type matters; AB RNA decrease is not a causal branch-choice test', 'selection_context': 'H/AB/down: rank=10, fitted_delta=-1.0392, observed_delta=-0.57475, local_fraction=0.6791', 'selection_reason': 'Significant whole-tree association; fitted and observed local directions agree; selected to display secreted semaphorin guidance signal modulating cochlear innervation. The stated protein mechanism motivates follow-up, not a causal claim from the dot plot.', 'direction_interpretation': 'Current fixed-tree local expression direction; pooled age/branch expression in literature is not automatically comparable', 'review_date': '2026-10-22', 'regulatory_class': 'H-panel gene; molecular role stated separately', 'trajectory_label_scope': 'Fixed project SGN annotation/tree; local expression association'}, 'Cntnap5a': {'evidence_scope': 'Mouse official gene/protein annotation; predominantly inferred function', 'biological_role': 'Neurexin-related transmembrane cell-adhesion/communication candidate', 'primary_sources': 'https://www.ncbi.nlm.nih.gov/gene/636808; https://www.informatics.jax.org/marker/MGI:3643623', 'uniprot_url': 'https://www.uniprot.org/uniprotkb/Q0V8T9/entry', 'red_recommended': False, 'highlight_basis': 'Mechanistically interpretable candidate; no automatic red for any ear association', 'limitations': 'Function is largely predicted by family/orthology; no direct SGN development study found; retain as black discovery candidate, not validated Ia marker', 'selection_context': 'H/Ia/up: rank=9, fitted_delta=0.25237, observed_delta=0.48376, local_fraction=0.1381', 'selection_reason': 'Significant whole-tree association; fitted and observed local directions agree; selected to display neurexin-related transmembrane cell-adhesion/communication candidate. The stated protein mechanism motivates follow-up, not a causal claim from the dot plot. This is a discovery candidate supported mainly by family/orthology annotation; function is not experimentally confirmed in SGNs.', 'direction_interpretation': 'Current fixed-tree local expression direction; pooled age/branch expression in literature is not automatically comparable', 'review_date': '2026-10-22', 'regulatory_class': 'H-panel gene; molecular role stated separately', 'trajectory_label_scope': 'Fixed project SGN annotation/tree; local expression association'}, 'Cacng5': {'evidence_scope': 'Mouse SGN single-cell expression/spatial gradients', 'biological_role': 'Calcium-channel gamma-family / AMPA-receptor regulatory candidate', 'primary_sources': 'https://pmc.ncbi.nlm.nih.gov/articles/PMC6150604/', 'uniprot_url': '', 'red_recommended': False, 'highlight_basis': 'Mechanistically interpretable candidate; no automatic red for any ear association', 'limitations': 'Subtype-dependent spatial gradients do not validate a small Ia temporal decline', 'selection_context': 'H/Ia/down: rank=8, fitted_delta=-0.28206, observed_delta=-0.059245, local_fraction=0.2319', 'selection_reason': 'Significant whole-tree association; fitted and observed local directions agree; selected to display calcium-channel gamma-family / ampa-receptor regulatory candidate. The stated protein mechanism motivates follow-up, not a causal claim from the dot plot.', 'direction_interpretation': 'Current fixed-tree local expression direction; pooled age/branch expression in literature is not automatically comparable', 'review_date': '2026-10-22', 'regulatory_class': 'H-panel gene; molecular role stated separately', 'trajectory_label_scope': 'Fixed project SGN annotation/tree; local expression association'}, 'Tgfb2': {'evidence_scope': 'Mouse SGN developmental expression and broader otic signaling', 'biological_role': 'TGF-beta secreted signaling ligand', 'primary_sources': 'https://pmc.ncbi.nlm.nih.gov/articles/PMC3167573/; https://pmc.ncbi.nlm.nih.gov/articles/PMC7461926/', 'uniprot_url': '', 'red_recommended': False, 'highlight_basis': 'Mechanistically interpretable candidate; no automatic red for any ear association', 'limitations': 'Current Ia local change is small; broad otic functions do not establish subtype-autonomous control', 'selection_context': 'H/Ia/down: rank=54, fitted_delta=-0.21857, observed_delta=-0.32046, local_fraction=0.1407', 'selection_reason': 'Significant whole-tree association; fitted and observed local directions agree; selected to display tgf-beta secreted signaling ligand. The stated protein mechanism motivates follow-up, not a causal claim from the dot plot.', 'direction_interpretation': 'Current fixed-tree local expression direction; pooled age/branch expression in literature is not automatically comparable', 'review_date': '2026-10-22', 'regulatory_class': 'H-panel gene; molecular role stated separately', 'trajectory_label_scope': 'Fixed project SGN annotation/tree; local expression association'}, 'Cplx2': {'evidence_scope': 'Mouse developing SGN transcriptomic program', 'biological_role': 'Complexin-dependent synaptic-vesicle release candidate; immature neuronal program', 'primary_sources': 'https://www.nature.com/articles/s41467-022-31580-1', 'uniprot_url': '', 'red_recommended': False, 'highlight_basis': 'Mechanistically interpretable candidate; no automatic red for any ear association', 'limitations': 'Broad neuronal exocytosis protein; not an Ia-specific fate marker', 'selection_context': 'H/Ia/down: rank=149, fitted_delta=-0.19169, observed_delta=-0.063686, local_fraction=0.1431', 'selection_reason': 'Significant whole-tree association; fitted and observed local directions agree; selected to display complexin-dependent synaptic-vesicle release candidate; immature neuronal program. The stated protein mechanism motivates follow-up, not a causal claim from the dot plot.', 'direction_interpretation': 'Current fixed-tree local expression direction; pooled age/branch expression in literature is not automatically comparable', 'review_date': '2026-10-22', 'regulatory_class': 'H-panel gene; molecular role stated separately', 'trajectory_label_scope': 'Fixed project SGN annotation/tree; local expression association'}, 'Tac1': {'evidence_scope': 'Mouse SGN Type-II developmental/transcriptomic expression', 'biological_role': 'Neuropeptide precursor; Type-II expression reference', 'primary_sources': 'https://www.nature.com/articles/s41467-022-31580-1; https://pmc.ncbi.nlm.nih.gov/articles/PMC6429032/', 'uniprot_url': '', 'red_recommended': True, 'highlight_basis': 'Established marker/regulatory reference; evidence class stated separately', 'limitations': 'Neuropeptide expression does not prove Type-II fate specification', 'selection_context': 'H/II/up: rank=6, fitted_delta=0.74879, observed_delta=0.55753, local_fraction=0.5889', 'selection_reason': 'Significant whole-tree association; fitted and observed local directions agree; selected to display neuropeptide precursor; type-ii expression reference. The stated protein mechanism motivates follow-up, not a causal claim from the dot plot.', 'direction_interpretation': 'Current fixed-tree local expression direction; pooled age/branch expression in literature is not automatically comparable', 'review_date': '2026-10-22', 'regulatory_class': 'H-panel gene; molecular role stated separately', 'trajectory_label_scope': 'Fixed project SGN annotation/tree; local expression association'}, 'Th': {'evidence_scope': 'Mouse Type-II tissue expression and tonotopic analysis', 'biological_role': 'Tyrosine hydroxylase; catecholamine synthesis and Type-II expression reference', 'primary_sources': 'https://pmc.ncbi.nlm.nih.gov/articles/PMC5975645/', 'uniprot_url': '', 'red_recommended': True, 'highlight_basis': 'Established marker/regulatory reference; evidence class stated separately', 'limitations': 'TH is tonotopically graded and labels a subset of Type II; not an exclusive identity gene', 'selection_context': 'H/II/up: rank=96, fitted_delta=0.23569, observed_delta=0.41851, local_fraction=0.258', 'selection_reason': 'Significant whole-tree association; fitted and observed local directions agree; selected to display tyrosine hydroxylase; catecholamine synthesis and type-ii expression reference. The stated protein mechanism motivates follow-up, not a causal claim from the dot plot.', 'direction_interpretation': 'Current fixed-tree local expression direction; pooled age/branch expression in literature is not automatically comparable', 'review_date': '2026-10-22', 'regulatory_class': 'H-panel gene; molecular role stated separately', 'trajectory_label_scope': 'Fixed project SGN annotation/tree; local expression association'}, 'Gpr22': {'evidence_scope': 'Mouse original SGN Fig2g Type-II increasing gene', 'biological_role': 'Orphan GPCR; neuronal signaling candidate', 'primary_sources': 'https://www.nature.com/articles/s41467-022-31580-1', 'uniprot_url': '', 'red_recommended': False, 'highlight_basis': 'Mechanistically interpretable candidate; no automatic red for any ear association', 'limitations': 'Original expression association agrees with II increase; ligand and SGN-specific function remain unverified', 'selection_context': 'H/II/up: rank=7, fitted_delta=0.74612, observed_delta=0.79134, local_fraction=0.5583', 'selection_reason': 'Significant whole-tree association; fitted and observed local directions agree; selected to display orphan gpcr; neuronal signaling candidate. The stated protein mechanism motivates follow-up, not a causal claim from the dot plot.', 'direction_interpretation': 'Current fixed-tree local expression direction; pooled age/branch expression in literature is not automatically comparable', 'review_date': '2026-10-22', 'regulatory_class': 'H-panel gene; molecular role stated separately', 'trajectory_label_scope': 'Fixed project SGN annotation/tree; local expression association'}, 'Ngfr': {'evidence_scope': 'Mouse mature Type-II expression and cochlear culture context', 'biological_role': 'p75 neurotrophin receptor; mature-II expression reference', 'primary_sources': 'https://pmc.ncbi.nlm.nih.gov/articles/PMC6429032/; https://pmc.ncbi.nlm.nih.gov/articles/PMC7316616/', 'uniprot_url': '', 'red_recommended': True, 'highlight_basis': 'Established marker/regulatory reference; evidence class stated separately', 'limitations': 'Present II local change is only about 3 percent of whole-tree amplitude; small dynamic effect despite statistical admission', 'selection_context': 'H/II/up: rank=405, fitted_delta=0.060264, observed_delta=0.34951, local_fraction=0.02972', 'selection_reason': 'Significant whole-tree association; fitted and observed local directions agree; selected to display p75 neurotrophin receptor; mature-ii expression reference. The stated protein mechanism motivates follow-up, not a causal claim from the dot plot. Effect size and study/source sensitivity must be read alongside rank and significance.', 'direction_interpretation': 'Current fixed-tree local expression direction; pooled age/branch expression in literature is not automatically comparable', 'review_date': '2026-10-22', 'regulatory_class': 'H-panel gene; molecular role stated separately', 'trajectory_label_scope': 'Fixed project SGN annotation/tree; local expression association'}, 'Sema3e': {'evidence_scope': 'Curated mouse semaphorin protein; general developmental guidance', 'biological_role': 'Secreted PLXND1-associated guidance/signaling candidate', 'primary_sources': '', 'uniprot_url': 'https://www.uniprot.org/uniprotkb/P70275/entry', 'red_recommended': False, 'highlight_basis': 'Mechanistically interpretable candidate; no automatic red for any ear association', 'limitations': 'No SGN Type-II-autonomous perturbation identified; do not transfer SEMA3A/3F cochlear phenotypes to SEMA3E', 'selection_context': 'H/II/down: rank=1, fitted_delta=-1.2338, observed_delta=-0.4767, local_fraction=0.5598', 'selection_reason': 'Significant whole-tree association; fitted and observed local directions agree; selected to display secreted plxnd1-associated guidance/signaling candidate. The stated protein mechanism motivates follow-up, not a causal claim from the dot plot.', 'direction_interpretation': 'Current fixed-tree local expression direction; pooled age/branch expression in literature is not automatically comparable', 'review_date': '2026-10-22', 'regulatory_class': 'H-panel gene; molecular role stated separately', 'trajectory_label_scope': 'Fixed project SGN annotation/tree; local expression association'}, 'Kcnip4': {'evidence_scope': 'Mouse curated protein/biochemical channel-regulation evidence', 'biological_role': 'Calcium-dependent Kv4 A-current auxiliary regulator', 'primary_sources': 'https://pubmed.ncbi.nlm.nih.gov/19109250/', 'uniprot_url': 'https://www.uniprot.org/uniprotkb/Q6PHZ8/entry', 'red_recommended': False, 'highlight_basis': 'Mechanistically interpretable candidate; no automatic red for any ear association', 'limitations': 'Isoform-specific functions; no matching embryonic Type-II temporal or causal study established', 'selection_context': 'H/II/down: rank=32, fitted_delta=-0.64906, observed_delta=-0.25493, local_fraction=0.7015', 'selection_reason': 'Significant whole-tree association; fitted and observed local directions agree; selected to display calcium-dependent kv4 a-current auxiliary regulator. The stated protein mechanism motivates follow-up, not a causal claim from the dot plot.', 'direction_interpretation': 'Current fixed-tree local expression direction; pooled age/branch expression in literature is not automatically comparable', 'review_date': '2026-10-22', 'regulatory_class': 'H-panel gene; molecular role stated separately', 'trajectory_label_scope': 'Fixed project SGN annotation/tree; local expression association'}, 'Epha4': {'evidence_scope': 'Mouse cochlear guidance genetics and SGN transcript localization', 'biological_role': 'Ephrin receptor signaling for cochlear afferent sorting/fasciculation', 'primary_sources': 'https://www.nature.com/articles/ncomms2445; https://pmc.ncbi.nlm.nih.gov/articles/PMC10862897/', 'uniprot_url': '', 'red_recommended': False, 'highlight_basis': 'Mechanistically interpretable candidate; no automatic red for any ear association', 'limitations': 'Receptor is also strongly expressed in otic mesenchyme; cell type and age matter; current II decline is correlative', 'selection_context': 'H/II/down: rank=27, fitted_delta=-0.66542, observed_delta=-0.15601, local_fraction=0.3835', 'selection_reason': 'Significant whole-tree association; fitted and observed local directions agree; selected to display ephrin receptor signaling for cochlear afferent sorting/fasciculation. The stated protein mechanism motivates follow-up, not a causal claim from the dot plot.', 'direction_interpretation': 'Current fixed-tree local expression direction; pooled age/branch expression in literature is not automatically comparable', 'review_date': '2026-10-22', 'regulatory_class': 'H-panel gene; molecular role stated separately', 'trajectory_label_scope': 'Fixed project SGN annotation/tree; local expression association'}, 'Nhlh2': {'evidence_scope': 'Mouse early inner-ear neurons; in situ/expression evidence', 'biological_role': 'Early neuronal bHLH regulatory candidate', 'primary_sources': 'https://pmc.ncbi.nlm.nih.gov/articles/PMC3657738/; https://pmc.ncbi.nlm.nih.gov/articles/PMC9478650/', 'uniprot_url': '', 'red_recommended': False, 'highlight_basis': 'Early neuronal expression reference', 'limitations': 'No gene-specific SGN/VGN branch-choice causality established', 'selection_context': 'I/CVG/up: rank=9, fitted_delta=0.37973, observed_delta=0.74232, local_fraction=0.1643; I/Ia/down: rank=29, fitted_delta=-0.15731, observed_delta=-0.051998, local_fraction=0.06805', 'selection_reason': 'Significant whole-tree association; fitted and observed local directions agree; selected to display early neuronal bhlh regulatory candidate. The stated protein mechanism motivates follow-up, not a causal claim from the dot plot.', 'direction_interpretation': 'Current fixed-tree local expression direction; pooled age/branch expression in literature is not automatically comparable', 'review_date': '2026-10-22', 'regulatory_class': 'Source-catalogue transcription factor/regulator', 'trajectory_label_scope': 'Undiff. CVG labels the early mixed common segment; it does not demonstrate uncommitted/bipotent cells'}, 'Hmga2': {'evidence_scope': 'Mouse fetal/young neural stem cells; genetic functional evidence', 'biological_role': 'Chromatin-associated regulator supporting neural stem-cell self-renewal', 'primary_sources': 'https://pmc.ncbi.nlm.nih.gov/articles/PMC2582221/', 'uniprot_url': 'https://www.uniprot.org/uniprotkb/P52927/entry', 'red_recommended': False, 'highlight_basis': 'Mechanistic progenitor-state candidate', 'limitations': 'No direct CVG/VGN trajectory-specific validation identified', 'selection_context': 'I/CVG/down: rank=1, fitted_delta=-0.76423, observed_delta=-0.48902, local_fraction=0.3646', 'selection_reason': 'Significant whole-tree association; fitted and observed local directions agree; selected to display chromatin-associated regulator supporting neural stem-cell self-renewal. The stated protein mechanism motivates follow-up, not a causal claim from the dot plot.', 'direction_interpretation': 'Current fixed-tree local expression direction; pooled age/branch expression in literature is not automatically comparable', 'review_date': '2026-10-22', 'regulatory_class': 'Source-catalogue transcription factor/regulator', 'trajectory_label_scope': 'Undiff. CVG labels the early mixed common segment; it does not demonstrate uncommitted/bipotent cells'}, 'Rest': {'evidence_scope': 'Mouse/general neurogenesis chromatin experiments', 'biological_role': 'RE1-binding repressor silencing neuronal genes in progenitors', 'primary_sources': 'https://doi.org/10.1016/j.cell.2005.03.013', 'uniprot_url': 'https://www.uniprot.org/uniprotkb/Q8VIG1/entry', 'red_recommended': False, 'highlight_basis': 'Mechanistically interpretable candidate; no automatic red for any ear association', 'limitations': 'Developmental neural mechanism outside current CVG cells; source-sensitive decline is not proof of neuronal commitment', 'selection_context': 'I/CVG/down: rank=7, fitted_delta=-0.38526, observed_delta=-0.3198, local_fraction=0.3405', 'selection_reason': 'Significant whole-tree association; fitted and observed local directions agree; selected to display re1-binding repressor silencing neuronal genes in progenitors. The stated protein mechanism motivates follow-up, not a causal claim from the dot plot. Effect size and study/source sensitivity must be read alongside rank and significance.', 'direction_interpretation': 'Current fixed-tree local expression direction; pooled age/branch expression in literature is not automatically comparable', 'review_date': '2026-10-22', 'regulatory_class': 'Source-catalogue transcription factor/regulator', 'trajectory_label_scope': 'Undiff. CVG labels the early mixed common segment; it does not demonstrate uncommitted/bipotent cells'}, 'Ebf3': {'evidence_scope': 'Mouse CNS expression; general neuronal differentiation; protein annotation', 'biological_role': 'COE-family transcriptional regulatory candidate', 'primary_sources': 'https://pubmed.ncbi.nlm.nih.gov/9389446/', 'uniprot_url': 'https://www.uniprot.org/uniprotkb/O08791/entry', 'red_recommended': False, 'highlight_basis': 'Neural differentiation-associated TF candidate', 'limitations': 'Vestibular prosensory epithelial enrichment in another study is not VGN-neuron evidence', 'selection_context': 'I/VGN/up: rank=2, fitted_delta=0.90169, observed_delta=0.84025, local_fraction=0.6042', 'selection_reason': 'Significant whole-tree association; fitted and observed local directions agree; selected to display coe-family transcriptional regulatory candidate. The stated protein mechanism motivates follow-up, not a causal claim from the dot plot.', 'direction_interpretation': 'Current fixed-tree local expression direction; pooled age/branch expression in literature is not automatically comparable', 'review_date': '2026-10-22', 'regulatory_class': 'Source-catalogue transcription factor/regulator', 'trajectory_label_scope': 'CVG to VGN denotes the VGN-candidate-enriched arm, not validated VGN identity for every cell'}, 'Klf7': {'evidence_scope': 'Original SGN inferred regulon; sensory-neuron experiments outside the ear', 'biological_role': 'Transcriptional regulator with sensory-neuron maturation/TrkA mechanism', 'primary_sources': 'https://www.nature.com/articles/s41467-022-31580-1; https://pmc.ncbi.nlm.nih.gov/articles/PMC1157008/', 'uniprot_url': '', 'red_recommended': False, 'highlight_basis': 'Mechanistic sensory-neuron candidate', 'limitations': 'Inferred SGN regulon and DRG function are not direct VGN fate validation', 'selection_context': 'I/VGN/up: rank=3, fitted_delta=0.73866, observed_delta=0.65439, local_fraction=0.31', 'selection_reason': 'Significant whole-tree association; fitted and observed local directions agree; selected to display transcriptional regulator with sensory-neuron maturation/trka mechanism. The stated protein mechanism motivates follow-up, not a causal claim from the dot plot.', 'direction_interpretation': 'Current fixed-tree local expression direction; pooled age/branch expression in literature is not automatically comparable', 'review_date': '2026-10-22', 'regulatory_class': 'Source-catalogue transcription factor/regulator', 'trajectory_label_scope': 'CVG to VGN denotes the VGN-candidate-enriched arm, not validated VGN identity for every cell'}, 'Zfp536': {'evidence_scope': 'Mouse P19 neuronal differentiation; human ortholog expression/function', 'biological_role': 'Retinoic-acid transcriptional response and neuronal differentiation regulator', 'primary_sources': 'https://pmc.ncbi.nlm.nih.gov/articles/PMC2698762/', 'uniprot_url': '', 'red_recommended': False, 'highlight_basis': 'Specific neuronal regulatory mechanism', 'limitations': 'Overexpression inhibited RA-induced differentiation in P19 cells; does not predict native VGN RNA must fall', 'selection_context': 'I/VGN/up: rank=4, fitted_delta=0.58252, observed_delta=0.44177, local_fraction=0.349; I/Ia/down: rank=18, fitted_delta=-0.18221, observed_delta=-0.065604, local_fraction=0.1092', 'selection_reason': 'Significant whole-tree association; fitted and observed local directions agree; selected to display retinoic-acid transcriptional response and neuronal differentiation regulator. The stated protein mechanism motivates follow-up, not a causal claim from the dot plot.', 'direction_interpretation': 'Current fixed-tree local expression direction; pooled age/branch expression in literature is not automatically comparable', 'review_date': '2026-10-22', 'regulatory_class': 'Source-catalogue transcription factor/regulator', 'trajectory_label_scope': 'CVG to VGN denotes the VGN-candidate-enriched arm, not validated VGN identity for every cell'}, 'Tbx2': {'evidence_scope': 'Mouse E10.5 otic epithelial vs neuroblast transcriptomes', 'biological_role': 'Otic epithelial/progenitor-state transcriptional reference', 'primary_sources': 'https://pmc.ncbi.nlm.nih.gov/articles/PMC10592509/', 'uniprot_url': '', 'red_recommended': False, 'highlight_basis': 'Direct early otic-state expression', 'limitations': 'Not Tbx1; not an established VGN-specific neuronal fate regulator', 'selection_context': 'I/VGN/down: rank=1, fitted_delta=-0.73774, observed_delta=-0.66004, local_fraction=0.4492', 'selection_reason': 'Significant whole-tree association; fitted and observed local directions agree; selected to display otic epithelial/progenitor-state transcriptional reference. The stated protein mechanism motivates follow-up, not a causal claim from the dot plot.', 'direction_interpretation': 'Current fixed-tree local expression direction; pooled age/branch expression in literature is not automatically comparable', 'review_date': '2026-10-22', 'regulatory_class': 'Source-catalogue transcription factor/regulator', 'trajectory_label_scope': 'CVG to VGN denotes the VGN-candidate-enriched arm, not validated VGN identity for every cell'}, 'E2f1': {'evidence_scope': 'General mouse curated cell-cycle regulator', 'biological_role': 'RB/E2F-dependent cell-cycle transcription; proliferative-state candidate', 'primary_sources': 'https://www.ncbi.nlm.nih.gov/protein/2494227; https://pmc.ncbi.nlm.nih.gov/articles/PMC6673327/', 'uniprot_url': 'https://www.uniprot.org/uniprotkb/Q61501/entry', 'red_recommended': False, 'highlight_basis': 'Mechanistically interpretable candidate; no automatic red for any ear association', 'limitations': 'Decline can reflect cell-cycle exit/composition; not a VGN identity marker Mature-neuron E2F1 can also suppress inappropriate cell-cycle reentry; its function is context dependent.', 'selection_context': 'I/VGN/down: rank=6, fitted_delta=-0.51608, observed_delta=-0.52605, local_fraction=0.2923', 'selection_reason': 'Significant whole-tree association; fitted and observed local directions agree; selected to display rb/e2f-dependent cell-cycle transcription; proliferative-state candidate. The stated protein mechanism motivates follow-up, not a causal claim from the dot plot.', 'direction_interpretation': 'Current fixed-tree local expression direction; pooled age/branch expression in literature is not automatically comparable', 'review_date': '2026-10-22', 'regulatory_class': 'Source-catalogue transcription factor/regulator', 'trajectory_label_scope': 'CVG to VGN denotes the VGN-candidate-enriched arm, not validated VGN identity for every cell'}, 'Myt1l': {'evidence_scope': 'Mouse neuronal identity perturbation and SGN-like reprogramming', 'biological_role': 'Repression of non-neuronal programs and maintenance of neuronal identity', 'primary_sources': 'https://www.nature.com/articles/nature21722; https://pmc.ncbi.nlm.nih.gov/articles/PMC11969255/', 'uniprot_url': 'https://www.uniprot.org/uniprotkb/P97500/entry', 'red_recommended': False, 'highlight_basis': 'Mechanistically interpretable candidate; no automatic red for any ear association', 'limitations': 'Pan-neuronal identity mechanism does not establish SGN/VGN choice or a subtype-specific role', 'selection_context': 'I/T/up: rank=10, fitted_delta=1.1995, observed_delta=0.59341, local_fraction=0.7727', 'selection_reason': 'Significant whole-tree association; fitted and observed local directions agree; selected to display repression of non-neuronal programs and maintenance of neuronal identity. The stated protein mechanism motivates follow-up, not a causal claim from the dot plot.', 'direction_interpretation': 'Current fixed-tree local expression direction; pooled age/branch expression in literature is not automatically comparable', 'review_date': '2026-10-22', 'regulatory_class': 'Source-catalogue transcription factor/regulator', 'trajectory_label_scope': 'Fixed project SGN annotation/tree; local expression association'}, 'Thra': {'evidence_scope': 'Mouse E18.5 spiral ganglion in situ expression', 'biological_role': 'Thyroid-hormone receptor and maturation-associated transcriptional candidate', 'primary_sources': 'https://academic.oup.com/endo/article-pdf/153/3/1548/10390603/endo1548.pdf', 'uniprot_url': '', 'red_recommended': False, 'highlight_basis': 'Direct SG expression and defined hormone-receptor mechanism', 'limitations': 'Probe covers TR-alpha isoforms; no current early CVG-to-SGN trajectory causality established', 'selection_context': 'I/T/up: rank=1, fitted_delta=1.9047, observed_delta=1.0564, local_fraction=0.8575; I/Ia/up: rank=4, fitted_delta=0.051232, observed_delta=0.065314, local_fraction=0.02307', 'selection_reason': 'Significant whole-tree association; fitted and observed local directions agree; selected to display thyroid-hormone receptor and maturation-associated transcriptional candidate. The stated protein mechanism motivates follow-up, not a causal claim from the dot plot. Effect size and study/source sensitivity must be read alongside rank and significance.', 'direction_interpretation': 'Current fixed-tree local expression direction; pooled age/branch expression in literature is not automatically comparable', 'review_date': '2026-10-22', 'regulatory_class': 'Source-catalogue transcription factor/regulator', 'trajectory_label_scope': 'Fixed project SGN annotation/tree; local expression association'}, 'Esrrg': {'evidence_scope': 'Mouse E15.5-P30 SGN expression and P1 in situ; hearing genetics', 'biological_role': 'Nuclear-receptor transcriptional candidate with SGN expression', 'primary_sources': 'https://pmc.ncbi.nlm.nih.gov/articles/PMC7299348/; https://pmc.ncbi.nlm.nih.gov/articles/PMC4330334/', 'uniprot_url': '', 'red_recommended': False, 'highlight_basis': 'Direct SGN expression with broader hearing function', 'limitations': 'No current-segment fate function established; pooled SGN constancy does not imply every segment is flat', 'selection_context': 'I/T/up: rank=9, fitted_delta=1.2275, observed_delta=0.82928, local_fraction=0.6316; I/II/down: rank=1, fitted_delta=-0.91144, observed_delta=-0.20311, local_fraction=0.469', 'selection_reason': 'Significant whole-tree association; fitted and observed local directions agree; selected to display nuclear-receptor transcriptional candidate with sgn expression. The stated protein mechanism motivates follow-up, not a causal claim from the dot plot.', 'direction_interpretation': 'Current fixed-tree local expression direction; pooled age/branch expression in literature is not automatically comparable', 'review_date': '2026-10-22', 'regulatory_class': 'Source-catalogue transcription factor/regulator', 'trajectory_label_scope': 'Fixed project SGN annotation/tree; local expression association'}, 'Hes6': {'evidence_scope': 'Early mouse CVG expression; general proneural mechanism', 'biological_role': 'Early neuronal differentiation-associated bHLH regulatory candidate', 'primary_sources': 'https://doi.org/10.1016/j.celrep.2022.110542; https://pmc.ncbi.nlm.nih.gov/articles/PMC193938/', 'uniprot_url': '', 'red_recommended': False, 'highlight_basis': 'Direct early CVG expression plus broader neuronal mechanism', 'limitations': 'HES1 antagonism is mainly established outside the ear; no current branch-choice causality', 'selection_context': 'I/T/down: rank=6, fitted_delta=-1.8321, observed_delta=-1.1238, local_fraction=0.9439', 'selection_reason': 'Significant whole-tree association; fitted and observed local directions agree; selected to display early neuronal differentiation-associated bhlh regulatory candidate. The stated protein mechanism motivates follow-up, not a causal claim from the dot plot.', 'direction_interpretation': 'Current fixed-tree local expression direction; pooled age/branch expression in literature is not automatically comparable', 'review_date': '2026-10-22', 'regulatory_class': 'Source-catalogue transcription factor/regulator', 'trajectory_label_scope': 'Fixed project SGN annotation/tree; local expression association'}, 'Zbtb18': {'evidence_scope': 'Mouse cortical progenitor/neuron functional annotation', 'biological_role': 'Transcriptional repression, progenitor division and postmitotic-neuron survival', 'primary_sources': '', 'uniprot_url': 'https://www.uniprot.org/uniprotkb/Q9WUK6/entry', 'red_recommended': False, 'highlight_basis': 'Specific neurodevelopmental regulatory candidate', 'limitations': 'Cortical evidence does not validate inner-ear function or dictate RNA direction', 'selection_context': 'I/T/down: rank=8, fitted_delta=-1.7068, observed_delta=-1.1767, local_fraction=0.9738', 'selection_reason': 'Significant whole-tree association; fitted and observed local directions agree; selected to display transcriptional repression, progenitor division and postmitotic-neuron survival. The stated protein mechanism motivates follow-up, not a causal claim from the dot plot.', 'direction_interpretation': 'Current fixed-tree local expression direction; pooled age/branch expression in literature is not automatically comparable', 'review_date': '2026-10-22', 'regulatory_class': 'Source-catalogue transcription factor/regulator', 'trajectory_label_scope': 'Fixed project SGN annotation/tree; local expression association'}, 'Mafb': {'evidence_scope': 'Mouse SGN developmental expression and synaptogenesis function', 'biological_role': 'Auditory-neuron differentiation and synaptic maturation regulator', 'primary_sources': 'https://pmc.ncbi.nlm.nih.gov/articles/PMC7299348/; https://www.nature.com/articles/s41467-022-31580-1', 'uniprot_url': '', 'red_recommended': True, 'highlight_basis': 'Established SGN developmental regulator', 'limitations': 'Expression is age/subtype dependent; current association does not prove branch causality', 'selection_context': 'I/AC/up: rank=1, fitted_delta=1.5161, observed_delta=1.1594, local_fraction=0.4678', 'selection_reason': 'Significant whole-tree association; fitted and observed local directions agree; selected to display auditory-neuron differentiation and synaptic maturation regulator. The stated protein mechanism motivates follow-up, not a causal claim from the dot plot.', 'direction_interpretation': 'Current fixed-tree local expression direction; pooled age/branch expression in literature is not automatically comparable', 'review_date': '2026-10-22', 'regulatory_class': 'Source-catalogue transcription factor/regulator', 'trajectory_label_scope': 'Fixed project SGN annotation/tree; local expression association'}, 'Prox1': {'evidence_scope': 'Mouse Type-I/II expression and developmental SGN programs', 'biological_role': 'Homeobox transcriptional regulator; Type-I maturation-state reference', 'primary_sources': 'https://www.nature.com/articles/s41467-022-31580-1; https://journals.plos.org/plosone/article?id=10.1371/journal.pone.0170568', 'uniprot_url': '', 'red_recommended': True, 'highlight_basis': 'Established marker/regulatory reference; evidence class stated separately', 'limitations': 'Neuron-specific versus epithelial knockout effects differ; do not infer neuron-autonomous Type-II turning from broad knockout', 'selection_context': 'I/AC/up: rank=2, fitted_delta=0.98898, observed_delta=0.7745, local_fraction=0.752; I/Ia/up: rank=3, fitted_delta=0.082896, observed_delta=0.14743, local_fraction=0.06303', 'selection_reason': 'Significant whole-tree association; fitted and observed local directions agree; selected to display homeobox transcriptional regulator; type-i maturation-state reference. The stated protein mechanism motivates follow-up, not a causal claim from the dot plot. Effect size and study/source sensitivity must be read alongside rank and significance.', 'direction_interpretation': 'Current fixed-tree local expression direction; pooled age/branch expression in literature is not automatically comparable', 'review_date': '2026-10-22', 'regulatory_class': 'Source-catalogue transcription factor/regulator', 'trajectory_label_scope': 'Fixed project SGN annotation/tree; local expression association'}, 'Thrb': {'evidence_scope': 'Mouse cochlear receptor expression and thyroid developmental genetics', 'biological_role': 'Thyroid-hormone nuclear receptor; auditory maturation regulation', 'primary_sources': 'https://academic.oup.com/endo/article-pdf/153/3/1548/10390603/endo1548.pdf; https://pmc.ncbi.nlm.nih.gov/articles/PMC2659284/', 'uniprot_url': '', 'red_recommended': False, 'highlight_basis': 'Mechanistically interpretable candidate; no automatic red for any ear association', 'limitations': 'E18.5 spiral ganglion signal weaker than organ of Corti; current AC rise/Ia decline not causal fate proof', 'selection_context': 'I/AC/up: rank=3, fitted_delta=0.89485, observed_delta=0.77265, local_fraction=0.9427; I/Ia/down: rank=50, fitted_delta=-0.13417, observed_delta=-0.056774, local_fraction=0.1413', 'selection_reason': 'Significant whole-tree association; fitted and observed local directions agree; selected to display thyroid-hormone nuclear receptor; auditory maturation regulation. The stated protein mechanism motivates follow-up, not a causal claim from the dot plot.', 'direction_interpretation': 'Current fixed-tree local expression direction; pooled age/branch expression in literature is not automatically comparable', 'review_date': '2026-10-22', 'regulatory_class': 'Source-catalogue transcription factor/regulator', 'trajectory_label_scope': 'Fixed project SGN annotation/tree; local expression association'}, 'Casz1': {'evidence_scope': 'Mouse differentiating CVG/SGN/VGN expression and lineage context', 'biological_role': 'Zinc-finger transcriptional regulator associated with neuronal differentiation', 'primary_sources': 'https://doi.org/10.1016/j.celrep.2022.110542; https://www.nature.com/articles/s41467-022-31580-1', 'uniprot_url': '', 'red_recommended': False, 'highlight_basis': 'Mechanistically interpretable candidate; no automatic red for any ear association', 'limitations': 'Expressed in both SGN and VGN; current II decrease is not proof of an inhibitory II fate role', 'selection_context': 'I/AC/up: rank=11, fitted_delta=0.446, observed_delta=0.2339, local_fraction=0.2617; I/II/down: rank=3, fitted_delta=-0.67135, observed_delta=-0.49573, local_fraction=0.3939', 'selection_reason': 'Significant whole-tree association; fitted and observed local directions agree; selected to display zinc-finger transcriptional regulator associated with neuronal differentiation. The stated protein mechanism motivates follow-up, not a causal claim from the dot plot.', 'direction_interpretation': 'Current fixed-tree local expression direction; pooled age/branch expression in literature is not automatically comparable', 'review_date': '2026-10-22', 'regulatory_class': 'Source-catalogue transcription factor/regulator', 'trajectory_label_scope': 'Fixed project SGN annotation/tree; local expression association'}, 'Etv4': {'evidence_scope': 'Developing mouse SGN trajectory/regulon and Type II expression', 'biological_role': 'Developmental/Type-II-associated transcription-factor reference', 'primary_sources': 'https://www.nature.com/articles/s41467-022-31580-1; https://pmc.ncbi.nlm.nih.gov/articles/PMC9860252/', 'uniprot_url': '', 'red_recommended': True, 'highlight_basis': 'Established developmental-state/II reference', 'limitations': 'Regulon activity and Etv4 transcript level are distinct measurements', 'selection_context': 'I/AC/down: rank=3, fitted_delta=-0.93674, observed_delta=-0.30812, local_fraction=0.4973; I/AB/down: rank=1, fitted_delta=-1.3968, observed_delta=-0.9889, local_fraction=0.7416; I/II/up: rank=1, fitted_delta=0.87891, observed_delta=0.77387, local_fraction=0.4666', 'selection_reason': 'Significant whole-tree association; fitted and observed local directions agree; selected to display developmental/type-ii-associated transcription-factor reference. The stated protein mechanism motivates follow-up, not a causal claim from the dot plot.', 'direction_interpretation': 'Current fixed-tree local expression direction; pooled age/branch expression in literature is not automatically comparable', 'review_date': '2026-10-22', 'regulatory_class': 'Source-catalogue transcription factor/regulator', 'trajectory_label_scope': 'Fixed project SGN annotation/tree; local expression association'}, 'Sox11': {'evidence_scope': 'Mouse E14.5 SGN in situ and SoxC inner-ear study', 'biological_role': 'SOXC neuronal differentiation-associated transcription factor', 'primary_sources': 'https://pmc.ncbi.nlm.nih.gov/articles/PMC4653225/', 'uniprot_url': '', 'red_recommended': False, 'highlight_basis': 'Mechanistically interpretable candidate; no automatic red for any ear association', 'limitations': 'Principal ear functional experiments concern hair cells; no SGN branch-specific perturbation claimed', 'selection_context': 'I/AC/down: rank=4, fitted_delta=-0.66081, observed_delta=-0.31549, local_fraction=0.5931; I/AB/down: rank=14, fitted_delta=-0.49895, observed_delta=-0.2289, local_fraction=0.4478', 'selection_reason': 'Significant whole-tree association; fitted and observed local directions agree; selected to display soxc neuronal differentiation-associated transcription factor. The stated protein mechanism motivates follow-up, not a causal claim from the dot plot.', 'direction_interpretation': 'Current fixed-tree local expression direction; pooled age/branch expression in literature is not automatically comparable', 'review_date': '2026-10-22', 'regulatory_class': 'Source-catalogue transcription factor/regulator', 'trajectory_label_scope': 'Fixed project SGN annotation/tree; local expression association'}, 'Irx1': {'evidence_scope': 'Mouse developing otic vesicle and neural regional expression', 'biological_role': 'Iroquois homeobox transcriptional patterning candidate', 'primary_sources': 'https://pmc.ncbi.nlm.nih.gov/articles/PMC2013721/', 'uniprot_url': 'https://www.uniprot.org/uniprotkb/P81068/entry', 'red_recommended': False, 'highlight_basis': 'Mechanistically interpretable candidate; no automatic red for any ear association', 'limitations': 'Whole-ear/regional expression is not an SGN AC-specific perturbation or matching trajectory direction', 'selection_context': 'I/AC/down: rank=5, fitted_delta=-0.62179, observed_delta=-0.46203, local_fraction=0.3194', 'selection_reason': 'Significant whole-tree association; fitted and observed local directions agree; selected to display iroquois homeobox transcriptional patterning candidate. The stated protein mechanism motivates follow-up, not a causal claim from the dot plot.', 'direction_interpretation': 'Current fixed-tree local expression direction; pooled age/branch expression in literature is not automatically comparable', 'review_date': '2026-10-22', 'regulatory_class': 'Source-catalogue transcription factor/regulator', 'trajectory_label_scope': 'Fixed project SGN annotation/tree; local expression association'}, 'Rxrg': {'evidence_scope': 'Mouse SGN subtype expression and inferred developmental regulon', 'biological_role': 'Retinoid-X nuclear receptor; auditory subtype expression reference', 'primary_sources': 'https://www.nature.com/articles/s41467-022-31580-1; https://pmc.ncbi.nlm.nih.gov/articles/PMC9860252/; https://www.nature.com/articles/s41467-018-06033-3', 'uniprot_url': '', 'red_recommended': True, 'highlight_basis': 'Established marker/regulatory reference; evidence class stated separately', 'limitations': 'Early regulon spans alternative branches; mature identity labels in 2018 require nomenclature coordination', 'selection_context': 'I/Ib/up: rank=1, fitted_delta=0.74324, observed_delta=0.53252, local_fraction=0.5216', 'selection_reason': 'Significant whole-tree association; fitted and observed local directions agree; selected to display retinoid-x nuclear receptor; auditory subtype expression reference. The stated protein mechanism motivates follow-up, not a causal claim from the dot plot.', 'direction_interpretation': 'Current fixed-tree local expression direction; pooled age/branch expression in literature is not automatically comparable', 'review_date': '2026-10-22', 'regulatory_class': 'Source-catalogue transcription factor/regulator', 'trajectory_label_scope': 'Fixed project SGN annotation/tree; local expression association'}, 'Prrx2': {'evidence_scope': 'Mouse developing SGN expression/trajectory', 'biological_role': 'Paired-related homeobox transcriptional candidate', 'primary_sources': 'https://www.nature.com/articles/s41467-022-31580-1', 'uniprot_url': '', 'red_recommended': False, 'highlight_basis': 'Mechanistically interpretable candidate; no automatic red for any ear association', 'limitations': 'Current Ia/Ib/II increases are association; no unique Ib identity or gene-specific SGN fate proof', 'selection_context': 'I/Ib/up: rank=2, fitted_delta=0.70756, observed_delta=0.49959, local_fraction=0.4848; I/Ia/up: rank=1, fitted_delta=0.21868, observed_delta=0.39397, local_fraction=0.1498; I/II/up: rank=2, fitted_delta=0.49211, observed_delta=0.16309, local_fraction=0.3372', 'selection_reason': 'Significant whole-tree association; fitted and observed local directions agree; selected to display paired-related homeobox transcriptional candidate. The stated protein mechanism motivates follow-up, not a causal claim from the dot plot.', 'direction_interpretation': 'Current fixed-tree local expression direction; pooled age/branch expression in literature is not automatically comparable', 'review_date': '2026-10-22', 'regulatory_class': 'Source-catalogue transcription factor/regulator', 'trajectory_label_scope': 'Fixed project SGN annotation/tree; local expression association'}, 'Pou4f2': {'evidence_scope': 'Mouse embryonic CVG protein and direct regulatory context', 'biological_role': 'POU-homeodomain neuronal differentiation regulator', 'primary_sources': 'https://doi.org/10.1016/j.gep.2014.03.001; https://pmc.ncbi.nlm.nih.gov/articles/PMC10883659/', 'uniprot_url': '', 'red_recommended': False, 'highlight_basis': 'Mechanistically interpretable candidate; no automatic red for any ear association', 'limitations': 'Not established as an Ib-specific marker or causal subtype regulator', 'selection_context': 'I/Ib/up: rank=5, fitted_delta=0.63582, observed_delta=0.40994, local_fraction=0.5594', 'selection_reason': 'Significant whole-tree association; fitted and observed local directions agree; selected to display pou-homeodomain neuronal differentiation regulator. The stated protein mechanism motivates follow-up, not a causal claim from the dot plot.', 'direction_interpretation': 'Current fixed-tree local expression direction; pooled age/branch expression in literature is not automatically comparable', 'review_date': '2026-10-22', 'regulatory_class': 'Source-catalogue transcription factor/regulator', 'trajectory_label_scope': 'Fixed project SGN annotation/tree; local expression association'}, 'Meis3': {'evidence_scope': 'Mouse TALE homeobox annotation; Xenopus neural-patterning experiments', 'biological_role': 'MEIS/PBX-associated regional transcriptional regulation candidate', 'primary_sources': 'https://pubmed.ncbi.nlm.nih.gov/22399680/; https://pmc.ncbi.nlm.nih.gov/articles/PMC3188567/', 'uniprot_url': 'https://www.uniprot.org/uniprotkb/P97368/entry', 'red_recommended': False, 'highlight_basis': 'Mechanistically interpretable candidate; no automatic red for any ear association', 'limitations': 'Functional neural-patterning evidence is Xenopus; no SGN Ib-specific role or matching direction established', 'selection_context': 'I/Ib/up: rank=6, fitted_delta=0.63575, observed_delta=0.44507, local_fraction=0.6904', 'selection_reason': 'Significant whole-tree association; fitted and observed local directions agree; selected to display meis/pbx-associated regional transcriptional regulation candidate. The stated protein mechanism motivates follow-up, not a causal claim from the dot plot.', 'direction_interpretation': 'Current fixed-tree local expression direction; pooled age/branch expression in literature is not automatically comparable', 'review_date': '2026-10-22', 'regulatory_class': 'Source-catalogue transcription factor/regulator', 'trajectory_label_scope': 'Fixed project SGN annotation/tree; local expression association'}, 'Bcl11a': {'evidence_scope': 'Mouse cortical projection-neuron genetic studies outside ear', 'biological_role': 'Zinc-finger transcriptional regulator of neuronal differentiation and connectivity', 'primary_sources': 'https://doi.org/10.1093/cercor/bhab437', 'uniprot_url': '', 'red_recommended': False, 'highlight_basis': 'Mechanistically interpretable candidate; no automatic red for any ear association', 'limitations': 'No targeted SGN developmental report identified in this review; cortical effects do not establish SGN-subtype causality', 'selection_context': 'I/Ib/down: rank=7, fitted_delta=-0.31469, observed_delta=-0.1492, local_fraction=0.1647; I/Ic/down: rank=4, fitted_delta=-0.24839, observed_delta=-0.16854, local_fraction=0.13', 'selection_reason': 'Significant whole-tree association; fitted and observed local directions agree; selected to display zinc-finger transcriptional regulator of neuronal differentiation and connectivity. The stated protein mechanism motivates follow-up, not a causal claim from the dot plot.', 'direction_interpretation': 'Current fixed-tree local expression direction; pooled age/branch expression in literature is not automatically comparable', 'review_date': '2026-10-22', 'regulatory_class': 'Source-catalogue transcription factor/regulator', 'trajectory_label_scope': 'Fixed project SGN annotation/tree; local expression association'}, 'Id1': {'evidence_scope': 'Mouse developing SGN expression/module', 'biological_role': 'HLH inhibitor of bHLH DNA-binding activity; early-state regulatory reference', 'primary_sources': 'https://www.nature.com/articles/s41467-022-31580-1', 'uniprot_url': '', 'red_recommended': True, 'highlight_basis': 'Established marker/regulatory reference; evidence class stated separately', 'limitations': 'Original early Ia/II association is not Ic fate validation; ID proteins are non-DNA-binding transcriptional regulators', 'selection_context': 'I/Ic/up: rank=3, fitted_delta=0.26326, observed_delta=0.71809, local_fraction=0.1106', 'selection_reason': 'Significant whole-tree association; fitted and observed local directions agree; selected to display hlh inhibitor of bhlh dna-binding activity; early-state regulatory reference. The stated protein mechanism motivates follow-up, not a causal claim from the dot plot.', 'direction_interpretation': 'Current fixed-tree local expression direction; pooled age/branch expression in literature is not automatically comparable', 'review_date': '2026-10-22', 'regulatory_class': 'Non-DNA-binding HLH inhibitor of bHLH transcription factors', 'trajectory_label_scope': 'Fixed project SGN annotation/tree; local expression association'}, 'Maf': {'evidence_scope': 'Mouse SGN gene perturbation and synaptic phenotype', 'biological_role': 'Transcriptional regulation of subtype-appropriate auditory-neuron synaptic properties', 'primary_sources': 'https://pubmed.ncbi.nlm.nih.gov/40482032/', 'uniprot_url': '', 'red_recommended': True, 'highlight_basis': 'Established marker/regulatory reference; evidence class stated separately', 'limitations': 'Current Ic rise is not by itself a causal measurement', 'selection_context': 'I/Ic/up: rank=4, fitted_delta=0.25532, observed_delta=0.52476, local_fraction=0.1072', 'selection_reason': 'Significant whole-tree association; fitted and observed local directions agree; selected to display transcriptional regulation of subtype-appropriate auditory-neuron synaptic properties. The stated protein mechanism motivates follow-up, not a causal claim from the dot plot.', 'direction_interpretation': 'Current fixed-tree local expression direction; pooled age/branch expression in literature is not automatically comparable', 'review_date': '2026-10-22', 'regulatory_class': 'Source-catalogue transcription factor/regulator', 'trajectory_label_scope': 'Fixed project SGN annotation/tree; local expression association'}, 'Tshz2': {'evidence_scope': 'Mouse developing SGN expression/module', 'biological_role': 'Teashirt zinc-finger developmental transcriptional candidate', 'primary_sources': 'https://www.nature.com/articles/s41467-022-31580-1', 'uniprot_url': '', 'red_recommended': False, 'highlight_basis': 'Mechanistically interpretable candidate; no automatic red for any ear association', 'limitations': 'Original early Ia/II-associated module does not establish current Ic-specific fate regulation', 'selection_context': 'I/Ic/up: rank=2, fitted_delta=0.39724, observed_delta=0.24356, local_fraction=0.1661', 'selection_reason': 'Significant whole-tree association; fitted and observed local directions agree; selected to display teashirt zinc-finger developmental transcriptional candidate. The stated protein mechanism motivates follow-up, not a causal claim from the dot plot.', 'direction_interpretation': 'Current fixed-tree local expression direction; pooled age/branch expression in literature is not automatically comparable', 'review_date': '2026-10-22', 'regulatory_class': 'Source-catalogue transcription factor/regulator', 'trajectory_label_scope': 'Fixed project SGN annotation/tree; local expression association'}, 'Jarid2': {'evidence_scope': 'Mouse ES-cell PRC2 recruitment and differentiation experiments', 'biological_role': 'PRC2 accessory chromatin regulator controlling developmental-gene repression', 'primary_sources': 'https://pmc.ncbi.nlm.nih.gov/articles/PMC3572404/; https://www.uniprot.org/citations/20075857', 'uniprot_url': 'https://www.uniprot.org/uniprotkb/Q62315/entry', 'red_recommended': False, 'highlight_basis': 'Mechanistically interpretable candidate; no automatic red for any ear association', 'limitations': 'Chromatin regulator in source TF catalogue; no SGN Ic-specific role reported; not a histone demethylase', 'selection_context': 'I/Ic/up: rank=8, fitted_delta=0.19868, observed_delta=0.26135, local_fraction=0.108', 'selection_reason': 'Significant whole-tree association; fitted and observed local directions agree; selected to display prc2 accessory chromatin regulator controlling developmental-gene repression. The stated protein mechanism motivates follow-up, not a causal claim from the dot plot. Effect size and study/source sensitivity must be read alongside rank and significance.', 'direction_interpretation': 'Current fixed-tree local expression direction; pooled age/branch expression in literature is not automatically comparable', 'review_date': '2026-10-22', 'regulatory_class': 'PRC2 accessory chromatin regulator; DNA/chromatin binding, not a canonical sequence-specific fate TF', 'trajectory_label_scope': 'Fixed project SGN annotation/tree; local expression association'}, 'Mycn': {'evidence_scope': 'Mouse E15-E18 SG/auditory nerve in situ expression', 'biological_role': 'MYC-family growth/differentiation transcriptional regulator', 'primary_sources': 'https://pubmed.ncbi.nlm.nih.gov/8150745/', 'uniprot_url': 'https://www.uniprot.org/uniprotkb/P03966/entry', 'red_recommended': False, 'highlight_basis': 'Mechanistically interpretable candidate; no automatic red for any ear association', 'limitations': 'Original days are reported as gestational days; no selective Ic perturbation; expression later becomes weaker in nervous structures', 'selection_context': 'I/Ic/down: rank=15, fitted_delta=-0.10368, observed_delta=-0.16638, local_fraction=0.1428', 'selection_reason': 'Significant whole-tree association; fitted and observed local directions agree; selected to display myc-family growth/differentiation transcriptional regulator. The stated protein mechanism motivates follow-up, not a causal claim from the dot plot.', 'direction_interpretation': 'Current fixed-tree local expression direction; pooled age/branch expression in literature is not automatically comparable', 'review_date': '2026-10-22', 'regulatory_class': 'Source-catalogue transcription factor/regulator', 'trajectory_label_scope': 'Fixed project SGN annotation/tree; local expression association'}, 'Jun': {'evidence_scope': 'Mouse SGN precursor expression, Sanders Fig1D; AP-1 protein function', 'biological_role': 'Immediate-early AP-1 transcriptional response regulator', 'primary_sources': 'https://pmc.ncbi.nlm.nih.gov/articles/PMC9860252/; https://www.nature.com/articles/nmeth.4437', 'uniprot_url': 'https://www.uniprot.org/uniprotkb/P05627/entry', 'red_recommended': False, 'highlight_basis': 'Mechanistically interpretable candidate; no automatic red for any ear association', 'limitations': 'Activity/stress and source-sensitive; Ic decline must not be called a demonstrated fate switch', 'selection_context': 'I/Ic/down: rank=2, fitted_delta=-0.27765, observed_delta=-0.33044, local_fraction=0.1624', 'selection_reason': 'Significant whole-tree association; fitted and observed local directions agree; selected to display immediate-early ap-1 transcriptional response regulator. The stated protein mechanism motivates follow-up, not a causal claim from the dot plot. Effect size and study/source sensitivity must be read alongside rank and significance.', 'direction_interpretation': 'Current fixed-tree local expression direction; pooled age/branch expression in literature is not automatically comparable', 'review_date': '2026-10-22', 'regulatory_class': 'Source-catalogue transcription factor/regulator', 'trajectory_label_scope': 'Fixed project SGN annotation/tree; local expression association'}, 'Pbx1': {'evidence_scope': 'Mouse SGN early-Ia/Ib transcriptomic module', 'biological_role': 'TALE-homeobox transcriptional cofactor; developmental-state candidate', 'primary_sources': 'https://www.nature.com/articles/s41467-022-31580-1', 'uniprot_url': '', 'red_recommended': False, 'highlight_basis': 'Mechanistically interpretable candidate; no automatic red for any ear association', 'limitations': 'Transient early expression association does not establish AB branch control', 'selection_context': 'I/AB/up: rank=9, fitted_delta=0.59401, observed_delta=0.53837, local_fraction=0.6342', 'selection_reason': 'Significant whole-tree association; fitted and observed local directions agree; selected to display tale-homeobox transcriptional cofactor; developmental-state candidate. The stated protein mechanism motivates follow-up, not a causal claim from the dot plot.', 'direction_interpretation': 'Current fixed-tree local expression direction; pooled age/branch expression in literature is not automatically comparable', 'review_date': '2026-10-22', 'regulatory_class': 'Source-catalogue transcription factor/regulator', 'trajectory_label_scope': 'Fixed project SGN annotation/tree; local expression association'}, 'Smad9': {'evidence_scope': 'Mouse original SGN Fig2i and inferred BMP-related regulon', 'biological_role': 'BMP-associated SMAD transcriptional effector', 'primary_sources': 'https://www.nature.com/articles/s41467-022-31580-1', 'uniprot_url': '', 'red_recommended': False, 'highlight_basis': 'Mechanistically interpretable candidate; no automatic red for any ear association', 'limitations': 'Expression/regulon association does not establish direct SGN AB causal function', 'selection_context': 'I/AB/up: rank=16, fitted_delta=0.51895, observed_delta=0.34856, local_fraction=0.5805', 'selection_reason': 'Significant whole-tree association; fitted and observed local directions agree; selected to display bmp-associated smad transcriptional effector. The stated protein mechanism motivates follow-up, not a causal claim from the dot plot.', 'direction_interpretation': 'Current fixed-tree local expression direction; pooled age/branch expression in literature is not automatically comparable', 'review_date': '2026-10-22', 'regulatory_class': 'Source-catalogue transcription factor/regulator', 'trajectory_label_scope': 'Fixed project SGN annotation/tree; local expression association'}, 'Tshz3': {'evidence_scope': 'Mouse developing SGN trajectory expression', 'biological_role': 'Teashirt zinc-finger developmental transcriptional candidate', 'primary_sources': 'https://www.nature.com/articles/s41467-022-31580-1', 'uniprot_url': '', 'red_recommended': False, 'highlight_basis': 'Mechanistically interpretable candidate; no automatic red for any ear association', 'limitations': 'Original nascent-II association is expression evidence, not current AB fate validation', 'selection_context': 'I/AB/up: rank=22, fitted_delta=0.47907, observed_delta=0.34178, local_fraction=0.623', 'selection_reason': 'Significant whole-tree association; fitted and observed local directions agree; selected to display teashirt zinc-finger developmental transcriptional candidate. The stated protein mechanism motivates follow-up, not a causal claim from the dot plot.', 'direction_interpretation': 'Current fixed-tree local expression direction; pooled age/branch expression in literature is not automatically comparable', 'review_date': '2026-10-22', 'regulatory_class': 'Source-catalogue transcription factor/regulator', 'trajectory_label_scope': 'Fixed project SGN annotation/tree; local expression association'}, 'Zfhx3': {'evidence_scope': 'Mouse original developing SGN expression trajectory', 'biological_role': 'Zinc-finger homeobox transcriptional developmental-state candidate', 'primary_sources': 'https://www.nature.com/articles/s41467-022-31580-1', 'uniprot_url': '', 'red_recommended': False, 'highlight_basis': 'Mechanistically interpretable candidate; no automatic red for any ear association', 'limitations': 'Current AB increase is correlative; do not confuse with Zfhx4', 'selection_context': 'I/AB/up: rank=36, fitted_delta=0.3968, observed_delta=0.077656, local_fraction=0.212', 'selection_reason': 'Significant whole-tree association; fitted and observed local directions agree; selected to display zinc-finger homeobox transcriptional developmental-state candidate. The stated protein mechanism motivates follow-up, not a causal claim from the dot plot.', 'direction_interpretation': 'Current fixed-tree local expression direction; pooled age/branch expression in literature is not automatically comparable', 'review_date': '2026-10-22', 'regulatory_class': 'Source-catalogue transcription factor/regulator', 'trajectory_label_scope': 'Fixed project SGN annotation/tree; local expression association'}, 'Zeb2': {'evidence_scope': 'Mouse delaminating otic neuroblast expression', 'biological_role': 'Zinc-finger transcriptional regulator; epithelial-to-neuronal transition candidate', 'primary_sources': 'https://pmc.ncbi.nlm.nih.gov/articles/PMC10592509/', 'uniprot_url': '', 'red_recommended': False, 'highlight_basis': 'Mechanistically interpretable candidate; no automatic red for any ear association', 'limitations': 'Early EMT-associated expression is not a targeted SGN-subtype causal experiment', 'selection_context': 'I/AB/down: rank=7, fitted_delta=-0.72967, observed_delta=-0.31356, local_fraction=0.2748', 'selection_reason': 'Significant whole-tree association; fitted and observed local directions agree; selected to display zinc-finger transcriptional regulator; epithelial-to-neuronal transition candidate. The stated protein mechanism motivates follow-up, not a causal claim from the dot plot.', 'direction_interpretation': 'Current fixed-tree local expression direction; pooled age/branch expression in literature is not automatically comparable', 'review_date': '2026-10-22', 'regulatory_class': 'Source-catalogue transcription factor/regulator', 'trajectory_label_scope': 'Fixed project SGN annotation/tree; local expression association'}, 'Arid1a': {'evidence_scope': 'Mouse cortical neural stem/progenitor conditional knockout', 'biological_role': 'BAF/SWI-SNF chromatin remodeling; progenitor proliferation/differentiation regulator', 'primary_sources': 'https://pmc.ncbi.nlm.nih.gov/articles/PMC8560606/', 'uniprot_url': 'https://www.uniprot.org/uniprotkb/A2BH40/entry', 'red_recommended': False, 'highlight_basis': 'Mechanistically interpretable candidate; no automatic red for any ear association', 'limitations': 'Chromatin regulatory subunit in source TF catalogue; not an established sequence-specific SGN fate TF', 'selection_context': 'I/Ia/down: rank=16, fitted_delta=-0.18445, observed_delta=-0.066451, local_fraction=0.164', 'selection_reason': 'Significant whole-tree association; fitted and observed local directions agree; selected to display baf/swi-snf chromatin remodeling; progenitor proliferation/differentiation regulator. The stated protein mechanism motivates follow-up, not a causal claim from the dot plot. Effect size and study/source sensitivity must be read alongside rank and significance.', 'direction_interpretation': 'Current fixed-tree local expression direction; pooled age/branch expression in literature is not automatically comparable', 'review_date': '2026-10-22', 'regulatory_class': 'BAF/SWI-SNF chromatin-remodeling subunit; DNA binding is not sequence specific', 'trajectory_label_scope': 'Fixed project SGN annotation/tree; local expression association'}, 'Cbfb': {'evidence_scope': 'Mouse developing SGN marker; somatosensory neuronal genetics', 'biological_role': 'RUNX transcriptional cofactor; original Sanders Fig3E Type-II expression', 'primary_sources': 'https://pmc.ncbi.nlm.nih.gov/articles/PMC9860252/; https://pmc.ncbi.nlm.nih.gov/articles/PMC4657622/', 'uniprot_url': 'https://www.uniprot.org/uniprotkb/Q08024/entry', 'red_recommended': False, 'highlight_basis': 'Mechanistically interpretable candidate; no automatic red for any ear association', 'limitations': 'Transcriptional cofactor without independent sequence-specific DNA binding; no SGN-II causal claim', 'selection_context': 'I/II/up: rank=4, fitted_delta=0.38688, observed_delta=0.4721, local_fraction=0.2551', 'selection_reason': 'Significant whole-tree association; fitted and observed local directions agree; selected to display runx transcriptional cofactor; original sanders fig3e type-ii expression. The stated protein mechanism motivates follow-up, not a causal claim from the dot plot.', 'direction_interpretation': 'Current fixed-tree local expression direction; pooled age/branch expression in literature is not automatically comparable', 'review_date': '2026-10-22', 'regulatory_class': 'Non-DNA-binding RUNX transcriptional cofactor', 'trajectory_label_scope': 'Fixed project SGN annotation/tree; local expression association'}, 'Hey1': {'evidence_scope': 'Mouse inner-ear Notch/Hedgehog/prosensory experiments', 'biological_role': 'Notch-associated bHLH transcriptional repressor', 'primary_sources': 'https://pmc.ncbi.nlm.nih.gov/articles/PMC4166165/', 'uniprot_url': '', 'red_recommended': False, 'highlight_basis': 'Mechanistically interpretable candidate; no automatic red for any ear association', 'limitations': 'Epithelial/otic signaling evidence does not establish SGN-II-autonomous fate function', 'selection_context': 'I/II/up: rank=6, fitted_delta=0.30927, observed_delta=0.46175, local_fraction=0.3998', 'selection_reason': 'Significant whole-tree association; fitted and observed local directions agree; selected to display notch-associated bhlh transcriptional repressor. The stated protein mechanism motivates follow-up, not a causal claim from the dot plot.', 'direction_interpretation': 'Current fixed-tree local expression direction; pooled age/branch expression in literature is not automatically comparable', 'review_date': '2026-10-22', 'regulatory_class': 'Source-catalogue transcription factor/regulator', 'trajectory_label_scope': 'Fixed project SGN annotation/tree; local expression association'}, 'Zfhx4': {'evidence_scope': 'Mouse neuronal-differentiation cell-line expression and curated protein', 'biological_role': 'Zinc-finger homeobox neural transcriptional candidate', 'primary_sources': 'https://www.ncbi.nlm.nih.gov/gene/80892', 'uniprot_url': 'https://www.uniprot.org/uniprotkb/Q9JJN2/entry', 'red_recommended': False, 'highlight_basis': 'Mechanistically interpretable candidate; no automatic red for any ear association', 'limitations': 'Neuronal-differentiation-associated expression outside ear; no SGN-II perturbation or matching time-direction identified', 'selection_context': 'I/II/down: rank=7, fitted_delta=-0.54806, observed_delta=-0.13044, local_fraction=0.7079', 'selection_reason': 'Significant whole-tree association; fitted and observed local directions agree; selected to display zinc-finger homeobox neural transcriptional candidate. The stated protein mechanism motivates follow-up, not a causal claim from the dot plot.', 'direction_interpretation': 'Current fixed-tree local expression direction; pooled age/branch expression in literature is not automatically comparable', 'review_date': '2026-10-22', 'regulatory_class': 'Source-catalogue transcription factor/regulator', 'trajectory_label_scope': 'Fixed project SGN annotation/tree; local expression association'}, 'Mef2c': {'evidence_scope': 'Mouse hearing/SGN pathology with heterozygosity', 'biological_role': 'Transcriptional-regulatory candidate with auditory-neuron relevance', 'primary_sources': 'https://pmc.ncbi.nlm.nih.gov/articles/PMC9617620/', 'uniprot_url': '', 'red_recommended': False, 'highlight_basis': 'Direct auditory-neuron pathology context', 'limitations': 'Multicellular postnatal phenotypes do not establish early SGN-autonomous fate function', 'selection_context': 'I/II/down: rank=102, fitted_delta=-0.17178, observed_delta=-0.057493, local_fraction=0.1484', 'selection_reason': 'Significant whole-tree association; fitted and observed local directions agree; selected to display transcriptional-regulatory candidate with auditory-neuron relevance. The stated protein mechanism motivates follow-up, not a causal claim from the dot plot.', 'direction_interpretation': 'Current fixed-tree local expression direction; pooled age/branch expression in literature is not automatically comparable', 'review_date': '2026-10-22', 'regulatory_class': 'Source-catalogue transcription factor/regulator', 'trajectory_label_scope': 'Fixed project SGN annotation/tree; local expression association'}}

CONTEXT_REVIEW = {
    ('H', 'VGN', 'Bdnf'): 'Target-derived trophic requirements do not specify neuronal Bdnf RNA direction.',
    ('I', 'VGN', 'Gata3'): 'Early vestibular subpopulations can express Gata3; decline is stage/compartment dependent.',
    ('H', 'Ic', 'Gata3'): 'Established SGN developmental reference; not an Ic-exclusive mature marker.',
    ('H', 'Ic', 'Igfbpl1'): 'Developmental-state/SGN reference; the selected increase does not establish Ic specificity.',
}

BLOCK_PURPOSE = {
    'CVG': 'Neuronal differentiation and loss of progenitor/cell-cycle-associated programs in the early prebranch compartment',
    'VGN': 'Candidate vestibular-arm differentiation and axon/signaling changes after the first fork',
    'T': 'SGN-lineage progression and neuronal maturation after the CVG fork',
    'AC': 'Shared Ib/Ic neuronal and signaling changes before their terminal divergence',
    'Ib': 'Ib-related terminal differentiation, adhesion and subtype-reference changes',
    'Ic': 'Ic-related terminal differentiation, signaling and regulatory-state changes',
    'AB': 'Shared Ia/II axon-adhesion and regulatory changes before their terminal divergence',
    'Ia': 'Ia-related terminal changes; limited Early support and small local TF changes require caution',
    'II': 'Type-II-related terminal identity and neuronal signaling changes',
}

def select_panel(data, letter, choices, red):
    candidate = data['candidate']
    lookup = pd.Index(candidate['gene_symbols'])
    stats = data['stats'].set_index('gene', drop=False)
    records, relative = [], []
    for column, segment in enumerate(SEGMENTS):
        selected = choices[segment]
        assert len(selected['up']) + len(selected['down']) == 8
        for direction in ('up', 'down'):
            order = candidate[direction + '_order'][segment]
            if letter == 'I':
                order = order[candidate['is_tf'][order]]
            ranks = {int(value): index + 1 for index, value in enumerate(order)}
            for gene in selected[direction]:
                index = int(lookup.get_loc(gene))
                values = np.asarray(candidate['mean_expression'][index], dtype=float)
                early, late = map(float, values[2 * column:2 * column + 2])
                change = float(candidate['fitted_change'][index, column])
                amplitude = float(candidate['fit_amplitude'][index])
                fraction = abs(change) / amplitude if amplitude else 0.
                sign = 1 if direction == 'up' else -1
                assert sign * change > 0 and sign * (late - early) > 0
                assert index in ranks and bool(stats.loc[gene, 'signi'])
                assert letter != 'I' or bool(candidate['is_tf'][index])
                assert not gene.startswith(('Rpl', 'Rps', 'mt-', 'MT-'))
                evidence = EVIDENCE[gene]
                provisional = fraction < .1 or gene == 'Jun'
                reason = []
                if fraction < .1:
                    reason.append('Local fitted change is <10% of the whole-tree fitted amplitude')
                if gene == 'Jun':
                    reason.append('AP-1 / immediate-early activity may reflect stress or source composition')
                context = CONTEXT_REVIEW.get((letter, segment, gene), '')
                record = dict(panel=letter, selected_segment=segment, gene=gene, direction=direction,
                    is_red=gene in red, is_tf=bool(candidate['is_tf'][index]),
                    provisional=bool(provisional), literature_review=bool(context),
                    automatic_rank=int(ranks[index]), automatic_rank_pool=('TF snapshot significant genes' if letter == 'I' else 'All significant genes'),
                    fitted_start=float(candidate['fitted_start'][index, column]),
                    fitted_end=float(candidate['fitted_end'][index, column]), fitted_change=change,
                    whole_tree_fitted_amplitude=amplitude, local_fraction=fraction,
                    early_mean=early, late_mean=late, observed_change=late - early,
                    early_n=int(candidate['group_cell_counts'][2 * column]),
                    late_n=int(candidate['group_cell_counts'][2 * column + 1]),
                    association_amplitude=float(stats.loc[gene, 'A']),
                    association_p_adj=float(stats.loc[gene, 'p_adj']),
                    provisional_reason='; '.join(reason), literature_context=context,
                    block_purpose=BLOCK_PURPOSE[segment],
                    selection_reason=(BLOCK_PURPOSE[segment] + '. ' + evidence['biological_role'] +
                        '. Chosen from the significant pool with concordant measured and fitted local direction; '
                        'biological relevance and repeat control supplement the automatic rank.'),
                    **{key: str(evidence.get(key, '')) for key in
                       ('evidence_scope', 'biological_role', 'primary_sources', 'uniprot_url', 'highlight_basis', 'limitations',
                        'selection_context', 'direction_interpretation', 'regulatory_class', 'trajectory_label_scope', 'review_date')})
                records.append(record)
                assert values.max() > 0
                relative.append(values / values.max())
    rows = pd.DataFrame(records)
    assert len(rows) == 72 and rows.gene.nunique() == (63 if letter == 'H' else 51)
    assert rows.gene.value_counts().max() <= (2 if letter == 'H' else 3)
    assert rows.loc[rows.gene.isin(['Gata3', 'Lypd1']), 'is_red'].all()
    assert not rows.loc[rows.gene.isin(['Stmn2', 'Stmn3', 'Spp1', 'Elavl3']), 'is_red'].any()
    return rows, np.asarray(relative)

def attach_source_flags(rows, source_summary):
    keys = ['panel', 'selected_segment', 'gene']
    assert not source_summary.duplicated(keys).any()
    rows = rows.merge(source_summary, on=keys, how='left', validate='one_to_one', sort=False)
    assert rows.source_sensitive.notna().all()
    for index, row in rows.iterrows():
        if bool(row.source_sensitive):
            reason = 'At least one comparable source stratum has an opposite measured direction (>=5 cells per phase; descriptive)'
            rows.loc[index, 'provisional_reason'] = '; '.join(filter(None, [row.provisional_reason, reason]))
            rows.loc[index, 'provisional'] = True
        if row.selected_segment == 'VGN' and row.gene in ['Nav2', 'Dact1']:
            reason = 'The second VGN study has only 4 Early / 12 Late cells and an opposite measured direction'
            rows.loc[index, 'provisional_reason'] = '; '.join(filter(None, [rows.loc[index, 'provisional_reason'], reason]))
            rows.loc[index, 'provisional'] = True
    for name in ('is_red', 'is_tf', 'provisional', 'literature_review', 'source_sensitive'):
        rows[name] = rows[name].astype(bool)
    return rows


In [ ]:
# 5. =========== Source composition checks
import json

import numpy as np

import pandas as pd

def compute_source_diagnostics(data, rows, min_source_n=5, stronger_source_n=10):
    """Return dict(long=table, summary=table, settings=dict) without input mutation.

    Required data keys are expression (cells x requested genes), requested
    (the expression column order), and cells (a DataFrame with seg, t, study_id,
    technology). Rows must contain panel, gene, direction, and selected_segment
    (segment is accepted as an input alias). Existing observed_change,
    early_mean and late_mean are checked against the recomputed pooled means.

    Early is t <= (segment min(t) + max(t))/2; Late is t > that midpoint.
    Variance is the sample variance (ddof=1); detection means expression > 0.
    Missing phases have NaN means/variances rather than invented zero values.

    source_sensitive is True if any study, technology, or study x technology
    group has at least min_source_n cells in each phase and a strictly opposite
    direction to the pooled mean change. Exact zero change is recorded as flat.
    Smaller groups with both phases are retained and flagged separately; their
    opposite trends do not trigger source_sensitive. The thresholds never
    alter the selected genes, fitted values, or displayed expression means.

    source_summary has unique panel/selected_segment/gene keys. Counts refer to
    source groups, not independent biological replicates; overlapping source
    levels are never summed into an independent replication count.
    """
    if (not isinstance(min_source_n, (int, np.integer)) or min_source_n < 1
            or not isinstance(stronger_source_n, (int, np.integer))
            or stronger_source_n < min_source_n):
        raise ValueError("Source support thresholds must be positive ordered integers.")
    minimum_flag = "both_phases_ge" + str(min_source_n)
    stronger_flag = "both_phases_ge" + str(stronger_source_n)
    cells = data["cells"]
    required_cell_columns = {"seg", "t", "study_id", "technology"}
    if not isinstance(cells, pd.DataFrame) or not required_cell_columns <= set(cells.columns):
        raise ValueError("cells needs seg, t, study_id, and technology columns.")
    requested = pd.Index(data["requested"], dtype=str)
    expression = np.asarray(data["expression"], dtype=np.float64)
    if requested.has_duplicates or expression.shape != (len(cells), len(requested)):
        raise ValueError("Expression axes must match cells and unique requested genes.")
    if not np.isfinite(expression).all() or np.any(expression < 0):
        raise ValueError("Measured expression must be finite and nonnegative.")
    selected = pd.DataFrame(rows).copy()
    if "selected_segment" not in selected and "segment" in selected:
        selected = selected.rename(columns={"segment": "selected_segment"})
    keys = ["panel", "selected_segment", "gene"]
    if not set(keys + ["direction"]) <= set(selected.columns) or selected.empty:
        raise ValueError("Selected rows need panel, selected_segment, gene, and direction.")
    if selected.duplicated(keys).any() or not selected.direction.isin(["up", "down"]).all():
        raise ValueError("Selected row keys must be unique and directions must be up/down.")
    if selected[keys].isna().any().any():
        raise ValueError("Selected row identity fields cannot be missing.")
    selected[keys] = selected[keys].astype(str)
    if not selected.gene.isin(requested).all():
        raise ValueError("A selected gene is missing from the measured-expression columns.")
    cell_segments = cells.seg.astype(str).to_numpy()
    time = pd.to_numeric(cells.t, errors="raise").to_numpy(dtype=np.float64)
    if not np.isfinite(time).all():
        raise ValueError("Fixed pseudotime must be finite.")

    long_rows, summary_rows = [], []
    for segment, selected_block in selected.groupby("selected_segment", sort=False):
        positions = np.flatnonzero(cell_segments == segment)
        if not len(positions):
            raise ValueError("A selected segment has no cells: " + segment)
        segment_time = time[positions]
        midpoint = (segment_time.min() + segment_time.max()) / 2
        early = segment_time <= midpoint
        if not early.any() or early.all():
            raise ValueError("Each segment must support both fixed pseudotime phases.")
        genes = pd.Index(pd.unique(selected_block.gene))
        columns = requested.get_indexer(genes)
        values = expression[np.ix_(positions, columns)]
        gene_position = {gene: j for j, gene in enumerate(genes)}
        configurations = [("pooled", "all", np.ones(len(positions), dtype=bool))]
        for level, fields in [("study", ["study_id"]),
                              ("technology", ["technology"]),
                              ("study_technology", ["study_id", "technology"])]:
            frame = cells.iloc[positions][fields].fillna("Not available").astype(str)
            source = frame.agg(" | ".join, axis=1).to_numpy()
            configurations.extend((level, name, source == name) for name in pd.unique(source))

        # Source masks and all selected-gene statistics are computed once per group.
        group_statistics = []
        for level, source, mask in configurations:
            e, l = values[mask & early], values[mask & ~early]
            ne, nl = len(e), len(l)
            blank = np.full(len(genes), np.nan)
            group_statistics.append(dict(
                level=level, source=str(source), n_early=ne, n_late=nl,
                early_mean=e.mean(axis=0) if ne else blank.copy(),
                late_mean=l.mean(axis=0) if nl else blank.copy(),
                early_variance=e.var(axis=0, ddof=1) if ne > 1 else blank.copy(),
                late_variance=l.var(axis=0, ddof=1) if nl > 1 else blank.copy(),
                early_detection_fraction=(e > 0).mean(axis=0) if ne else blank.copy(),
                late_detection_fraction=(l > 0).mean(axis=0) if nl else blank.copy()))

        for _, row in selected_block.iterrows():
            j = gene_position[row.gene]
            pooled_early = float(group_statistics[0]["early_mean"][j])
            pooled_late = float(group_statistics[0]["late_mean"][j])
            pooled_delta = pooled_late - pooled_early
            for field, actual in [("early_mean", pooled_early), ("late_mean", pooled_late),
                                  ("observed_change", pooled_delta)]:
                if field in selected.columns and not np.isclose(
                        float(row[field]), actual, rtol=1e-12, atol=1e-12):
                    raise ValueError("Recomputed pooled expression differs for " +
                                     "/".join([row.panel, segment, row.gene, field]))
            identity = dict(panel=row.panel, selected_segment=segment, gene=row.gene)
            reference = np.sign(pooled_delta)
            records = []
            for group in group_statistics:
                ne, nl = group["n_early"], group["n_late"]
                has_both = bool(ne and nl)
                em, lm = float(group["early_mean"][j]), float(group["late_mean"][j])
                delta = lm - em
                record = dict(identity, selected_direction=row.direction,
                    level=group["level"], source=group["source"], n_early=ne, n_late=nl,
                    early_mean=em, late_mean=lm, observed_change=delta,
                    early_variance=float(group["early_variance"][j]),
                    late_variance=float(group["late_variance"][j]),
                    early_detection_fraction=float(group["early_detection_fraction"][j]),
                    late_detection_fraction=float(group["late_detection_fraction"][j]),
                    has_both_phases=has_both,
                    direction_matches_pooled=bool(delta * reference > 0) if has_both else None,
                    opposite_to_pooled=bool(delta * reference < 0) if has_both else None,
                    source_change_is_flat=bool(delta == 0) if has_both else None,
                    pooled_observed_change=pooled_delta,
                    fitted_change=float(row.get("fitted_change", np.nan)),
                    local_fraction=float(row.get("local_fraction", np.nan)))
                record[minimum_flag] = bool(ne >= min_source_n and nl >= min_source_n)
                record[stronger_flag] = bool(ne >= stronger_source_n and nl >= stronger_source_n)
                records.append(record)

            summary = dict(identity, pooled_early_mean=pooled_early,
                           pooled_late_mean=pooled_late, pooled_observed_change=pooled_delta,
                           min_source_cells_per_phase=int(min_source_n),
                           stronger_source_cells_per_phase=int(stronger_source_n))
            for level in ("study", "technology", "study_technology"):
                eligible = [q for q in records if q["level"] == level and q[minimum_flag]]
                stronger = [q for q in records if q["level"] == level and q[stronger_flag]]
                concordant = sum(bool(q["direction_matches_pooled"]) for q in eligible)
                opposite = sum(bool(q["opposite_to_pooled"]) for q in eligible)
                minimum_suffix = "_ge" + str(min_source_n)
                stronger_suffix = "_ge" + str(stronger_source_n)
                summary.update({
                    level + "_supported_sources" + minimum_suffix: len(eligible),
                    level + "_concordant_sources" + minimum_suffix: concordant,
                    level + "_opposite_sources" + minimum_suffix: opposite,
                    level + "_flat_sources" + minimum_suffix: sum(bool(q["source_change_is_flat"]) for q in eligible),
                    level + "_all_opposite" + minimum_suffix: bool(eligible and opposite == len(eligible)),
                    level + "_supported_sources" + stronger_suffix: len(stronger),
                    level + "_concordant_sources" + stronger_suffix: sum(bool(q["direction_matches_pooled"]) for q in stronger)})
            supported_opposite = [q for q in records if q["level"] != "pooled"
                                  and q[minimum_flag] and q["opposite_to_pooled"]]
            small_opposite = [q for q in records if q["level"] != "pooled" and q["has_both_phases"]
                              and not q[minimum_flag] and q["opposite_to_pooled"]]
            def describe(groups):
                return json.dumps([{key: q[key] for key in
                    ("level", "source", "n_early", "n_late", "observed_change")}
                    for q in groups], ensure_ascii=False, separators=(",", ":"))
            summary.update(source_sensitive=bool(supported_opposite),
                source_opposite_groups=describe(supported_opposite),
                small_source_opposite=bool(small_opposite),
                small_source_opposite_groups=describe(small_opposite))
            summary["multiple_study_support_ge" + str(min_source_n)] = summary[
                "study_supported_sources_ge" + str(min_source_n)] >= 2
            for record in records:
                record.update({key: value for key, value in summary.items() if key not in identity})
            long_rows.extend(records)
            summary_rows.append(summary)

    diagnostics = pd.DataFrame(long_rows)
    summary = pd.DataFrame(summary_rows)
    if summary.duplicated(keys).any() or len(summary) != len(selected):
        raise AssertionError("Source summaries are not unique per displayed row.")
    # Restore the caller's exact selected-row order even when panels are interleaved.
    order = selected[keys].assign(_source_row_order=np.arange(len(selected)))
    summary = order.merge(summary, on=keys, how="left", validate="one_to_one")
    summary = summary.sort_values("_source_row_order").drop(columns="_source_row_order").reset_index(drop=True)
    summary["source_sensitive"] = summary.source_sensitive.astype(bool)
    summary["small_source_opposite"] = summary.small_source_opposite.astype(bool)
    settings = dict(
        schema="trajectory-dot-plot-source-diagnostics-v1",
        min_source_cells_per_phase=int(min_source_n),
        stronger_source_cells_per_phase=int(stronger_source_n),
        phase_rule="Early: t <= (min(t)+max(t))/2 within each fixed segment; Late: t > midpoint",
        levels=["pooled", "study", "technology", "study_technology"],
        expression="Measured log10 library-normalized expression; sample variance uses ddof=1",
        direction_rule="Strict sign reversal relative to pooled Late-minus-Early mean; zero is flat",
        source_sensitive_rule="Any non-pooled source group with both phases at the minimum size and an opposite direction",
        small_source_rule="Both phases present but either below minimum; stored separately and excluded from source_sensitive",
        scope="Descriptive only; no source adjustment, independent-replicate test, selection change, or cross-source validation claim")
    return dict(long=diagnostics, summary=summary, settings=settings)


In [ ]:
# 6. =========== Editable PDF export
from pathlib import Path

import numpy as np

from collections import Counter

from io import BytesIO

import importlib.util

import re

import sys

import matplotlib as mpl

from matplotlib import font_manager

from matplotlib.path import Path as MplPath

from matplotlib.patches import PathPatch

from matplotlib.collections import QuadMesh

from matplotlib.backend_bases import RendererBase

from matplotlib.backends.backend_pdf import RendererPdf

from copy import copy

# Set the file or directory path.
HI_FONT_DIR=ROOT

HI_FONT_FILES=["arial.ttf","ariali.ttf","arialbd.ttf"]

HI_PDF_UNDERPAINT_PT=.2

def require(condition, message):
    if not condition:
        raise AssertionError(message)

def _load_pypdf():
    """Use normal pypdf or the bundled pure-Python package without changing sys.path."""
    try:
        import pypdf
        return pypdf
    except ImportError:
        # Set the file or directory path.
        package=ROOT
        # Set the file or directory path.
        require((ROOT).is_file(),"pypdf is unavailable for semantic text preservation")
        # Set the file or directory path.
        spec=importlib.util.spec_from_file_location("pypdf",ROOT,submodule_search_locations=[str(package)])
        module=importlib.util.module_from_spec(spec)
        sys.modules["pypdf"]=module
        spec.loader.exec_module(module)
        return module

def _text_block(operations,start):
    """Read one normal backend q/cm/BT...ET/Q text block, preserving its tokens."""
    if start+7>=len(operations) or [operations[start+k][1] for k in range(3)]!=[b"q",b"cm",b"BT"]:
        return None
    end=start+3
    while end<len(operations) and operations[end][1]!=b"ET":
        if operations[end][1] not in {b"Tf",b"Td",b"TD",b"Tm",b"TJ",b"Tj",b"Tc",b"Tw",b"Tz",b"TL",b"Ts",b"Tr"}:
            return None
        end+=1
    if end+1>=len(operations) or operations[end+1][1]!=b"Q":
        return None
    decoded=[]
    for operands,op in operations[start+3:end]:
        if op==b"TJ":
            strings=[item for item in operands[0] if isinstance(item,(str,bytes))]
        elif op==b"Tj":
            strings=[operands[0]]
        else:
            continue
        for item in strings:
            raw=item.original_bytes if hasattr(item,"original_bytes") else item
            decoded.append(raw.decode("utf-16-be") if isinstance(raw,bytes) else raw)
    return dict(start=start,end=end+2,text="".join(decoded),matrix=operations[start+1][0],content=operations[start+3:end])

def _semantic_multiline_pdf(raw_pdf,multiline_labels):
    """One BT/ET per source multiline label; Tm retains each original baseline.

    This changes no painted geometry or character. It only combines consecutive
    backend line objects known to belong to the same source Text artist.
    """
    pypdf=_load_pypdf()
    from pypdf.generic import ContentStream, DecodedStreamObject
    reader=pypdf.PdfReader(BytesIO(raw_pdf),strict=True)
    require(len(reader.pages)==1,"The trajectory dot plot must be one PDF page")
    writer=pypdf.PdfWriter()
    writer.clone_document_from_reader(reader)
    remaining=Counter(tuple(label.split("\n")) for label in multiline_labels)
    page=writer.pages[0]
    original=page.get_contents().get_data()
    # Match backend text blocks only. Preserve all untouched PDF bytes, avoiding
    # numeric reserialization of thousands of circles and gradient rectangles.
    pattern=re.compile(rb"(?m)^q\n([^\r\n]+ cm)\nBT\n(.*?)\nET\nQ(?=\n|$)",re.S)
    blocks=[]
    for match in pattern.finditer(original):
        temporary=DecodedStreamObject()
        temporary.set_data(match.group(0))
        parsed=ContentStream(temporary,writer).operations
        block=_text_block(parsed,0)
        require(block is not None,"Unexpected PDF backend text-block shape")
        block.update(raw_start=match.start(),raw_end=match.end(),raw_matrix=match.group(1)[:-3],raw_content=match.group(2))
        blocks.append(block)
    merged=[]
    at=0
    copied_until=0
    merged_count=0
    while at<len(blocks):
        first=blocks[at]
        candidates=[lines for lines,count in remaining.items() if count and first and lines[0]==first["text"]]
        chosen=None
        for lines in candidates:
            candidate_blocks=[]
            for offset,line in enumerate(lines):
                if at+offset>=len(blocks):
                    break
                block=blocks[at+offset]
                if block["text"]!=line:
                    break
                if offset and original[candidate_blocks[-1]["raw_end"]:block["raw_start"]].strip():
                    break
                require(tuple(float(v) for v in block["matrix"][:4])==(1.,0.,0.,1.),"Rotated multiline text requires explicit review")
                candidate_blocks.append(block)
            if len(candidate_blocks)==len(lines):
                chosen=(lines,candidate_blocks)
                break
        if chosen:
            lines,candidate_blocks=chosen
            merged.append(original[copied_until:first["raw_start"]])
            merged.append(b"q\nBT\n")
            for block in candidate_blocks:
                merged.extend([block["raw_matrix"],b" Tm\n",block["raw_content"],b"\n"])
            merged.append(b"ET\nQ")
            copied_until=candidate_blocks[-1]["raw_end"]
            remaining[lines]-=1
            merged_count+=1
            at+=len(lines)
        else:
            at+=1
    require(not any(remaining.values()),f"Unmatched source multiline labels: {remaining}")
    merged.append(original[copied_until:])
    stream=DecodedStreamObject()
    stream.set_data(b"".join(merged))
    page.replace_contents(stream)
    page.compress_content_streams()
    result=BytesIO()
    writer.write(result)
    return result.getvalue(),merged_count

def image_to_vector_runs(image_artist):
    """Preserve the displayed 8-bit RGB palette as editable rectangle subpaths."""
    data=image_artist.get_array()
    require(not np.ma.getmaskarray(data).any(),"Masked heatmap pixels are not expected")
    rgba=image_artist.to_rgba(data,bytes=True)
    require(np.all(rgba[...,3]==255),"Unexpected transparent image pixels")
    rgb=rgba[...,:3]
    rows,columns=rgb.shape[:2]
    keys=(rgb[...,0].astype(np.uint32)<<16)|(rgb[...,1].astype(np.uint32)<<8)|rgb[...,2].astype(np.uint32)
    start=np.ones((rows,columns),dtype=bool)
    start[:,1:]=keys[:,1:]!=keys[:,:-1]
    rr,cc=np.nonzero(start)
    next_c=np.r_[cc[1:],columns]
    next_r=np.r_[rr[1:],rows]
    end=np.where(next_r==rr,next_c,columns)
    palette_keys=keys[rr,cc]
    xleft,xright,ybottom,ytop=map(float,image_artist.get_extent())
    dx=(xright-xleft)/columns
    yfirst,dy=(ytop,(ybottom-ytop)/rows) if image_artist.origin=="upper" else (ybottom,(ytop-ybottom)/rows)
    x0=xleft+cc*dx;x1=xleft+end*dx
    y0=yfirst+rr*dy;y1=yfirst+(rr+1)*dy
    vertices=np.stack([np.column_stack((x0,y0)),np.column_stack((x1,y0)),
                       np.column_stack((x1,y1)),np.column_stack((x0,y1)),
                       np.column_stack((x0,y0))],axis=1)
    code_pattern=np.array([MplPath.MOVETO,MplPath.LINETO,MplPath.LINETO,MplPath.LINETO,MplPath.CLOSEPOLY],dtype=np.uint8)
    axes=image_artist.axes
    for key in np.unique(palette_keys):
        chosen=vertices[palette_keys==key]
        color=((int(key)>>16)/255.,((int(key)>>8)&255)/255.,(int(key)&255)/255.)
        path=MplPath(chosen.reshape(-1,2),np.tile(code_pattern,len(chosen)))
        patch=PathPatch(path,facecolor=color,edgecolor="none",linewidth=0.,
                        antialiased=False,clip_on=False,transform=image_artist.get_transform(),
                        zorder=image_artist.get_zorder())
        underpaint=PathPatch(path,facecolor=color,edgecolor=color,linewidth=HI_PDF_UNDERPAINT_PT,
                             antialiased=False,clip_on=False,transform=image_artist.get_transform(),
                             zorder=image_artist.get_zorder()-.01)
        axes.add_artist(underpaint)
        axes.add_artist(patch)
    image_artist.remove()
    return int(len(vertices))

def export_editable_pdf(fig,path):
    """Export an existing accepted Figure; call after PNG saving; does not close it.

    The figure's PDF-only image/mesh artists are replaced in place. All labels
    stay live Arial text, including one semantic text object per multiline label.
    Returns gradient-run counts, merged-label count and expected clip count.
    """
    from matplotlib.text import Text
    path=Path(path)
    multiline_labels=[text.get_text() for text in fig.findobj(Text)
                      if text.get_visible() and "\n" in text.get_text()]
    # PNG has already been written from the unchanged accepted Agg canvas.
    color_runs=[]
    for axes in fig.axes:
        for image_artist in list(axes.images):
            color_runs.append(image_to_vector_runs(image_artist))
    for artist in fig.findobj():
        if hasattr(artist,"set_clip_on"):artist.set_clip_on(False)
        if hasattr(artist,"set_clip_path"):artist.set_clip_path(None)
        if hasattr(artist,"set_clip_box"):artist.set_clip_box(None)
        if hasattr(artist,"set_rasterized"):artist.set_rasterized(False)
        if isinstance(artist,QuadMesh):
            # Underpaint the colorbar first; keep its original fill on top.
            underpaint=copy(artist)
            underpaint.set_edgecolor("face")
            underpaint.set_linewidth(HI_PDF_UNDERPAINT_PT)
            underpaint.set_zorder(artist.get_zorder()-.01)
            underpaint.set_clip_on(False)
            underpaint.set_clip_path(None)
            underpaint.set_clip_box(None)
            underpaint.set_rasterized(False)
            artist.axes.add_collection(underpaint,autolim=False)
            artist.set_edgecolor("none")
            artist.set_linewidth(0.)
            artist.set_rasterized(False)
    # Disable PDF backend reuse into Form XObjects, including repeated tick marks.
    previous_markers=RendererPdf.draw_markers
    previous_collection=RendererPdf.draw_path_collection
    try:
        RendererPdf.draw_markers=RendererBase.draw_markers
        RendererPdf.draw_path_collection=RendererBase.draw_path_collection
        with mpl.rc_context({"pdf.fonttype":42,"pdf.use14corefonts":False,
                             "text.usetex":False,"path.simplify":False,
                             "pdf.compression":6}):
            raw_pdf=BytesIO()
            fig.savefig(raw_pdf,format="pdf",dpi=600,
                        metadata={"Title":path.stem,"Author":"Developmental map of auditory neurons",
                                  "Subject":"Editable vector trajectory dot plot; live Arial text; no raster images"})
    finally:
        RendererPdf.draw_markers=previous_markers
        RendererPdf.draw_path_collection=previous_collection
    final_bytes,merged=_semantic_multiline_pdf(raw_pdf.getvalue(),multiline_labels)
    path.write_bytes(final_bytes)
    return dict(gradient_runs=color_runs,semantic_multiline_labels=merged,expected_clips=0)

for font_file in HI_FONT_FILES:
    # Set the file or directory path.
    require((ROOT).is_file(), f"Missing Arial font: {font_file}")
    # Set the file or directory path.
    font_manager.fontManager.addfont(str(ROOT))


In [ ]:
# 7. =========== Panel layout and drawing
from pathlib import Path

import numpy as np

import pandas as pd

import matplotlib

matplotlib.use("Agg")

import matplotlib.pyplot as plt

from matplotlib.colors import LinearSegmentedColormap, PowerNorm

from matplotlib.patches import Rectangle

from matplotlib.path import Path as LayoutPath

from matplotlib.transforms import Bbox

DPI, FIGSIZE = 600, (11.3, 13.4)

MAX_DOT_AREA, DOT_STROKE, FRAME_STROKE = 70.0, .32, .45

DOT_FRAME_GAP_PT = 2.855658458254973

FRAME_GAP_PT, ROW_PITCH_PT = 7.69887147540984, 9.731001996024414

COLUMN_GAP_RATIO, PANEL_GAP_PT = .5, 63.2448

LEFT_PT, BODY_BOTTOM_PT = 85.0, 100.0

GENE_SIZE, HEADER_SIZE, PHASE_SIZE = 8.2, 8.0, 6.0

DIRECTION_SIZE, PANEL_SIZE, TITLE_SIZE = 7.0, 17.0, 11.0

GENE_OFFSET_PT, DIRECTION_OFFSET_PT = 11.35041049180328, 4.711491147540982

RED_COLOR, BLACK_COLOR = "#C32032", "#1B1B1B"

BLOCK_ALPHA, BLOCK_STROKE = .19, .6

COLOR_GAMMA = 1.5

COLOR_CMAP = plt.get_cmap("RdBu_r").resampled(1024)

COLOR_NORM = PowerNorm(gamma=COLOR_GAMMA, vmin=0, vmax=1)

GRADIENT_STEPS, STRIP_TOP, STRIP_BOTTOM = 150, -2.3, -1.62

HEADING_BOTTOM, PHASE_BOTTOM, ELBOW_Y = -3.5, -.9, -2.85

LINK_COLOR, LINK_WIDTH, ARROW_SCALE = "#3E3E3E", .65, 6.0

LINK_BASE_LEVELS = {
    ("CVG", "VGN"): -5.7, ("CVG", "T"): -8.8,
    ("T", "AC"): -5.7, ("T", "AB"): -7.8,
    ("AC", "Ib"): -5.7, ("AC", "Ic"): -6.7,
    ("AB", "Ia"): -5.7, ("AB", "II"): -6.7,
}

LINK_BASELINE_Y, LINK_HEIGHT_SCALE = -5.1, .5

LINK_LEVELS = {key: LINK_BASELINE_Y + LINK_HEIGHT_SCALE * (level - LINK_BASELINE_Y)
               for key, level in LINK_BASE_LEVELS.items()}

Y_TOP, Y_BOTTOM_EXTRA = -9.2, .7

TITLE_ABOVE_AXIS_PT, PANEL_LETTER_OFFSET_PT = 29.0, 63.0

TEXT_LINE_CLEARANCE_PT = .45

LEGEND_VALUES = np.array([.2, .4, .6, .8, 1.])

LEGEND_TITLE_Y_PT, LEGEND_SUBTITLE_Y_PT = 91., 77.

LEGEND_DOT_Y_PT, LEGEND_VALUE_Y_PT = 60., 49.5

LEGEND_TICK_TOP_PT, LEGEND_TICK_BOTTOM_PT = 54., 51.5

LEGEND_TICK_WIDTH = .45

LEGEND_PITCH_PT, LEGEND_FONT, LEGEND_TITLE_SIZE = 24.0, 8.0, 9.0

FOOTNOTE_SIZE = 7.0

def _geometry():
    n = len(SEGMENTS)
    assert n == 9 and SEGMENTS == ["CVG", "VGN", "T", "AC", "Ib", "Ic", "AB", "Ia", "II"]
    diameter = np.sqrt(MAX_DOT_AREA) + DOT_STROKE
    pair_gap = COLUMN_GAP_RATIO * diameter
    pair_center = diameter + pair_gap
    pad = diameter / 2 + FRAME_STROKE / 2 + DOT_FRAME_GAP_PT
    frame_width = pair_center + 2 * pad
    pitch = frame_width + FRAME_GAP_PT + FRAME_STROKE
    # The small outer margin is the accepted seven-block figure's half-margin.
    margin = .7495554098360654
    x = np.array([[j * pitch, j * pitch + pair_center] for j in range(n)]).ravel()
    xlim = (-pad - margin, x[-1] + pad + margin)
    ylim = (n * 8 - 1 + Y_BOTTOM_EXTRA, Y_TOP)
    return dict(n=n, rows=n * 8, diameter=diameter, pair_gap=pair_gap,
                pair_center=pair_center, pad=pad, frame_width=frame_width,
                pitch=pitch, x=x, xlim=xlim, ylim=ylim,
                width_pt=xlim[1] - xlim[0],
                height_pt=(ylim[0] - ylim[1]) * ROW_PITCH_PT)

def _check_panel_input(rows, relative, letter):
    required = {"gene", "direction", "is_red", "provisional", "literature_review", "selected_segment"}
    assert required <= set(rows.columns), (letter, "Missing row metadata", required - set(rows.columns))
    assert len(rows) == 72 and rows.index.equals(pd.RangeIndex(72))
    assert np.shape(relative) == (72, 18)
    assert np.isfinite(relative).all() and np.min(relative) >= 0 and np.max(relative) <= 1 + 1e-12
    assert rows.selected_segment.tolist() == [s for s in SEGMENTS for _ in range(8)]
    assert rows.direction.isin(["up", "down"]).all()
    assert rows.gene.notna().all() and rows.gene.astype(str).str.len().gt(0).all()
    for segment in SEGMENTS:
        part = rows.loc[rows.selected_segment.eq(segment)]
        assert len(part) == 8 and part.gene.nunique() == 8
    for field in ["is_red", "provisional", "literature_review"]:
        assert rows[field].map(lambda value: isinstance(value, (bool, np.bool_))).all(), (letter, field)
    assert rows.loc[rows.gene.isin(["Gata3", "Lypd1"]), "is_red"].all(), (letter, "Required red reference labels")

def draw_panel(fig, box, letter, rows, relative, geometry):
    g = geometry
    n, x, pad = g["n"], g["x"], g["pad"]
    ax = fig.add_axes(box)
    for j, segment in enumerate(SEGMENTS):
        end = EDGES[segment][1]
        color = NODE_COLORS[end] if end in NODE_COLORS else NODE_COLORS[str(end)]
        ax.add_patch(Rectangle((x[2 * j] - pad, j * 8 - .5), g["frame_width"], 8,
                     facecolor=color, edgecolor=color, alpha=BLOCK_ALPHA, lw=BLOCK_STROKE, zorder=0))
        ax.add_patch(Rectangle((x[2 * j] - pad, -.5), g["frame_width"], g["rows"],
                     facecolor="none", edgecolor="#666666", lw=FRAME_STROKE, zorder=2))
    xx, yy = np.meshgrid(x, np.arange(g["rows"]))
    dots = ax.scatter(xx.ravel(), yy.ravel(), s=relative.ravel() * MAX_DOT_AREA,
               c=relative.ravel(), cmap=COLOR_CMAP, norm=COLOR_NORM,
               edgecolors="#262626", linewidths=DOT_STROKE, zorder=3)
    texts, connectors = [], []
    for j, row in rows.iterrows():
        suffix = ("†" if row.provisional else "") + ("‡" if row.literature_review else "")
        texts.append(ax.text(-pad - GENE_OFFSET_PT, j, row.gene + suffix,
                     ha="right", va="center", fontsize=GENE_SIZE, fontstyle="italic",
                     color=RED_COLOR if row.is_red else BLACK_COLOR, clip_on=False))
        texts.append(ax.text(-pad - DIRECTION_OFFSET_PT, j, "↑" if row.direction == "up" else "↓",
                     ha="center", va="center", fontsize=DIRECTION_SIZE, color="#666666", clip_on=False))
    for j, segment in enumerate(SEGMENTS):
        start, end = EDGES[segment]
        start_color = NODE_COLORS[start] if start in NODE_COLORS else NODE_COLORS[str(start)]
        end_color = NODE_COLORS[end] if end in NODE_COLORS else NODE_COLORS[str(end)]
        cmap = LinearSegmentedColormap.from_list(segment, [start_color, end_color])
        ax.imshow(np.linspace(0, 1, GRADIENT_STEPS)[None, :],
                  extent=(x[2 * j] - pad, x[2 * j + 1] + pad, STRIP_TOP, STRIP_BOTTOM),
                  cmap=cmap, origin="lower", aspect="auto", zorder=2)
        texts.append(ax.text(x[2 * j] + g["pair_center"] / 2, HEADING_BOTTOM, NAMES[segment],
                     ha="center", va="bottom", fontsize=HEADER_SIZE, linespacing=1.1, clip_on=False))
        for k, label in enumerate(["Early", "Late"]):
            texts.append(ax.text(x[2 * j + k], PHASE_BOTTOM, label, ha="center", va="bottom",
                         fontsize=PHASE_SIZE, color="#555555", clip_on=False))
    for (parent, child), level in LINK_LEVELS.items():
        a, b = SEGMENTS.index(parent), SEGMENTS.index(child)
        inset = 1.2849521311475407
        source = x[2 * a + 1] + pad - inset
        target = x[2 * b] - pad + inset
        track0 = x[2 * a + 1] + pad + FRAME_STROKE / 2 + FRAME_GAP_PT / 3
        track1 = x[2 * b] - pad - FRAME_STROKE / 2 - FRAME_GAP_PT / 2
        line, = ax.plot([source, source, track0, track0, track1, track1, target],
                        [STRIP_TOP - .08, ELBOW_Y, ELBOW_Y, level, level, ELBOW_Y, ELBOW_Y],
                        color=LINK_COLOR, lw=LINK_WIDTH, clip_on=False, zorder=1)
        arrow = ax.annotate("", xy=(target, STRIP_TOP - .02), xytext=(target, ELBOW_Y),
                    arrowprops=dict(arrowstyle="-|>", color=LINK_COLOR, lw=LINK_WIDTH,
                                    mutation_scale=ARROW_SCALE), zorder=4)
        connectors.extend([(f"{letter}:{parent}>{child}:stem", line),
                           (f"{letter}:{parent}>{child}:arrow", arrow.arrow_patch)])
    ax.set_xlim(*g["xlim"])
    ax.set_ylim(*g["ylim"])
    ax.set_xticks([])
    ax.set_yticks([])
    for spine in ax.spines.values():
        spine.set_visible(False)
    title_y = BODY_BOTTOM_PT + g["height_pt"] + TITLE_ABOVE_AXIS_PT
    left_pt = box[0] * fig.get_figwidth() * 72
    width_pt, height_pt = np.asarray(FIGSIZE) * 72
    texts.append(fig.text((left_pt - PANEL_LETTER_OFFSET_PT) / width_pt, title_y / height_pt,
                          letter, fontsize=PANEL_SIZE, weight="bold", va="bottom"))
    texts.append(fig.text((left_pt + g["width_pt"] / 2) / width_pt, title_y / height_pt,
                          "Genes" if letter == "H" else "Transcription factors / regulators",
                          fontsize=TITLE_SIZE, ha="center", va="bottom"))
    assert np.allclose(dots.get_sizes(), relative.ravel() * MAX_DOT_AREA)
    assert np.array_equal(np.asarray(dots.get_array()), relative.ravel())
    assert dots.norm is COLOR_NORM and dots.cmap is COLOR_CMAP
    # Measure center and visible-edge gaps in points on the actual final axes.
    pt = lambda xy: ax.transData.transform(xy) / fig.dpi * 72
    origin, late, below = pt((x[0], 0)), pt((x[1], 0)), pt((x[0], 1))
    assert np.isclose(late[0] - origin[0] - g["diameter"], g["pair_gap"], atol=1e-8)
    assert np.isclose(abs(below[1] - origin[1]), ROW_PITCH_PT, atol=1e-8)
    for j in range(n):
        left, right = pt((x[2 * j] - pad, 0))[0], pt((x[2 * j + 1] + pad, 0))[0]
        assert np.isclose(pt((x[2 * j], 0))[0] - left - FRAME_STROKE / 2 - g["diameter"] / 2,
                          DOT_FRAME_GAP_PT, atol=1e-8)
        assert np.isclose(right - pt((x[2 * j + 1], 0))[0] - FRAME_STROKE / 2 - g["diameter"] / 2,
                          DOT_FRAME_GAP_PT, atol=1e-8)
        if j < n - 1:
            assert np.isclose(pt((x[2 * j + 2] - pad, 0))[0] - right - FRAME_STROKE,
                              FRAME_GAP_PT, atol=1e-8)
    return texts, connectors

def _draw_legend(fig):
    width_pt, height_pt = np.asarray(FIGSIZE) * 72
    texts = [fig.text(.5, LEGEND_TITLE_Y_PT / height_pt,
                       "Dotplot legend (panel I)", ha="center", va="top", fontsize=LEGEND_TITLE_SIZE),
             fig.text(.5, LEGEND_SUBTITLE_Y_PT / height_pt,
                       "Relative mean expression", ha="center", va="top", fontsize=LEGEND_TITLE_SIZE)]
    span = (len(LEGEND_VALUES) - 1) * LEGEND_PITCH_PT
    positions = width_pt / 2 + np.arange(len(LEGEND_VALUES)) * LEGEND_PITCH_PT - span / 2
    guide = fig.add_axes([0, 0, 1, 1], frameon=False)
    guide.set(xlim=(0, width_pt), ylim=(0, height_pt))
    guide.axis("off")
    samples = guide.scatter(positions, np.full(len(positions), LEGEND_DOT_Y_PT),
                   s=LEGEND_VALUES * MAX_DOT_AREA, c=LEGEND_VALUES,
                   cmap=COLOR_CMAP, norm=COLOR_NORM, edgecolors="#262626", linewidths=DOT_STROKE)
    assert np.allclose(samples.get_sizes(), LEGEND_VALUES * MAX_DOT_AREA)
    assert samples.cmap is COLOR_CMAP and samples.norm is COLOR_NORM
    for x, value in zip(positions, LEGEND_VALUES):
        guide.plot([x,x], [LEGEND_TICK_TOP_PT,LEGEND_TICK_BOTTOM_PT],
                   color="#262626", lw=LEGEND_TICK_WIDTH, clip_on=False)
        texts.append(fig.text(x / width_pt, LEGEND_VALUE_Y_PT / height_pt,
                              f"{value:.1f}", ha="center", va="top", fontsize=LEGEND_FONT))
    for y, label in FOOTNOTES:
        texts.append(fig.text(.5, y / height_pt, label, ha="center", fontsize=FOOTNOTE_SIZE, color="#444444"))
    return texts

def _check_layout(fig, connectors):
    from matplotlib.text import Text
    fig.canvas.draw()
    renderer, bounds = fig.canvas.get_renderer(), fig.bbox
    texts = [t for t in fig.findobj(Text) if t.get_visible() and t.get_text()]
    boxes = [t.get_window_extent(renderer) for t in texts]
    clipped = [t.get_text() for t, b in zip(texts, boxes)
               if not (bounds.contains(b.x0, b.y0) and bounds.contains(b.x1, b.y1))]
    overlapping = [(texts[j].get_text(), texts[k].get_text())
                   for j, b in enumerate(boxes) for k in range(j + 1, len(boxes)) if b.overlaps(boxes[k])]
    assert not clipped, ("Clipped text", clipped)
    assert not overlapping, ("Overlapping labels", overlapping)
    padding = (TEXT_LINE_CLEARANCE_PT + LINK_WIDTH / 2) * fig.dpi / 72
    collisions = []
    for name, artist in connectors:
        path = artist.get_path().transformed(artist.get_transform())
        # CLOSEPOLY's dummy (0,0) vertex is not painted; evaluate the real
        # flattened subpaths so arrowheads cannot create false diagonal hits.
        segments = []
        previous = start = None
        for coordinates, code in path.iter_segments(curves=False, simplify=False):
            if code == LayoutPath.MOVETO:
                previous = start = np.asarray(coordinates[-2:])
            elif code == LayoutPath.LINETO:
                following = np.asarray(coordinates[-2:])
                segments.append(LayoutPath([previous, following]))
                previous = following
            elif code == LayoutPath.CLOSEPOLY and previous is not None:
                segments.append(LayoutPath([previous, start]))
                previous = start
        for text, b in zip(texts, boxes):
            expanded = Bbox.from_extents(b.x0 - padding, b.y0 - padding, b.x1 + padding, b.y1 + padding)
            if any(segment.intersects_bbox(expanded, filled=False) for segment in segments):
                collisions.append((name, text.get_text()))
    assert not collisions, ("Connectors intersect labels", collisions)
    return dict(text_objects=len(texts), text_clipped=False, text_overlap=False,
                connector_text_overlap=False, checked_connector_paths=len(connectors))

def build_figure(rows, relative):
    _check_panel_input(rows, relative, PANEL)
    assert set(LINK_LEVELS) == {("CVG", "VGN"), ("CVG", "T"), ("T", "AC"), ("T", "AB"),
                                ("AC", "Ib"), ("AC", "Ic"), ("AB", "Ia"), ("AB", "II")}
    plt.rcParams.update({"font.family": "Arial", "font.size": 9., "figure.facecolor": "white"})
    fig = plt.figure(figsize=FIGSIZE, dpi=DPI, facecolor="white")
    g = _geometry()
    width_pt, height_pt = np.asarray(FIGSIZE) * 72
    assert LEFT_PT + g["width_pt"] < width_pt
    box = lambda left: [left / width_pt, BODY_BOTTOM_PT / height_pt,
                        g["width_pt"] / width_pt, g["height_pt"] / height_pt]
    _, connectors = draw_panel(fig, box(LEFT_PT), PANEL, rows, relative, g)
    _draw_legend(fig)
    layout = _check_layout(fig, connectors)
    return fig, dict(figsize=list(FIGSIZE), dpi=DPI, source_blocks=9, rows_per_panel=[72],
                     columns_per_panel=18, relative_mean_range=[0., 1.],
                     dot_area_formula="70 * relative_mean_expression",
                     color_formula=f"RdBu_r(relative_mean_expression ** {COLOR_GAMMA:g})",
                     color_normalization=dict(type="PowerNorm", gamma=COLOR_GAMMA, vmin=0., vmax=1.,
                                              shared_by_all_segments=True),
                     combined_legend_values=LEGEND_VALUES.tolist(),
                     spacing=dict(max_dot_visible_diameter_pt=float(g["diameter"]),
                                  horizontal_dot_gap_pt=float(g["pair_gap"]),
                                  horizontal_dot_gap_over_max_diameter=COLUMN_GAP_RATIO,
                                  dot_to_frame_gap_pt=DOT_FRAME_GAP_PT, frame_gap_pt=FRAME_GAP_PT,
                                  row_pitch_pt=ROW_PITCH_PT, panel_gap_pt=PANEL_GAP_PT,
                                  matrix_width_pt=float(g["width_pt"]),
                                  checked_in_display_coordinates=True),
                     connector_geometry=dict(
                         height_scale=LINK_HEIGHT_SCALE, title_top_baseline_y=LINK_BASELINE_Y,
                         height_scope="Line hierarchy above fixed branch titles",
                         branches=[dict(parent=parent, child=child,
                                        original_level=float(LINK_BASE_LEVELS[(parent, child)]),
                                        current_level=float(level))
                                   for (parent, child), level in LINK_LEVELS.items()],
                         title_and_matrix_positions_unchanged=True),
                     layout=layout)

def plot_all(rows, relative):
    fig, metadata = build_figure(rows, relative)
    try:
        output = Path(FIGURE_DIR)
        output.mkdir(parents=True, exist_ok=True)
        # Set the file or directory path.
        fig.savefig(ROOT, dpi=DPI, facecolor="white")
        # Set the file or directory path.
        metadata["pdf"] = export_editable_pdf(fig, ROOT)
    finally:
        plt.close(fig)
    print("Fig3I nine-block plot: 72 rows and 18 columns per panel; physical spacing and text/connector checks passed.")
    return metadata

# Keep the original physical matrix size and margins; remove the second panel and its gap.
FIGSIZE = (FIGSIZE[0] - (_geometry()["width_pt"] + PANEL_GAP_PT) / 72, FIGSIZE[1])
FOOTNOTES = [
    (32., "I: 72 rows / 51 TFs; Ia: 3 up + 5 down; area = 70 x relative mean."),
    (20., "Red: identity/state references. † Provisional/source-sensitive. ‡ Context review."),
    (8., "Undiff. CVG: prebranch state. VGN: inferred candidate path; no lineage-tracing claim."),
]



In [ ]:
#8. =========== Load, check and draw panel I
TREE_SHA_BEFORE = runtime_file_hash(BUNDLE)
ANALYSIS = load_results()
TREE = read_fixed_tree()
NODE_COLORS = {int(key): str(value) for key, value in TREE['segment_gradient_coloring']['node_colors'].items()}
assert list(SEGMENT_EDGES) == SEGMENTS
ROWS, RELATIVE = select_panel(ANALYSIS, PANEL, SELECTION_I, RED_I)
SOURCE_AUDIT = compute_source_diagnostics(ANALYSIS, ROWS)
ROWS = attach_source_flags(ROWS, SOURCE_AUDIT['summary'])
assert ROWS.gene.nunique() == 51
assert sum(ROWS.direction.eq('up')) == 35
assert len(ROWS) - ROWS.gene.nunique() == 21
PLOT_METADATA = plot_all(ROWS, RELATIVE)
PLOT_METADATA['source_diagnostic_settings'] = SOURCE_AUDIT['settings']
assert runtime_file_hash(BUNDLE) == TREE_SHA_BEFORE, 'The accepted tree must remain unchanged.'
print(f"Panel {PANEL}: {len(ROWS)} rows / {ROWS.gene.nunique()} unique genes.")


In [ ]:
#9. =========== Check the exported image and show the panel
from PIL import Image as PILImage
from IPython.display import Image, display

# Set the exported PNG path from the preceding plotting step.
PNG_FILE = ROOT
with PILImage.open(PNG_FILE) as image:
    assert image.size == tuple(round(value * DPI) for value in FIGSIZE)
    assert all(abs(value - DPI) < 1 for value in image.info['dpi'])
assert runtime_file_hash(BUNDLE) == TREE_SHA_BEFORE
display(Image(filename=str(PNG_FILE), width=700))



# Original shared nine-segment analysis cells
These three existing method cells retain source-count normalization, the nine
natural segments, the original R association/smoothing engine and original
result-writing functions. The unchanged TF reference TSV remains in metadata.
The same complete source block is copied into H and I. No loader, dispatcher,
model parameter, R source string or validation gate is changed in this merge.
Use each original source section in its original execution context rather than
treating the merged sections as a new Run All workflow.


In [ ]:
# Publication source; no analysis was run while preparing this file.
# Set ROOT and replace every input/output ROOT placeholder independently.
# Preserve the accepted parameters, feature axes, cell IDs and strict checks.
from pathlib import Path
ROOT = Path('.')
#1. =========== Shared paths and reproducible runtime options
from pathlib import Path
import ast, hashlib, io, json, os, pickle, shutil, subprocess, sys, tempfile, warnings
import h5py
import numpy as np
import pandas as pd
ROOT = Path('.')
DATA_DIR = ROOT  # Set this path independently; use the producer notes above.
FIGURE_DIR = ROOT  # Set this path independently; use the producer notes above.
BUNDLE = ROOT  # Set this path independently; use the producer notes above.
RNA = ROOT  # Set this path independently; use the producer notes above.
NAME_STEM = 'Trajectory dot plot-2'
RESULT = ROOT  # Set this path independently; use the producer notes above.
GENE_TABLE = ROOT  # Set this path independently; use the producer notes above.
SCRATCH = ROOT  # Set this path independently; use the producer notes above.
REBUILD_ANALYSIS = False  # False redraws the compact result; True reruns all nine-segment statistics.
R_SCRIPT = ROOT  # Set this path independently; use the producer notes above.
WORKERS = 6              # Full rebuild only; each worker uses a single BLAS thread.
sys.dont_write_bytecode = True
os.environ['PYTHONDONTWRITEBYTECODE'] = '1'
SCRATCH.mkdir(parents=True, exist_ok=True)
os.environ['MPLCONFIGDIR'] = str(SCRATCH / 'mpl')

# Public AnimalTFDB mouse snapshot archived by TOME, 2021-10-16.
TF_FILE = ROOT  # Set this path independently; use the producer notes above.
TF_BYTES = TF_FILE.read_bytes()
assert hashlib.sha256(TF_BYTES).hexdigest() == 'c09a702a433584eb8db30024b3712feaa69672dbdedf7a68489a1e719f884dd3'
TF_SET = set(pd.read_csv(io.BytesIO(TF_BYTES), sep='\t').Symbol.astype(str))
assert len(TF_SET) == 1623


In [ ]:
"""Prepare original-scale RNA and nine natural branches on the accepted CVG graft.

Default reconstruction reads the formal tree and RNA only. The source-audited
73-cell E10 denominator mapping is embedded below; original source files and
old temporary expression files are not required. No tree fitting is performed.
"""

#1. =========== Shared paths and fixed analysis contracts
from pathlib import Path
import hashlib
import json
import pickle
import sys
import warnings

sys.dont_write_bytecode = True
import h5py
import numpy as np
import pandas as pd
from scipy import sparse

BUNDLE_REL = 'ROOT'
RNA_REL = 'ROOT'
N_CELLS, N_GENES = 7355, 11467
DEFINITION = 'log10(1 + common_axis_raw_count / full_available_source_endogenous_gene_counts * 1e6)'
SEGMENTS = ['CVG', 'VGN', 'T', 'AC', 'Ib', 'Ic', 'AB', 'Ia', 'II']
SEGMENT_EDGES = {'CVG': (159, 154), 'VGN': (154, 142), 'T': (154, 35),
                 'AC': (35, 91), 'Ib': (91, 106), 'Ic': (91, 81),
                 'AB': (35, 87), 'Ia': (87, 60), 'II': (87, 38)}
EXPECTED_COUNTS = {'CVG': 120, 'VGN': 175, 'T': 2037, 'AC': 904, 'Ib': 693,
                   'Ic': 226, 'AB': 1705, 'Ia': 967, 'II': 528}
FIXED_PATHS = {'VGN': ['CVG', 'VGN'], 'Ib': ['CVG', 'T', 'AC', 'Ib'],
               'Ic': ['CVG', 'T', 'AC', 'Ic'], 'Ia': ['CVG', 'T', 'AB', 'Ia'],
               'II': ['CVG', 'T', 'AB', 'II']}
RNA_SHA256 = '8625e64dfb80313661f2258c1363e63ebe3e6f5446c4e2da3c920e46137e1e2c'
CELL_IDS_SHA256 = 'b218cf7833f360806ee12a825d72573908fd2591a68b439ed5398f7672dbf8f6'
GENE_IDS_SHA256 = '7db6e382689d9fec20945bd8fac266d08fe93fa862d272643dcb8e9e57b1241e'
GENE_SYMBOLS_SHA256 = 'd25503efca13dd889c747343b6793b9c30031bb5b3de7106aae34042ed64ce34'
E10_LIBRARY_TOTALS = {'GSE231513__GSE231513_plate_6__A11.6': 852041,
 'GSE231513__GSE231513_plate_1__A2.1': 1073121,
 'GSE231513__GSE231513_plate_3__A2.3': 901128,
 'GSE231513__GSE231513_plate_2__A3.2': 1028658,
 'GSE231513__GSE231513_plate_5__A3.5': 855488,
 'GSE231513__GSE231513_plate_1__A4.1': 966208,
 'GSE231513__GSE231513_plate_5__A6.5': 993685,
 'GSE231513__GSE231513_plate_6__A6.6': 1222986,
 'GSE231513__GSE231513_plate_5__A7.5': 904523,
 'GSE231513__GSE231513_plate_5__A8.5': 656357,
 'GSE231513__GSE231513_plate_5__B1.5': 882066,
 'GSE231513__GSE231513_plate_3__B10.3': 1378809,
 'GSE231513__GSE231513_plate_4__B11.4': 1041840,
 'GSE231513__GSE231513_plate_3__B12.3': 760942,
 'GSE231513__GSE231513_plate_6__B12.6': 633822,
 'GSE231513__GSE231513_plate_3__B3.3': 1477897,
 'GSE231513__GSE231513_plate_5__B3.5': 1033262,
 'GSE231513__GSE231513_plate_6__B3.6': 1813341,
 'GSE231513__GSE231513_plate_3__B5.3': 1333949,
 'GSE231513__GSE231513_plate_3__B7.3': 1214746,
 'GSE231513__GSE231513_plate_1__C1.1': 831935,
 'GSE231513__GSE231513_plate_1__C11.1': 977948,
 'GSE231513__GSE231513_plate_4__C11.4': 942666,
 'GSE231513__GSE231513_plate_2__C2.2': 866818,
 'GSE231513__GSE231513_plate_3__C3.3': 1440530,
 'GSE231513__GSE231513_plate_1__C4.1': 1332501,
 'GSE231513__GSE231513_plate_4__C6.4': 1145905,
 'GSE231513__GSE231513_plate_5__C6.5': 860244,
 'GSE231513__GSE231513_plate_5__C7.5': 691748,
 'GSE231513__GSE231513_plate_5__C9.5': 847399,
 'GSE231513__GSE231513_plate_5__D10.5': 1170898,
 'GSE231513__GSE231513_plate_5__D11.5': 889184,
 'GSE231513__GSE231513_plate_2__D3.2': 1806080,
 'GSE231513__GSE231513_plate_4__D3.4': 1427982,
 'GSE231513__GSE231513_plate_5__D4.5': 1201422,
 'GSE231513__GSE231513_plate_5__D6.5': 749032,
 'GSE231513__GSE231513_plate_2__D7.2': 1577385,
 'GSE231513__GSE231513_plate_1__D9.1': 1391500,
 'GSE231513__GSE231513_plate_1__E10.1': 1183067,
 'GSE231513__GSE231513_plate_3__E10.3': 1852218,
 'GSE231513__GSE231513_plate_4__E10.4': 1147509,
 'GSE231513__GSE231513_plate_6__E10.6': 1423529,
 'GSE231513__GSE231513_plate_4__E11.4': 1112740,
 'GSE231513__GSE231513_plate_6__E12.6': 496957,
 'GSE231513__GSE231513_plate_6__E3.6': 948824,
 'GSE231513__GSE231513_plate_5__E4.5': 942422,
 'GSE231513__GSE231513_plate_5__E5.5': 747641,
 'GSE231513__GSE231513_plate_5__E6.5': 928515,
 'GSE231513__GSE231513_plate_1__F10.1': 1105920,
 'GSE231513__GSE231513_plate_4__F10.4': 1145493,
 'GSE231513__GSE231513_plate_5__F10.5': 970432,
 'GSE231513__GSE231513_plate_4__F11.4': 391500,
 'GSE231513__GSE231513_plate_4__F12.4': 847175,
 'GSE231513__GSE231513_plate_4__F4.4': 1161986,
 'GSE231513__GSE231513_plate_4__F5.4': 752327,
 'GSE231513__GSE231513_plate_4__F7.4': 1074123,
 'GSE231513__GSE231513_plate_4__F9.4': 983843,
 'GSE231513__GSE231513_plate_1__G1.1': 748949,
 'GSE231513__GSE231513_plate_2__G1.2': 918962,
 'GSE231513__GSE231513_plate_2__G10.2': 932600,
 'GSE231513__GSE231513_plate_4__G11.4': 886468,
 'GSE231513__GSE231513_plate_6__G5.6': 792860,
 'GSE231513__GSE231513_plate_4__G6.4': 1017637,
 'GSE231513__GSE231513_plate_4__G7.4': 1017004,
 'GSE231513__GSE231513_plate_3__G9.3': 931514,
 'GSE231513__GSE231513_plate_4__G9.4': 583618,
 'GSE231513__GSE231513_plate_4__H12.4': 864874,
 'GSE231513__GSE231513_plate_1__H3.1': 937632,
 'GSE231513__GSE231513_plate_2__H3.2': 818250,
 'GSE231513__GSE231513_plate_6__H3.6': 849871,
 'GSE231513__GSE231513_plate_1__H5.1': 1019427,
 'GSE231513__GSE231513_plate_2__H8.2': 1300366,
 'GSE231513__GSE231513_plate_5__H9.5': 863050}
DENOMINATOR_PROVENANCE = {'sources': [{'path': 'Processed Data\\02_qc\\E9_GSE178931__GSM5401072.h5ad',
              'sha256': '7587ebb9bf0171e3996de3b68348af2ffa2a3e33d5f79d7e441ce50c897c6595',
              'selected_cells': 12,
              'source_features': 31054,
              'endogenous_features': 31053,
              'excluded_technical_features': 1,
              'matrix_unit': 'UMI_counts',
              'X_semantics': 'all_local_raw_10x_UMI_counts_before_our_QC_and_lineage_selection',
              'denominator': 'Sum over every available nontechnical source feature; mitochondrial/Hb genes '
                             'retained when supplied',
              'common_counts_verified_elementwise': True,
              'total_counts_endogenous_verified': True,
              'library_total_min': 8623.0,
              'library_total_max': 56267.0},
             {'path': 'Processed Data\\02_qc\\E11_GSE178931__GSM5401073.h5ad',
              'sha256': '6faf2728a6355a35939f285535a9ed7c2616bd20afb8c91c7b3878cc6e606848',
              'selected_cells': 105,
              'source_features': 31054,
              'endogenous_features': 31053,
              'excluded_technical_features': 1,
              'matrix_unit': 'UMI_counts',
              'X_semantics': 'all_local_raw_10x_UMI_counts_before_our_QC_and_lineage_selection',
              'denominator': 'Sum over every available nontechnical source feature; mitochondrial/Hb genes '
                             'retained when supplied',
              'common_counts_verified_elementwise': True,
              'total_counts_endogenous_verified': True,
              'library_total_min': 3870.0,
              'library_total_max': 32231.0},
             {'path': 'Processed Data\\02_qc\\E11_GSE178931__GSM5401074.h5ad',
              'sha256': '354e0c17c0d8607eadc55e749f42a772c78ad90a3df0c532175f8a62a96ef996',
              'selected_cells': 96,
              'source_features': 31054,
              'endogenous_features': 31053,
              'excluded_technical_features': 1,
              'matrix_unit': 'UMI_counts',
              'X_semantics': 'all_local_raw_10x_UMI_counts_before_our_QC_and_lineage_selection',
              'denominator': 'Sum over every available nontechnical source feature; mitochondrial/Hb genes '
                             'retained when supplied',
              'common_counts_verified_elementwise': True,
              'total_counts_endogenous_verified': True,
              'library_total_min': 3366.0,
              'library_total_max': 29509.0},
             {'path': 'Processed Data\\02_qc\\E11_GSE178931__GSM5401075.h5ad',
              'sha256': '93e0ff31a77ca2334158b62b4cdcfca0df4ac6c923298cb387de2e6af3f760eb',
              'selected_cells': 130,
              'source_features': 31054,
              'endogenous_features': 31053,
              'excluded_technical_features': 1,
              'matrix_unit': 'UMI_counts',
              'X_semantics': 'all_local_raw_10x_UMI_counts_before_our_QC_and_lineage_selection',
              'denominator': 'Sum over every available nontechnical source feature; mitochondrial/Hb genes '
                             'retained when supplied',
              'common_counts_verified_elementwise': True,
              'total_counts_endogenous_verified': True,
              'library_total_min': 4747.0,
              'library_total_max': 25430.0},
             {'path': 'Processed Data\\02_qc\\E13_GSE178931__GSM5401076.h5ad',
              'sha256': '7f0bfb39f80d5fa44b15b4ee0ac38e7fce44fba8b5dc102c2d341cdbbbe8ad19',
              'selected_cells': 222,
              'source_features': 31054,
              'endogenous_features': 31053,
              'excluded_technical_features': 1,
              'matrix_unit': 'UMI_counts',
              'X_semantics': 'all_local_raw_10x_UMI_counts_before_our_QC_and_lineage_selection',
              'denominator': 'Sum over every available nontechnical source feature; mitochondrial/Hb genes '
                             'retained when supplied',
              'common_counts_verified_elementwise': True,
              'total_counts_endogenous_verified': True,
              'library_total_min': 1952.0,
              'library_total_max': 37655.0},
             {'path': 'Processed Data\\02_qc\\E13_GSE178931__GSM5401077.h5ad',
              'sha256': '3935712177dae7f01e17546583f144798d00d864755fc8a7629b130230cb70bc',
              'selected_cells': 203,
              'source_features': 31054,
              'endogenous_features': 31053,
              'excluded_technical_features': 1,
              'matrix_unit': 'UMI_counts',
              'X_semantics': 'all_local_raw_10x_UMI_counts_before_our_QC_and_lineage_selection',
              'denominator': 'Sum over every available nontechnical source feature; mitochondrial/Hb genes '
                             'retained when supplied',
              'common_counts_verified_elementwise': True,
              'total_counts_endogenous_verified': True,
              'library_total_min': 2383.0,
              'library_total_max': 20348.0},
             {'path': 'Processed Data\\02_qc\\E13_GSE178931__GSM5401078.h5ad',
              'sha256': '9a1ea52294875253813c57ecb08dbac1821fd42092299d0036f9c6d30448270c',
              'selected_cells': 186,
              'source_features': 31054,
              'endogenous_features': 31053,
              'excluded_technical_features': 1,
              'matrix_unit': 'UMI_counts',
              'X_semantics': 'all_local_raw_10x_UMI_counts_before_our_QC_and_lineage_selection',
              'denominator': 'Sum over every available nontechnical source feature; mitochondrial/Hb genes '
                             'retained when supplied',
              'common_counts_verified_elementwise': True,
              'total_counts_endogenous_verified': True,
              'library_total_min': 2861.0,
              'library_total_max': 26024.0},
             {'path': 'Processed Data\\02_qc\\E13_GSE178931__GSM5401079.h5ad',
              'sha256': 'ea304ddc09f194e0812a07ac047f354d627cd4d38d425d82ba7b8409a4e0d0e0',
              'selected_cells': 207,
              'source_features': 31054,
              'endogenous_features': 31053,
              'excluded_technical_features': 1,
              'matrix_unit': 'UMI_counts',
              'X_semantics': 'all_local_raw_10x_UMI_counts_before_our_QC_and_lineage_selection',
              'denominator': 'Sum over every available nontechnical source feature; mitochondrial/Hb genes '
                             'retained when supplied',
              'common_counts_verified_elementwise': True,
              'total_counts_endogenous_verified': True,
              'library_total_min': 2144.0,
              'library_total_max': 25044.0},
             {'path': 'Processed '
                      'Data\\02_qc\\E14_E15_E16_E17_E18_P3_unresolved_GSE165502__all_4992_raw_cells.h5ad',
              'sha256': '25b6c6fee17a201df668a08bcff5a49dbb4f65d86ea0ae29a774cd10996f4828',
              'selected_cells': 1505,
              'source_features': 24581,
              'endogenous_features': 24489,
              'excluded_technical_features': 92,
              'matrix_unit': 'read_counts',
              'X_semantics': 'full_raw_Smart_seq2_read_counts_without_author_tree_selection',
              'denominator': 'Sum over every available nontechnical source feature; mitochondrial/Hb genes '
                             'retained when supplied',
              'common_counts_verified_elementwise': True,
              'total_counts_endogenous_verified': True,
              'library_total_min': 4340.0,
              'library_total_max': 906235.0},
             {'path': 'Processed Data\\02_qc\\E14_E16_E18_P1_GSE195500__all_5441_local_cells.h5ad',
              'sha256': 'd32caab48378d2f6f483187af8c6bb1dcea00106c65f15d3026c704e43922c91',
              'selected_cells': 4616,
              'source_features': 17999,
              'endogenous_features': 17999,
              'excluded_technical_features': 0,
              'matrix_unit': 'UMI_counts',
              'X_semantics': 'author_SGN_subset_raw_10x_UMI_counts_before_our_QC',
              'denominator': 'Sum over every available nontechnical source feature; mitochondrial/Hb genes '
                             'retained when supplied',
              'common_counts_verified_elementwise': True,
              'total_counts_endogenous_verified': True,
              'library_total_min': 1031.0,
              'library_total_max': 10927.0},
             {'path': 'Tmp\\e10_counts\\full_counts\\counts_unstranded.tsv.gz',
              'sha256': 'd56ed2d7f8c6d06b6f1d37385c36a79565da6e8d854613cd830d25f20b6694c2',
              'md5': 'ca63f58d71b2da398a95628c8d28a468',
              'selected_cells': 73,
              'source_features': 55487,
              'endogenous_features': 55487,
              'excluded_technical_features': 0,
              'matrix_unit': 'unstranded_read_pair_assignments',
              'denominator': 'All GENCODE M25 assigned gene counts; ERCC/reporter absent from counting '
                             'reference',
              'common_counts_verified_elementwise': True,
              'run_record_assigned_pairs_verified': True,
              'library_total_min': 391500.0,
              'library_total_max': 1852218.0}],
 'limits': ['Full available source axes may already reflect author gene filtering.',
            'GSE195500 supplied 17999-gene source axis has no mitochondrial features; unavailable genes were '
            'not invented.',
            'Read-pair/read-count and UMI sources remain distinct measurement technologies.',
            'Only source technical/spike/reporter features are removed from denominators; no cell filtering '
            'or QC was rerun.'],
 'validation': {'all_source_common_counts_equal_CSS': True,
                'source_endogenous_totals_equal_recomputed': True,
                'E10_run_records_equal': True,
                'all_expression_readback_equal': True,
                'all_7355_cells_retained': True,
                'eligible_genes': 11467}}


def digest(path):
    result = hashlib.sha256()
    with Path(path).open('rb') as stream:
        for block in iter(lambda: stream.read(4 * 1024 * 1024), b''):
            result.update(block)
    return result.hexdigest()


def array_hash(value):
    return hashlib.sha256(np.ascontiguousarray(value).tobytes()).hexdigest()


def sequence_hash(value):
    text = json.dumps([str(v) for v in value], ensure_ascii=False, separators=(',', ':'))
    return hashlib.sha256(text.encode('utf-8')).hexdigest()


def read_elem(group):
    if isinstance(group, h5py.Dataset):
        value = group[()]
        if isinstance(value, bytes):
            return value.decode('utf-8')
        if getattr(value, 'dtype', None) is not None and value.dtype.kind in 'OSU':
            return np.asarray([v.decode('utf-8') if isinstance(v, bytes) else str(v)
                               for v in np.atleast_1d(value)], dtype=object)
        return value
    encoding = group.attrs.get('encoding-type', '')
    if isinstance(encoding, bytes):
        encoding = encoding.decode()
    if encoding == 'categorical':
        categories = read_elem(group['categories'])
        return np.asarray([categories[v] if v >= 0 else None for v in group['codes'][()]], dtype=object)
    if 'values' in group:
        value = read_elem(group['values'])
        if 'mask' in group and np.any(group['mask'][()]):
            value = np.asarray(value, dtype=object)
            value[group['mask'][()].astype(bool)] = None
        return value
    return {key: read_elem(value) for key, value in group.items()}


def read_frame(group):
    index_key = group.attrs['_index']
    if isinstance(index_key, bytes):
        index_key = index_key.decode()
    return pd.DataFrame({key: read_elem(value) for key, value in group.items() if key != index_key},
                        index=pd.Index(read_elem(group[index_key]), dtype=object))


def read_csr(group):
    assert group.attrs['encoding-type'] == 'csr_matrix'
    return sparse.csr_matrix((group['data'][()], group['indices'][()], group['indptr'][()]),
                             shape=tuple(group.attrs['shape']))


#2. =========== Original cell order and complete root/fork/tip chains
def load_fixed(project_root=ROOT):
    project_root = Path(project_root)
    bundle_path, rna_path = BUNDLE, RNA
    with warnings.catch_warnings(), bundle_path.open('rb') as stream:
        warnings.simplefilter('ignore', DeprecationWarning)
        bundle = pickle.load(stream)
    assert digest(rna_path) == RNA_SHA256, 'Formal RNA source changed.'
    with h5py.File(rna_path, 'r') as handle:
        obs, genes = read_frame(handle['obs']), read_frame(handle['var'])
        counts = read_csr(handle['layers/counts'])
    assert counts.shape == (N_CELLS, N_GENES)
    assert obs.index.equals(bundle['obs'].index) and obs.index.is_unique
    assert sequence_hash(obs.index) == CELL_IDS_SHA256
    assert sequence_hash(genes.index) == GENE_IDS_SHA256
    assert sequence_hash(genes.gene_symbol) == GENE_SYMBOLS_SHA256
    assert genes.index.is_unique and genes.gene_symbol.is_unique
    assert np.issubdtype(counts.dtype, np.integer) and np.all(counts.data >= 0)
    assert bundle['obs'].equals(bundle['archived_inputs']['old']['obs']), 'Accepted annotations changed.'
    for column in bundle['obs'].columns.intersection(obs.columns):
        assert np.array_equal(obs[column].astype(str), bundle['obs'][column].astype(str)), column

    projection = bundle['projection']
    model = bundle['model']
    assert projection['root'] == 159 and bundle['local_fork'] == 154
    assert model['F'].shape == (30, 160) and model['B'].shape == (160, 160)
    assert projection['edges'].shape == (159, 2)
    assert len(bundle['publication_display']['branches']) == 9
    expected_pairs = {tuple(pair): name for name, pair in SEGMENT_EDGES.items()}
    branch_to_name, edge_to_segment = {}, np.full(159, '', dtype=object)
    covered = np.zeros(159, dtype=int)
    for branch in bundle['publication_display']['branches']:
        name = expected_pairs[(int(branch['start']), int(branch['end']))]
        edge_ids = np.asarray(branch['edge_ids'], dtype=int)
        path = branch['path']
        assert np.array_equal(projection['edges'][edge_ids], np.asarray(list(zip(path[:-1], path[1:]))))
        edge_to_segment[edge_ids] = name
        covered[edge_ids] += 1
        branch_to_name[int(branch['id'])] = name
    assert np.all(covered == 1) and set(edge_to_segment) == set(SEGMENTS)
    cells = bundle['obs'].copy(deep=True)
    cells['t'] = np.asarray(projection['t'], dtype=np.float64)
    cells['seg'] = edge_to_segment[projection['edge_id']]
    cells['complete_branch_id'] = np.asarray(bundle['publication_display']['cell_branch'], dtype=int)
    assert np.array_equal(cells.seg, [branch_to_name[int(v)] for v in cells.complete_branch_id])
    cells['metric_segment_id'] = np.asarray(bundle['linear_display_geometry']['segment_ids'], dtype=int)
    cells['candidate_call'] = bundle['local_candidate_calls'].reindex(cells.index).fillna('Not assessed in local 478')
    cells['projection_component'] = bundle['projection_component'].reindex(cells.index)
    assert cells.projection_component.notna().all()
    assert cells.seg.value_counts().to_dict() == EXPECTED_COUNTS
    assert np.isfinite(cells.t).all() and cells.groupby('seg').t.nunique().ge(5).all()
    for key in ['biological_replicate_id', 'technology', 'source_state_evaluation_allowed',
                'source_rna_mixture_review_flag', 'count_unit_verified']:
        if key in obs:
            cells[key] = obs[key]
    genes['gene_id'] = genes.index
    chrom = genes.chromosome.fillna('').astype(str).str.lower()
    genes['eligible'] = ~(chrom.isin(['chrm', 'chrmt', 'mt', 'm']) |
                          genes.gene_symbol.astype(str).str.match(r'(?i)^mt-|^ERCC[-_]'))
    genes['detected_cells'] = np.asarray((counts > 0).sum(axis=0)).ravel()
    assert genes.eligible.all(), 'The fixed 11467-gene tested axis changed.'
    return bundle, obs, cells, genes, counts


#3. =========== Source-audited full-library denominators
def recover_libraries(obs, genes, counts, frozen_h5=None):
    totals = pd.to_numeric(obs.total_counts_endogenous, errors='coerce').to_numpy(dtype='float64')
    e10 = obs.study_id.eq('GSE231513').to_numpy()
    assert int(e10.sum()) == 73 and set(obs.index[e10]) == set(E10_LIBRARY_TOTALS)
    assert np.isnan(totals[e10]).all() and np.isfinite(totals[~e10]).all()
    totals[e10] = [E10_LIBRARY_TOTALS[cell] for cell in obs.index[e10]]
    assert totals[e10].sum() == 74875349
    common = np.asarray(counts.sum(axis=1)).ravel()
    assert np.isfinite(totals).all() and np.all(totals > 0) and np.all(totals >= common)
    assert np.array_equal(totals, np.floor(totals))
    if frozen_h5 is not None:
        with h5py.File(frozen_h5, 'r') as store:
            assert np.array_equal(store['cells/_index'].asstr()[()], obs.index.astype(str))
            assert np.array_equal(store['genes/gene_id'].asstr()[()], genes.index.astype(str))
            assert np.array_equal(store['genes/gene_symbol'].asstr()[()], genes.gene_symbol.astype(str))
            assert np.array_equal(store['normalization/library_total'][()], totals)
            assert np.array_equal(store['normalization/common_axis_total'][()], common)
    return totals, common


def group_composition(cells):
    result = {}
    columns = ['stage_standardized', 'study_id', 'state', 'technology', 'candidate_call', 'projection_component']
    def describe(frame):
        return {key: {str(k): int(v) for k, v in frame[key].value_counts(dropna=False).items()}
                for key in columns if key in frame}
    for segment in SEGMENTS:
        frame = cells.loc[cells.seg.eq(segment)]
        low, high = float(frame.t.min()), float(frame.t.max())
        midpoint = (low + high) / 2
        early, late = frame.loc[frame.t <= midpoint], frame.loc[frame.t > midpoint]
        assert len(early) > 0 and len(late) > 0
        result[segment] = dict(cells=len(frame), t_min=low, t_max=high, midpoint=midpoint,
                               distinct_t=int(frame.t.nunique()), early_cells=len(early), late_cells=len(late),
                               all=describe(frame), early=describe(early), late=describe(late))
    return result


#4. =========== Full original-scale expression and bounded validation
def prepare_analysis_inputs(workdir, project_root=ROOT, frozen_h5=None):
    GENE_BLOCK = 64  # CSC blocks avoid repeatedly scanning the full CSR matrix.
    project_root, workdir = Path(project_root), Path(workdir)
    bundle_path, rna_path = BUNDLE, RNA
    bundle, obs, cells, genes, counts = load_fixed(project_root)
    totals, common = recover_libraries(obs, genes, counts, frozen_h5)
    cells['library_total'], cells['common_axis_total'] = totals, common
    cells['library_fraction_common'] = common / totals
    counts = counts.tocsc()
    workdir.mkdir(parents=True, exist_ok=True)
    output = workdir / 'expression.h5'
    with h5py.File(output, 'w') as handle:
            handle.attrs.update(schema='Trajectory_dot_plot_9segment_expression_v2', axes='cells_by_genes',
                calculation=DEFINITION, source_h5ad_sha256=RNA_SHA256, tree_bundle_sha256=digest(bundle_path),
                denominator_scope='Full available source endogenous feature axis, not common intersection',
                mitochondrial_in_denominator='Retained if present in the supplied source matrix')
            values = handle.create_dataset('expression', shape=(N_CELLS, N_GENES), dtype='float64',
                chunks=(N_CELLS, 1), compression='gzip', compression_opts=1, shuffle=True)
            values.attrs.update(axes='cells_by_genes', calculation=DEFINITION)
            handle.create_dataset('library_total', data=totals)
            for key, value in [('cell_ids', cells.index), ('gene_ids', genes.index), ('gene_symbols', genes.gene_symbol)]:
                handle.create_dataset(key, data=np.asarray(value, dtype=object), dtype=h5py.string_dtype('utf-8'))
            for start in range(0, N_GENES, GENE_BLOCK):
                stop = min(start + GENE_BLOCK, N_GENES)
                block = counts[:, start:stop].toarray().astype('float64')
                block /= totals[:, None]
                block *= 1e6
                np.log10(1 + block, out=block)
                assert np.isfinite(block).all() and np.all(block >= 0)
                values[:, start:stop] = block
    with h5py.File(output, 'r') as handle:
            assert np.array_equal(handle['library_total'][()], totals)
            for start in range(0, N_GENES, GENE_BLOCK):
                stop = min(start + GENE_BLOCK, N_GENES)
                block = counts[:, start:stop].toarray().astype('float64')
                block /= totals[:, None]
                block *= 1e6
                np.log10(1 + block, out=block)
                assert np.array_equal(handle['expression'][:, start:stop], block)
    cells.to_csv(workdir / 'cells.tsv', sep='\t', index_label='cell_id')
    columns = ['gene_id', 'gene_symbol', 'chromosome', 'target_biotype', 'eligible', 'detected_cells']
    genes[columns].to_csv(workdir / 'genes.tsv', sep='\t', index=False)
    composition = group_composition(cells)
    meta = dict(schema='Trajectory_dot_plot_9segment_inputs_v2', cells=N_CELLS, genes=N_GENES,
        axes='cells_by_genes', dtype='float64', calculation=DEFINITION,
        tree_bundle=BUNDLE_REL, tree_bundle_sha256=digest(bundle_path),
        source_h5ad=RNA_REL, source_h5ad_sha256=RNA_SHA256,
        tree_model_F_sha256=array_hash(bundle['model']['F']), tree_model_B_sha256=array_hash(bundle['model']['B']),
        original_t_sha256=array_hash(bundle['projection']['t']), cell_ids_sha256=sequence_hash(cells.index),
        gene_ids_sha256=sequence_hash(genes.index), gene_symbols_sha256=sequence_hash(genes.gene_symbol),
        root=159, first_global_fork=154, first_SGN_fork=35, segment_order=SEGMENTS,
        segment_edges=SEGMENT_EDGES, paths=FIXED_PATHS, segment_cells=EXPECTED_COUNTS,
        normalization_runtime='7282 formal CSS endogenous totals plus 73 embedded source-audited E10 assigned-pair totals',
        denominator_provenance=DENOMINATOR_PROVENANCE,
        denominator_frozen_formal_crosscheck=(str(frozen_h5) if frozen_h5 is not None else None),
        expression_sha256=digest(output), cells_sha256=digest(workdir / 'cells.tsv'),
        genes_sha256=digest(workdir / 'genes.tsv'),
        composition=composition,
        validation=dict(all_7355_cells_retained=True, original_annotations_unchanged=True,
            fixed_tree_projection_unchanged=True, complete_branch_partition=True,
            all_9_segments_at_least_5_distinct_t=True, five_paths=True,
            eligible_genes=int(genes.eligible.sum()), all_expression_values_exactly_verified=N_CELLS*N_GENES,
            normalization_rebuilt_from_formal_raw_counts=True,
            frozen_full_library_vector_exact_equal=(frozen_h5 is not None), E10_cells=73, E10_total=74875349),
        limits=['Conditional component graft, not an all-cell tree refit.',
                'VGN arm is candidate-enriched; original annotations are unchanged.',
                'GSE195500 supplied source axis lacks mitochondrial features.',
                'Read-pair, read-count and UMI technologies remain distinct.',
                'Publication log1p display coordinates are not used for fitting.'],
        software=dict(python=sys.version, numpy=np.__version__, pandas=pd.__version__, h5py=h5py.__version__),
        temporary_files=[str(output), str(workdir / 'cells.tsv'), str(workdir / 'genes.tsv'),
                         str(workdir / 'input_meta.json')])
    (workdir / 'input_meta.json').write_text(json.dumps(meta, ensure_ascii=False, indent=2), encoding='utf-8')
    print(f'Prepared {N_CELLS:,} cells x {N_GENES:,} genes; nine natural segments; five paths.', flush=True)
    print('Segment cells:', {segment: int(cells.seg.eq(segment).sum()) for segment in SEGMENTS}, flush=True)
    print('Full source denominators and all 84,339,785 expression values verified.', flush=True)
    return cells, genes[columns], meta

In [ ]:
#3. =========== Author-style nine-segment association and five-path smoothing
R_SOURCE = r"""#1. =========== Shared configuration and source provenance
# Run: Rscript engine.R --workdir "Tmp/Trajectory dot plot-2" --workers 6
# Validate the optimized implementation without writing files: --self-test
# Validate existing input files without fitting or writing files: --check-only
# Reuse a verified completed association table: --resume-association --overwrite
#
# scFates v0.4.0, commit 167cc97b3d9b5a4d08c28d27e4630b2d7dc0261f:
# https://github.com/LouisFaure/scFates/blob/v0.4.0/scFates/tools/test_association.py
# Git blob SHA1: fdaa1195c4243bb61b09b263e1ff354d8f67df4d
# Functions: test_assoc, apply_filters; actual correction is Bonferroni.
# https://github.com/LouisFaure/scFates/blob/v0.4.0/scFates/tools/fit.py
# Git blob SHA1: fffeeb6b5154f2ea470eded4dadf40180f32eb1d
# Functions: fit, gt_fun; gamma = 1.5; default ts basis dimension = 10.
# https://github.com/LouisFaure/scFates/blob/v0.4.0/scFates/tools/utils.py
# Git blob SHA1: 230935f5ccc5a798d4d4ed627d7767560d71b5e1
# Function: getpath; each cell belongs to every root-to-tip path containing its segment.
#
# Compatibility adaptation: upstream rpy2 m[5] and m[42] referred to
# deviance and residual degrees of freedom in its mgcv object layout.
# Current mgcv 1.9-4 places offset at R element 43, so named components
# deviance and df.residual are used here. The statistical formula is preserved,
# including upstream odf = sum(segment residual df) - number of segments.
# This script never modifies its expression HDF5 file or cells.tsv.

RANDOM_SEED <- 42L
DEFAULT_WORKERS <- 6L
EXPECTED_CELLS <- 7355L
EXPECTED_GENES <- 11467L
FIXED_ROOT <- 159L
FIXED_SEGMENTS <- c("CVG", "VGN", "T", "AC", "Ib", "Ic", "AB", "Ia", "II")
FIXED_PATHS <- list(VGN = c("CVG", "VGN"),
                    Ib = c("CVG", "T", "AC", "Ib"),
                    Ic = c("CVG", "T", "AC", "Ic"),
                    Ia = c("CVG", "T", "AB", "Ia"),
                    II = c("CVG", "T", "AB", "II"))
FIXED_EDGES <- list(CVG = c(159L, 154L), VGN = c(154L, 142L), T = c(154L, 35L),
                    AC = c(35L, 91L), Ib = c(91L, 106L), Ic = c(91L, 81L),
                    AB = c(35L, 87L), Ia = c(87L, 60L), II = c(87L, 38L))
EXPECTED_SEGMENT_COUNTS <- c(CVG = 120L, VGN = 175L, T = 2037L, AC = 904L,
                             Ib = 693L, Ic = 226L, AB = 1705L, Ia = 967L, II = 528L)
# T is the complete SGN stem (metric segments 2, 3, 4); the 7-cell
# constant-t connector is never fitted as an independent association segment.
Sys.setenv(OPENBLAS_NUM_THREADS = "1", OMP_NUM_THREADS = "1", MKL_NUM_THREADS = "1")
set.seed(RANDOM_SEED)

parse_options <- function(args) {
    opt <- list(workdir = "Tmp/Trajectory dot plot-2", workers = DEFAULT_WORKERS,
                overwrite = FALSE, self_test = FALSE, check_only = FALSE, resume_association = FALSE)
    i <- 1L
    while (i <= length(args)) {
        a <- args[[i]]
        if (a %in% c("--workdir", "--workers")) {
            if (i == length(args)) stop("Missing value after ", a)
            opt[[substring(a, 3L)]] <- args[[i + 1L]]
            i <- i + 2L
        } else if (a %in% c("--overwrite", "--self-test", "--check-only", "--resume-association")) {
            opt[[gsub("-", "_", substring(a, 3L), fixed = TRUE)]] <- TRUE
            i <- i + 1L
        } else stop("Unknown argument: ", a)
    }
    opt$workers <- suppressWarnings(as.integer(opt$workers))
    if (is.na(opt$workers) || opt$workers < 1L || opt$workers > 8L)
        stop("--workers must be an integer between 1 and 8.")
    opt
}

require_packages <- function(packages) {
    missing <- packages[!vapply(packages, requireNamespace, logical(1), quietly = TRUE)]
    if (length(missing)) stop("Missing R packages: ", paste(missing, collapse = ", "))
}

#2. =========== Association parameters and reusable GAM designs
ASSOCIATION_K <- 5L       # Original exp ~ s(t, k = 5); default thin-plate basis.
BONFERRONI_CUT <- 1e-4    # Strict adjusted p < 1e-4 across all input genes.
AMPLITUDE_CUT <- 1        # Strict max(prediction) - min(prediction) > 1.
STABILITY_CUT <- 0.8      # n_map = 1 gives st in {0, 1}.
FIT_GAMMA <- 1.5          # Original pathwise exp ~ s(t, bs = 'ts').
PROGRESS_GENES <- 1000L   # Report progress after each block, including the last.
FIT_BLOCK_GENES <- 192L   # Bound the returned fitted matrix held by the main process.
association_signature <- function() {
    paste("scFates-v0.4.0", paste0("k=", ASSOCIATION_K), "gaussian-identity", "GCV.Cp",
          "association-gamma=1", "bonferroni", paste0("n=", EXPECTED_GENES),
          paste0("p_adj<", BONFERRONI_CUT), paste0("A>", AMPLITUDE_CUT),
          paste0("st>", STABILITY_CUT), paste0("mgcv=", getNamespaceVersion("mgcv")),
          paste0("segments=", paste(names(FIXED_EDGES), vapply(FIXED_EDGES, paste, character(1), collapse = ">"), sep = ":", collapse = ",")),
          paste0("paths=", paste(names(FIXED_PATHS), vapply(FIXED_PATHS, paste, character(1), collapse = ">"), sep = ":", collapse = ",")), sep = ";")
}

make_design <- function(t, kind) {
    d <- data.frame(exp = rep(0, length(t)), t = t)
    f <- switch(kind, association = stats::as.formula(sprintf("exp ~ s(t, k=%d)", ASSOCIATION_K)),
                null = exp ~ 1, fit = exp ~ s(t, bs = "ts"))
    mgcv::gam(f, data = d, family = stats::gaussian(), method = "GCV.Cp", fit = FALSE)
}

make_state <- function(t, seg) {
    # Preserve within-segment cell order and first-seen segment order.
    sidx <- lapply(unique(seg), function(s) which(seg == s))
    names(sidx) <- unique(seg)
    pidx <- lapply(FIXED_PATHS, function(p) which(seg %in% p))
    if (any(vapply(sidx, function(i) length(unique(t[i])), integer(1)) < ASSOCIATION_K))
        stop("A segment has fewer than five distinct fixed pseudotime values.")
    if (any(vapply(pidx, function(i) length(unique(t[i])), integer(1)) < 10L))
        stop("A root-to-tip path has fewer than ten distinct fixed pseudotime values.")
    multiplicity <- tabulate(unlist(pidx, use.names = FALSE), nbins = length(t))
    if (any(multiplicity < 1L)) stop("Some cells do not belong to a fixed path.")
    list(t = t, seg = seg, ncell = length(t),
         sidx = sidx, pidx = pidx, multiplicity = multiplicity,
         association_designs = lapply(sidx, function(i) make_design(t[i], "association")),
         null_design = make_design(t, "null"),
         fit_designs = lapply(pidx, function(i) make_design(t[i], "fit")))
}

fit_from_design <- function(design, response, gamma = 1) {
    # Reuse only the response-independent basis, constraints and penalties.
    # Each call starts from the original unestimated smoothing parameters.
    design$y <- response
    design$mf$exp <- response
    mgcv::gam(G = design, gamma = gamma, method = "GCV.Cp")
}

association_one <- function(y, state) {
    models <- lapply(seq_along(state$sidx), function(j)
        fit_from_design(state$association_designs[[j]], y[state$sidx[[j]]]))
    dev <- vapply(models, function(m) m$deviance, numeric(1))
    rdf <- vapply(models, function(m) m$df.residual, numeric(1))
    predicted <- unlist(lapply(models, mgcv::predict.gam), use.names = FALSE)
    m0 <- fit_from_design(state$null_design, y)
    odf <- sum(rdf) - length(models)
    df_num <- m0$df.residual - odf
    total_dev <- sum(dev)
    fstat <- if (total_dev == 0) 0 else
        (m0$deviance - total_dev) / df_num / (total_dev / odf)
    p <- stats::pf(fstat, df_num, odf, lower.tail = FALSE)
    a <- max(predicted) - min(predicted)
    if (!is.finite(p) || !is.finite(a) || odf <= 0 || df_num <= 0)
        stop("Non-finite association statistic or invalid residual degrees of freedom.")
    c(p_val = p, A = a, F_stat = fstat, df_num = df_num,
      df_den = odf, null_deviance = m0$deviance, segment_deviance = total_dev)
}

#3. =========== Original root-to-tip smoothing and shared prediction averaging
smooth_one <- function(y, state) {
    prediction_sum <- numeric(state$ncell)
    for (j in seq_along(state$pidx)) {
        idx <- state$pidx[[j]]
        m <- fit_from_design(state$fit_designs[[j]], y[idx], gamma = FIT_GAMMA)
        prediction_sum[idx] <- prediction_sum[idx] + as.numeric(mgcv::predict.gam(m))
    }
    result <- prediction_sum / state$multiplicity
    if (any(!is.finite(result))) stop("Non-finite fitted prediction.")
    result
}

worker_gene <- function(payload, mode) {
    warnings <- character()
    value <- tryCatch(withCallingHandlers({
        y <- payload$y
        if (mode == "association") association_one(y, worker_state) else smooth_one(y, worker_state)
    }, warning = function(w) {
        warnings <<- unique(c(warnings, conditionMessage(w)))
        invokeRestart("muffleWarning")
    }), error = function(e) e)
    if (inherits(value, "error"))
        return(list(ok = FALSE, value = NULL, error = conditionMessage(value), warnings = warnings))
    list(ok = TRUE, value = value, error = "", warnings = warnings)
}

read_expression_block <- function(inputs, indices) {
    # Read each requested input block once. Workers receive response vectors,
    # not private copies of the complete 7355 x 11467 double matrix.
    block <- rhdf5::h5read(inputs$h5, "expression",
                         index = list(seq_along(inputs$cell_ids), indices), native = TRUE, drop = FALSE)
    if (!identical(dim(block), c(length(inputs$cell_ids), length(indices))))
        stop("Expression block axes do not match cells x requested genes.")
    storage.mode(block) <- "double"
    if (any(!is.finite(block))) stop("Expression block contains non-finite values.")
    block
}

map_genes <- function(indices, mode, cluster, inputs) {
    block <- read_expression_block(inputs, indices)
    payload <- lapply(seq_along(indices), function(k) list(index = indices[k], y = block[, k]))
    if (is.null(cluster)) return(lapply(payload, worker_gene, mode = mode))
    parallel::parLapplyLB(cluster, payload, worker_gene, mode = mode,
                         chunk.size = max(1L, floor(length(indices) / length(cluster) / 2L)))
}

#4. =========== Input validation and HDF5 axes
read_inputs <- function(workdir) {
    h5 <- file.path(workdir, "expression.h5")
    cellfile <- file.path(workdir, "cells.tsv")
    if (!file.exists(h5) || !file.exists(cellfile))
        stop("Expected expression.h5 and cells.tsv in --workdir.")
    read_strings <- function(name) enc2utf8(as.character(rhdf5::h5read(h5, name, native = TRUE)))
    cell_ids <- read_strings("cell_ids")
    gene_symbols <- read_strings("gene_symbols")
    gene_ids <- read_strings("gene_ids")
    if (length(cell_ids) != EXPECTED_CELLS || anyDuplicated(cell_ids))
        stop("Input must contain 7355 unique fixed-tree cell IDs.")
    if (length(gene_symbols) != EXPECTED_GENES || anyDuplicated(gene_symbols) ||
        length(gene_ids) != length(gene_symbols))
        stop("Expected 11467 unique gene symbols and matching gene IDs.")
    info <- rhdf5::h5ls(h5, recursive = FALSE, native = TRUE)
    dims <- as.integer(strsplit(gsub(" ", "", info$dim[info$name == "expression"]), "x")[[1]])
    if (!identical(dims, c(EXPECTED_CELLS, EXPECTED_GENES)))
        stop("/expression must have HDF5 row-major axes cells x genes.")
    cells <- utils::read.delim(cellfile, check.names = FALSE, stringsAsFactors = FALSE,
                              quote = "", comment.char = "", fileEncoding = "UTF-8")
    id_candidates <- c("cell_id", "cell_ids", "cell", "obs_names", "barcode", names(cells)[1])
    id_key <- id_candidates[vapply(id_candidates, function(k)
        k %in% names(cells) && !anyDuplicated(cells[[k]]) &&
            setequal(as.character(cells[[k]]), cell_ids), logical(1))][1]
    if (is.na(id_key) || !all(c("t", "seg") %in% names(cells)))
        stop("cells.tsv needs an exact cell-ID column and fixed t / seg columns.")
    cells <- cells[match(cell_ids, cells[[id_key]]), , drop = FALSE]
    t <- suppressWarnings(as.numeric(cells$t))
    seg <- as.character(cells$seg)
    if (any(!is.finite(t)) || !setequal(unique(seg), FIXED_SEGMENTS) || anyNA(seg))
        stop("Invalid fixed t or segment labels in cells.tsv.")
    observed_counts <- table(factor(seg, levels = FIXED_SEGMENTS))
    if (!identical(as.integer(observed_counts), unname(EXPECTED_SEGMENT_COUNTS[FIXED_SEGMENTS])))
        stop("Segment counts differ from the accepted nine complete branches.")
    list(h5 = normalizePath(h5, winslash = "/", mustWork = TRUE),
         cellfile = normalizePath(cellfile, winslash = "/", mustWork = TRUE),
         input_md5 = as.list(unname(tools::md5sum(c(h5, cellfile)))),
         cell_ids = cell_ids, gene_symbols = gene_symbols, gene_ids = gene_ids,
         cells = cells, state = make_state(t, seg))
}

write_utf8 <- function(value, file, dataset) {
    value <- enc2utf8(as.character(value))
    size <- max(1L, nchar(value, type = "bytes")) + 1L
    rhdf5::h5createDataset(file, dataset, dims = length(value), storage.mode = "character",
                         size = size, encoding = "UTF-8", native = TRUE)
    rhdf5::h5write(value, file, dataset, native = TRUE)
}

write_association <- function(tab, file) {
    utils::write.table(tab, file = file, sep = "\t", quote = FALSE, row.names = FALSE,
                       col.names = TRUE, na = "NA", fileEncoding = "UTF-8")
}

read_completed_association <- function(file, inputs) {
    if (!file.exists(file)) stop("--resume-association requires an existing association.tsv.")
    tab <- utils::read.delim(file, check.names = FALSE, stringsAsFactors = FALSE,
                            quote = "", comment.char = "", fileEncoding = "UTF-8")
    required <- c("gene", "gene_id", "gene_index", "p_val", "p_adj", "A", "signi", "st",
                  "F_stat", "df_num", "df_den", "null_deviance", "segment_deviance",
                  "fit_status", "association_warning", "fit_warning", "error")
    if (!all(required %in% names(tab)) || nrow(tab) != length(inputs$gene_symbols))
        stop("Association table has an incomplete schema or gene family.")
    if (!identical(as.character(tab$gene), inputs$gene_symbols) ||
        !identical(as.character(tab$gene_id), inputs$gene_ids) ||
        !identical(as.integer(tab$gene_index), seq_along(inputs$gene_symbols) - 1L))
        stop("Association gene / gene_id / gene_index order does not match the current input.")
    table_md5 <- unname(tools::md5sum(file))
    if (!all(c("association_signature", "expression_md5", "cells_md5") %in% names(tab)))
        stop("Association table lacks parameter and input-content signatures.")
    if (anyNA(tab$association_signature) || !all(tab$association_signature == association_signature()) ||
        !all(tab$expression_md5 == inputs$input_md5[[1]]) ||
        !all(tab$cells_md5 == inputs$input_md5[[2]]))
        stop("Association parameters, graph segmentation, or input contents do not match this run.")
    if (any(!is.finite(as.matrix(tab[c("p_val", "p_adj", "A", "df_num", "df_den",
                                      "null_deviance", "segment_deviance")]))))
        stop("Association resume requires complete finite statistics for every input gene.")
    if (any(tab$p_val < 0 | tab$p_val > 1 | tab$p_adj < 0 | tab$p_adj > 1 | tab$A < 0) ||
        any(tab$df_num <= 0 | tab$df_den <= 0) ||
        any(abs(tab$df_num + tab$df_den - (inputs$state$ncell - 1)) > 1e-7))
        stop("Association statistics or residual degrees of freedom are invalid.")
    expected_adjusted <- stats::p.adjust(tab$p_val, method = "bonferroni", n = nrow(tab))
    if (any(abs(tab$p_adj - expected_adjusted) > 1e-11 * pmax(1e-300, expected_adjusted)))
        stop("Stored p_adj differs from Bonferroni correction of the stored p_val.")
    expected_signi <- tab$p_adj < BONFERRONI_CUT & tab$A > AMPLITUDE_CUT
    if (!is.logical(tab$signi) || anyNA(tab$signi) ||
        !identical(tab$signi, expected_signi) ||
        !identical(as.integer(tab$st), as.integer(expected_signi)))
        stop("Stored A / p_adj / signi / st do not obey the fixed thresholds.")
    tab$association_signature <- association_signature()
    tab$fit_status <- ifelse(tab$signi, "fit_pending", "not_significant")
    tab$fit_warning <- ""
    tab$error <- ""
    attr(tab, "resume_validation") <- list(table_md5 = table_md5,
        input_content_md5_match = TRUE,
        gene_and_gene_id_exact_order = TRUE, parameter_signature = association_signature(),
        correction_and_thresholds_recomputed_from_stored_statistics = TRUE)
    tab
}

write_fitted_block <- function(file, block, gene_positions) {
    # Use rhdf5's default R-oriented interface consistently. R dimensions are
    # genes x cells; physical HDF5 dimensions are cells x genes. In rhdf5 2.54.1,
    # native=TRUE dataset creation and indexed writes did not agree on axes.
    rhdf5::h5write(t(block), file, "fitted",
                  index = list(gene_positions, seq_len(nrow(block))), native = FALSE)
    # Check a small cross-axis sample after every block; do not accept a failed
    # low-level HDF5 write that only printed a diagnostic and returned normally.
    cell_check <- unique(c(1L, as.integer(ceiling(nrow(block) / 2)), nrow(block)))
    gene_check <- unique(c(1L, ncol(block)))
    observed <- rhdf5::h5read(file, "fitted",
        index = list(cell_check, gene_positions[gene_check]), native = TRUE, drop = FALSE)
    expected <- block[cell_check, gene_check, drop = FALSE]
    if (!identical(dim(observed), dim(expected)) ||
        !isTRUE(all.equal(observed, expected, tolerance = 0, check.attributes = FALSE)))
        stop("Fitted HDF5 block failed its cross-axis read-back check.")
}

#5. =========== In-memory comparison with literal original GAM calls
self_test <- function() {
    require_packages("mgcv")
    set.seed(RANDOM_SEED)
    seg <- rep(FIXED_SEGMENTS, each = 27L)
    start <- c(CVG = 0, VGN = 1, T = 1, AC = 2, Ib = 3, Ic = 3, AB = 2, Ia = 3, II = 3)
    t <- unlist(lapply(FIXED_SEGMENTS, function(s) seq(start[s], start[s] + 1, length.out = 27L)))
    state <- make_state(t, seg)
    x <- cbind(zero = 0, flat = 2, increasing = 2 * t,
               decreasing = 6 - 2 * t, transient = 3 * exp(-(t - 1.5)^2 / 0.2),
               branch = 2 * t + 2 * (seg %in% c("Ia", "AB")),
               noise = rnorm(length(t)), low_amplitude = 0.1 * sin(t * 3))
    maximum_association_error <- 0
    maximum_prediction_error <- 0
    for (j in seq_len(ncol(x))) {
        y <- x[, j]
        # Literal scFates test_assoc calls, with named df.residual extraction.
        models <- lapply(state$sidx, function(i)
            mgcv::gam(exp ~ s(t, k = 5), data = data.frame(exp = y[i], t = t[i])))
        m0 <- mgcv::gam(exp ~ 1, data = data.frame(exp = y, t = t))
        dev <- sum(vapply(models, function(m) m$deviance, numeric(1)))
        odf <- sum(vapply(models, function(m) m$df.residual, numeric(1))) - length(models)
        df_num <- m0$df.residual - odf
        f <- if (dev == 0) 0 else (m0$deviance - dev) / df_num / (dev / odf)
        pr <- unlist(lapply(models, mgcv::predict.gam), use.names = FALSE)
        reference <- c(p_val = stats::pf(f, df_num, odf, lower.tail = FALSE),
                       A = max(pr) - min(pr), F_stat = f, df_num = df_num,
                       df_den = odf, null_deviance = m0$deviance, segment_deviance = dev)
        optimized <- association_one(y, state)
        err <- max(abs(reference - optimized) / pmax(1, abs(reference)))
        maximum_association_error <- max(maximum_association_error, err)
        if (!is.finite(err) || err > 1e-9)
            stop("Association self-test failed for ", colnames(x)[j], ": ", err)
        # Literal gt_fun pathwise fits followed by per-cell nanmean.
        predictions <- matrix(NA_real_, nrow = length(t), ncol = length(state$pidx))
        for (b in seq_along(state$pidx)) {
            i <- state$pidx[[b]]
            m <- mgcv::gam(exp ~ s(t, bs = "ts"), data = data.frame(exp = y[i], t = t[i]), gamma = 1.5)
            predictions[i, b] <- mgcv::predict.gam(m)
        }
        reference_fit <- rowMeans(predictions, na.rm = TRUE)
        optimized_fit <- smooth_one(y, state)
        fit_error <- max(abs(reference_fit - optimized_fit))
        maximum_prediction_error <- max(maximum_prediction_error, fit_error)
        if (!is.finite(fit_error) || fit_error > 1e-8 * max(1, abs(reference_fit)))
            stop("Smoothing self-test failed for ", colnames(x)[j], ": ", fit_error)
    }
    cat(sprintf("Self-test passed: %d synthetic genes; max relative association error %.3g; max absolute fitted error %.3g.\n",
                ncol(x), maximum_association_error, maximum_prediction_error))
    invisible(TRUE)
}

#6. =========== Parallel association, smoothing, and explicit output status
main <- function() {
    opt <- parse_options(commandArgs(trailingOnly = TRUE))
    if (opt$self_test) return(self_test())
    require_packages(c("mgcv", "rhdf5", "jsonlite"))
    workdir <- normalizePath(opt$workdir, winslash = "/", mustWork = TRUE)
    inputs <- read_inputs(workdir)
    if (opt$check_only) {
        if (opt$resume_association) {
            checked <- read_completed_association(file.path(workdir, "association.tsv"), inputs)
            cat(sprintf("Association resume check passed: %d genes, %d significant; matching IDs, statistics and parameters.\n",
                        nrow(checked), sum(checked$signi)))
        }
        probe <- read_expression_block(inputs, unique(c(1L, as.integer(ceiling(length(inputs$gene_symbols)/2)), length(inputs$gene_symbols))))
        cat(sprintf("Input check passed: %d cells, %d genes, %d fixed segments, %d fixed paths; finite cross-axis probes.\n",
                    length(inputs$cell_ids), length(inputs$gene_symbols), length(FIXED_SEGMENTS), length(FIXED_PATHS)))
        return(invisible(TRUE))
    }
    paths <- file.path(workdir, c("association.tsv", "fitted.h5", "fitmeta.json"))
    resume_validation <- NULL
    if (opt$resume_association) {
        tab <- read_completed_association(paths[1], inputs)
        resume_validation <- attr(tab, "resume_validation")
    }
    replace_paths <- if (opt$resume_association) paths[2:3] else paths
    if (any(file.exists(replace_paths)) && !opt$overwrite)
        stop("Output exists; use --overwrite only for this run's prior generated outputs.")
    if (opt$overwrite) unlink(replace_paths[file.exists(replace_paths)])
    start_time <- proc.time()[["elapsed"]]
    n <- length(inputs$gene_symbols)
    if (!opt$resume_association) tab <- data.frame(gene = inputs$gene_symbols, gene_id = inputs$gene_ids,
                      gene_index = seq_len(n) - 1L, p_val = NA_real_, p_adj = NA_real_,
                      A = NA_real_, signi = FALSE, fit_status = "association_pending",
                      st = 0L, F_stat = NA_real_, df_num = NA_real_, df_den = NA_real_,
                      null_deviance = NA_real_, segment_deviance = NA_real_,
                      association_warning = "", fit_warning = "", error = "",
                      association_signature = association_signature(),
                      expression_md5 = inputs$input_md5[[1]], cells_md5 = inputs$input_md5[[2]],
                      stringsAsFactors = FALSE)
    assign("worker_state", inputs$state, envir = .GlobalEnv)
    cluster <- NULL
    if (opt$workers > 1L) {
        cat(sprintf("Starting %d single-thread PSOCK workers.\n", opt$workers))
        flush.console()
        cluster <- parallel::makePSOCKcluster(opt$workers, master = "127.0.0.1", setup_timeout = 30L)
        on.exit(parallel::stopCluster(cluster), add = TRUE)
        parallel::clusterCall(cluster, function() {
            Sys.setenv(OPENBLAS_NUM_THREADS = "1", OMP_NUM_THREADS = "1", MKL_NUM_THREADS = "1")
            requireNamespace("mgcv", quietly = TRUE)
            set.seed(42L)
            NULL
        })
        parallel::clusterExport(cluster, c("worker_state", "fit_from_design", "association_one",
                                           "smooth_one", "worker_gene", "FIT_GAMMA"), envir = .GlobalEnv)
    }
    if (!opt$resume_association) {
    cat(sprintf("Association started: %d cells x %d genes; %d workers; fixed pseudotime.\n",
                inputs$state$ncell, n, opt$workers))
    for (lo in seq.int(1L, n, by = PROGRESS_GENES)) {
        idx <- seq.int(lo, min(n, lo + PROGRESS_GENES - 1L))
        result <- map_genes(idx, "association", cluster, inputs)
        for (k in seq_along(idx)) {
            j <- idx[k]
            r <- result[[k]]
            tab$association_warning[j] <- paste(r$warnings, collapse = " | ")
            if (r$ok) {
                tab[j, names(r$value)] <- as.list(r$value)
                tab$fit_status[j] <- "not_significant"
            } else {
                tab$fit_status[j] <- "association_error"
                tab$error[j] <- r$error
            }
        }
        cat(sprintf("Association: %d/%d genes; %.1f s.\n", max(idx), n,
                    proc.time()[["elapsed"]] - start_time))
        flush.console()
    }
    # Preserve the full input family size even if a gene returned an error.
    tab$p_adj <- stats::p.adjust(tab$p_val, method = "bonferroni", n = n)
    tab$signi <- !is.na(tab$p_adj) & tab$p_adj < BONFERRONI_CUT & tab$A > AMPLITUDE_CUT
    tab$st <- as.integer(tab$signi)
    tab$signi <- tab$st > STABILITY_CUT
    } else {
        cat(sprintf("Association resumed from validated table: %d genes; %d significant; no association models rerun.\n",
                    n, sum(tab$signi)))
    }
    tab$fit_status[tab$signi] <- "fit_pending"
    selected <- which(tab$signi)
    write_association(tab, paths[1])
    cat(sprintf("Association complete: %d significant genes; %d association errors.\n",
                length(selected), sum(tab$fit_status == "association_error")))
    if (!length(selected)) stop("No significant genes; association.tsv retained, no fitted HDF5 written.")
    rhdf5::h5createFile(paths[2])
    rhdf5::h5createDataset(paths[2], "fitted", dims = c(length(selected), inputs$state$ncell),
                         storage.mode = "double", native = FALSE,
                         chunk = c(min(32L, length(selected)), min(512L, inputs$state$ncell)),
                         level = 1L, fillValue = NA_real_)
    file_info <- rhdf5::h5ls(paths[2], recursive = FALSE, native = TRUE)
    physical_dims <- as.integer(strsplit(gsub(" ", "", file_info$dim[file_info$name == "fitted"]), "x")[[1]])
    if (!identical(physical_dims, c(inputs$state$ncell, length(selected))))
        stop("Physical fitted HDF5 axes must be cells x selected genes before any model fitting.")
    write_utf8(inputs$cell_ids, paths[2], "cell_ids")
    write_utf8(inputs$gene_symbols[selected], paths[2], "gene_symbols")
    write_utf8(inputs$gene_ids[selected], paths[2], "gene_ids")
    rhdf5::h5write(as.integer(selected - 1L), paths[2], "gene_index", native = TRUE)
    handle <- rhdf5::H5Fopen(paths[2])
    dataset <- rhdf5::H5Dopen(handle, "fitted")
    rhdf5::h5writeAttribute("cells_by_genes", dataset, "orientation", encoding = "UTF-8")
    rhdf5::h5writeAttribute(c("cell", "gene"), dataset, "axes", encoding = "UTF-8")
    rhdf5::H5Dclose(dataset)
    rhdf5::H5Fclose(handle)
    last_report <- 0L
    for (lo in seq.int(1L, length(selected), by = FIT_BLOCK_GENES)) {
        pos <- seq.int(lo, min(length(selected), lo + FIT_BLOCK_GENES - 1L))
        idx <- selected[pos]
        result <- map_genes(idx, "fit", cluster, inputs)
        block <- matrix(NA_real_, nrow = inputs$state$ncell, ncol = length(idx))
        for (k in seq_along(idx)) {
            j <- idx[k]
            r <- result[[k]]
            tab$fit_warning[j] <- paste(r$warnings, collapse = " | ")
            if (r$ok) {
                block[, k] <- r$value
                tab$fit_status[j] <- "ok"
            } else {
                tab$fit_status[j] <- "fit_error"
                tab$error[j] <- r$error
            }
        }
        write_fitted_block(paths[2], block, pos)
        if (max(pos) - last_report >= PROGRESS_GENES || max(pos) == length(selected)) {
            cat(sprintf("Smoothing: %d/%d significant genes; %.1f s.\n", max(pos), length(selected),
                        proc.time()[["elapsed"]] - start_time))
            last_report <- max(pos)
            flush.console()
        }
    }
    rhdf5::h5write(as.integer(tab$fit_status[selected] == "ok"), paths[2], "fit_ok", native = TRUE)
    write_association(tab, paths[1])
    script_arg <- grep("^--file=", commandArgs(), value = TRUE)
    script_path <- if (length(script_arg)) sub("^--file=", "", script_arg[1]) else NA_character_
    metadata <- list(
        status = if (any(tab$fit_status %in% c("association_error", "fit_error"))) "complete_with_failures" else "complete",
        upstream = list(repository = "LouisFaure/scFates", version = "v0.4.0",
                        commit = "167cc97b3d9b5a4d08c28d27e4630b2d7dc0261f",
                        test_association_blob_sha1 = "fdaa1195c4243bb61b09b263e1ff354d8f67df4d",
                        fit_blob_sha1 = "fffeeb6b5154f2ea470eded4dadf40180f32eb1d",
                        utils_getpath_blob_sha1 = "230935f5ccc5a798d4d4ed627d7767560d71b5e1"),
        adaptation = list(df_extraction = "Named deviance / df.residual; upstream positional index shifted in current mgcv.",
                          optimization = "Response-independent GAM designs reused; all smoothing parameters reestimated per gene.",
                          original_df_formula = "odf = sum(segment df.residual) - number_of_segments",
                          failure_policy = "Errors are explicit; failed significant-gene fitted columns are NA and fit_ok=0.",
                          version_limit = "Original historical mgcv binary is unavailable; current mgcv is recorded."),
        input = list(expression_h5 = inputs$h5, cells_tsv = inputs$cellfile,
                     expression_bytes = unname(file.info(inputs$h5)$size),
                     expression_mtime = as.character(file.info(inputs$h5)$mtime),
                     orientation = "cells_by_genes", expression_access = "Single HDF5 read per submitted gene block; no full expression matrix copied to workers.",
                     expression_md5 = inputs$input_md5[[1]], cells_md5 = inputs$input_md5[[2]]),
        fixed_tree = list(n_cells = inputs$state$ncell, root = FIXED_ROOT, edges = FIXED_EDGES,
                          paths = FIXED_PATHS, segment_counts = as.list(table(inputs$state$seg)),
                          pseudotime = "Imported fixed t; no recomputation or re-rooting."),
        association = list(n_genes = n, n_map = 1L, formula = sprintf("exp ~ s(t, k=%d)", ASSOCIATION_K),
                           spline_df = ASSOCIATION_K,
                           resumed = opt$resume_association, resume_validation = resume_validation,
                           parameter_signature = association_signature(),
                           family = "gaussian(identity)", method = "GCV.Cp", gamma = 1,
                           correction = "bonferroni", family_size = n,
                           adjusted_p_strictly_less_than = BONFERRONI_CUT,
                           amplitude_strictly_greater_than = AMPLITUDE_CUT,
                           stability_strictly_greater_than = STABILITY_CUT,
                           significant_genes = length(selected),
                           failed_genes = sum(tab$fit_status == "association_error")),
        fit = list(formula = "exp ~ s(t, bs='ts')", k = "mgcv default 10",
                   family = "gaussian(identity)", method = "GCV.Cp", gamma = FIT_GAMMA,
                   averaging = "Arithmetic mean of all complete-path predictions covering a cell.",
                   successful_genes = sum(tab$fit_status == "ok"),
                   failed_genes = sum(tab$fit_status == "fit_error"),
                   output = paths[2], dataset = "/fitted", orientation = "cells_by_genes",
                   datatype = "float64", dimensions = c(inputs$state$ncell, length(selected))),
        runtime = list(R = R.version.string, mgcv = as.character(getNamespaceVersion("mgcv")),
                       rhdf5 = as.character(getNamespaceVersion("rhdf5")),
                       workers = opt$workers, seed = RANDOM_SEED,
                       seconds = proc.time()[["elapsed"]] - start_time,
                       script = script_path,
                       script_md5 = if (!is.na(script_path) && file.exists(script_path)) unname(tools::md5sum(script_path)) else NA_character_))
    current_md5 <- as.list(unname(tools::md5sum(c(inputs$h5, inputs$cellfile))))
    if (!identical(current_md5, inputs$input_md5)) stop("Input bytes changed during model fitting.")
    jsonlite::write_json(metadata, paths[3], pretty = TRUE, auto_unbox = TRUE, na = "null", digits = NA)
    cat(sprintf("Completed: %d fitted genes; %d association errors; %d fit errors; %.1f s.\n",
                sum(tab$fit_status == "ok"), sum(tab$fit_status == "association_error"),
                sum(tab$fit_status == "fit_error"), proc.time()[["elapsed"]] - start_time))
    invisible(metadata)
}

if (sys.nframe() == 0L) main()
"""

"""Compact nine-segment result I/O and full rebuild, for direct notebook embedding.

Uses the adapter functions and the notebook's explicit paths, TF snapshot,
segment contracts, display selections and embedded R_SOURCE. Importing this
source performs no analysis and writes no files.
"""

#1. =========== Typed result tables and fixed-tree content fingerprints
import hashlib
import io
import json
import os
from pathlib import Path
import pickle
import subprocess
import tempfile
import warnings

import h5py
import numpy as np
import pandas as pd

RESULT_SCHEMA = 'Trajectory_dot_plot_v2_9segments'
TREE_CORE_SCHEMA = 'CVG-nine-natural-segments-core-v1'


def runtime_file_hash(path, algorithm='sha256'):
    value = hashlib.new(algorithm)
    with Path(path).open('rb') as stream:
        for block in iter(lambda: stream.read(4 * 1024 * 1024), b''):
            value.update(block)
    return value.hexdigest()


def canonical_tree_value(value):
    if isinstance(value, pd.DataFrame):
        return {'kind': 'DataFrame', 'index': canonical_tree_value(value.index),
                'columns': canonical_tree_value(value.columns),
                'series': [canonical_tree_value(value.iloc[:, i]) for i in range(value.shape[1])]}
    if isinstance(value, pd.Series):
        result = {'kind': 'Series', 'name': canonical_tree_value(value.name), 'dtype': str(value.dtype)}
        if isinstance(value.dtype, pd.CategoricalDtype):
            result.update(categories=canonical_tree_value(value.cat.categories),
                          ordered=bool(value.cat.ordered), codes=canonical_tree_value(value.cat.codes.to_numpy()))
        else:
            result['values'] = canonical_tree_value(value.to_numpy())
        return result
    if isinstance(value, pd.Index):
        return {'kind': 'Index', 'name': canonical_tree_value(value.name), 'dtype': str(value.dtype),
                'values': canonical_tree_value(value.to_numpy())}
    if isinstance(value, np.ndarray):
        result = {'kind': 'ndarray', 'dtype': value.dtype.str, 'shape': list(value.shape)}
        if value.dtype.hasobject:
            result['values'] = [canonical_tree_value(v) for v in value.ravel(order='C')]
        else:
            result['bytes_sha256'] = hashlib.sha256(np.ascontiguousarray(value).tobytes()).hexdigest()
        return result
    if isinstance(value, np.generic):
        return {'kind': 'numpy-scalar', 'dtype': value.dtype.str, 'bytes': value.tobytes().hex()}
    if isinstance(value, dict):
        pairs = [[canonical_tree_value(k), canonical_tree_value(v)] for k, v in value.items()]
        pairs.sort(key=lambda item: json.dumps(item[0], sort_keys=True, ensure_ascii=False, separators=(',', ':')))
        return {'kind': 'dict', 'items': pairs}
    if isinstance(value, (list, tuple)):
        return {'kind': type(value).__name__, 'items': [canonical_tree_value(v) for v in value]}
    if isinstance(value, float):
        return {'kind': 'float', 'hex': value.hex()}
    if value is None or isinstance(value, (str, int, bool)):
        return {'kind': type(value).__name__, 'value': value}
    raise TypeError('Unsupported fixed-tree value: ' + type(value).__name__)


def fixed_tree_core_hash(tree):
    projection_keys = ('root', 'edges', 'edge_id', 'alpha', 't', 'root_distance', 'parent', 'residual')
    core = dict(obs=tree['obs'], F=tree['model']['F'], B=tree['model']['B'],
                parameters=tree['model']['parameters'],
                projection={key: tree['projection'][key] for key in projection_keys},
                segments=[{key: segment[key] for key in ('id', 'start', 'end', 'path', 'edge_ids', 'role', 'label')}
                          for segment in tree['segments']],
                natural_branches=[{key: branch[key] for key in ('id', 'start', 'end', 'path', 'edge_ids')}
                                  for branch in tree['publication_display']['branches']],
                local_candidate_calls=tree['local_candidate_calls'],
                projection_component=tree['projection_component'])
    payload = dict(schema=TREE_CORE_SCHEMA, core=canonical_tree_value(core))
    text = json.dumps(payload, sort_keys=True, ensure_ascii=False, separators=(',', ':')).encode('utf-8')
    return hashlib.sha256(text).hexdigest()


def read_fixed_tree():
    with warnings.catch_warnings(), Path(BUNDLE).open('rb') as stream:
        warnings.simplefilter('ignore', DeprecationWarning)
        tree = pickle.load(stream)
    assert len(tree['obs']) == N_CELLS and tree['projection']['root'] == 159
    assert tree['model']['B'].shape == (160, 160)
    return tree


def write_table(group, frame):
    assert frame.columns.is_unique and '_index' not in frame.columns
    group.attrs['columns'] = json.dumps(list(frame.columns))
    group.attrs['index_name'] = frame.index.name or ''
    group.create_dataset('_index', data=frame.index.astype(str).to_numpy(dtype=object), dtype=h5py.string_dtype('utf-8'))
    for column in frame:
        values = frame[column]
        if pd.api.types.is_numeric_dtype(values.dtype) or pd.api.types.is_bool_dtype(values.dtype):
            group.create_dataset(column, data=values.to_numpy(), compression='gzip', shuffle=True)
        else:
            group.create_dataset(column, data=values.fillna('').astype(str).to_numpy(dtype=object),
                                 dtype=h5py.string_dtype('utf-8'), compression='gzip')


def read_table(group):
    columns = json.loads(group.attrs['columns'])
    values = {column: (group[column].asstr()[:] if h5py.check_string_dtype(group[column].dtype)
                       else group[column][:]) for column in columns}
    return pd.DataFrame(values, index=pd.Index(group['_index'].asstr()[:], name=group.attrs['index_name'] or None))


def requested_genes():
    names = set()
    for panel in (SELECTION_H, SELECTION_I):
        assert set(panel) == set(SEGMENTS)
        for segment in SEGMENTS:
            assert set(panel[segment]) == {'up', 'down'}
            names.update(panel[segment]['up'])
            names.update(panel[segment]['down'])
    assert names, 'No displayed genes were requested.'
    return sorted(names)


def read_h5_columns(dataset, columns):
    columns = np.asarray(columns, dtype=np.int64)
    assert columns.ndim == 1 and len(np.unique(columns)) == len(columns)
    assert len(columns) > 0 and columns.min() >= 0 and columns.max() < dataset.shape[1]
    order = np.argsort(columns)
    return dataset[:, columns[order]][:, np.argsort(order)]


#2. =========== Original midpoint groups and all-candidate summaries
def fixed_grouping(cells):
    indices, extremes, counts, midpoints, limits = [], [], [], [], []
    names = []
    for segment in SEGMENTS:
        positions = np.flatnonzero(cells.seg.eq(segment).to_numpy())
        t = cells.iloc[positions].t.to_numpy(dtype=float)
        assert len(positions) == EXPECTED_COUNTS[segment] and np.isfinite(t).all()
        low, high = float(t.min()), float(t.max())
        midpoint = (low + high) / 2
        early, late = positions[t <= midpoint], positions[t > midpoint]
        assert len(early) > 0 and len(late) > 0
        indices.extend([early, late])
        counts.extend([len(early), len(late)])
        extremes.append([int(positions[t.argmin()]), int(positions[t.argmax()])])
        midpoints.append(midpoint)
        limits.append([low, high])
        names.extend([segment + '_early', segment + '_late'])
    return dict(indices=indices, extremes=np.asarray(extremes, dtype=np.int64),
                counts=np.asarray(counts, dtype=np.int64), names=np.asarray(names, dtype=str),
                midpoints=np.asarray(midpoints), limits=np.asarray(limits))


def all_group_means(workdir, expression, gene_symbols, grouping):
    cache_path = Path(workdir) / 'group_means.npz'
    if cache_path.exists():
        # This optional object-array cache is generated by the current local run.
        # It is never required by a fresh rebuild or by the formal result loader.
        with np.load(cache_path, allow_pickle=True) as cache:
            assert np.array_equal(np.asarray(cache['gene_symbols'], dtype=str), gene_symbols)
            assert np.array_equal(cache['extremes'], grouping['extremes'])
            assert np.array_equal(cache['group_counts'], grouping['counts'])
            means = np.asarray(cache['mean'], dtype=np.float64)
        assert means.shape == (N_GENES, 2 * len(SEGMENTS))
        assert np.isfinite(means).all() and np.all(means >= 0)
        # Validate the numeric mapping at deterministic genes before cache reuse.
        check = np.unique(np.linspace(0, N_GENES - 1, 17, dtype=int))
        values = read_h5_columns(expression, check)
        expected = np.asarray([values[group].mean(axis=0) for group in grouping['indices']]).T
        np.testing.assert_allclose(means[check], expected, rtol=0, atol=1e-14)
        return means, 'Current-run axis/endpoint/count-verified group_means.npz'
    means = np.empty((N_GENES, 2 * len(SEGMENTS)), dtype=np.float64)
    for start in range(0, N_GENES, 128):
        stop = min(start + 128, N_GENES)
        block = expression[:, start:stop]
        assert np.isfinite(block).all() and np.all(block >= 0)
        for column, positions in enumerate(grouping['indices']):
            means[start:stop, column] = block[positions].mean(axis=0)
    return means, 'Computed once from measured normalized RNA in gene blocks'


def automatic_orders(symbols, changes):
    up, down = {}, {}
    symbols = np.asarray(symbols, dtype=str)
    for column, segment in enumerate(SEGMENTS):
        delta = changes[:, column]
        positive, negative = np.flatnonzero(delta > 0), np.flatnonzero(delta < 0)
        up[segment] = positive[np.lexsort((symbols[positive], -delta[positive]))]
        down[segment] = negative[np.lexsort((symbols[negative], delta[negative]))]
    return up, down


def validate_association(stats, genes, fitmeta):
    assert len(stats) == len(genes) == N_GENES
    assert np.array_equal(stats.gene.astype(str), genes.gene_symbol.astype(str))
    assert np.array_equal(stats.gene_id.astype(str), genes.gene_id.astype(str))
    assert np.array_equal(stats.gene_index.to_numpy(), np.arange(N_GENES))
    assert pd.api.types.is_bool_dtype(stats.signi)
    numeric = ['p_val', 'p_adj', 'A', 'df_num', 'df_den', 'null_deviance', 'segment_deviance']
    assert np.isfinite(stats[numeric].to_numpy(dtype=float)).all()
    assert stats.p_val.between(0, 1).all() and stats.p_adj.between(0, 1).all()
    assert (stats.A >= 0).all() and (stats.df_num > 0).all() and (stats.df_den > 0).all()
    corrected = np.minimum(stats.p_val.to_numpy() * N_GENES, 1.0)
    np.testing.assert_allclose(stats.p_adj, corrected, rtol=1e-11, atol=1e-300)
    expected = (stats.p_adj < 1e-4) & (stats.A > 1)
    assert np.array_equal(stats.signi.to_numpy(), expected.to_numpy())
    assert np.array_equal(stats.st.to_numpy(), expected.astype(int).to_numpy())
    assert fitmeta['status'] == 'complete'
    a, fit, tree = fitmeta['association'], fitmeta['fit'], fitmeta['fixed_tree']
    assert a['n_genes'] == a['family_size'] == N_GENES and a['n_map'] == 1
    assert a['correction'] == 'bonferroni' and a['spline_df'] == 5
    assert a['adjusted_p_strictly_less_than'] == 1e-4
    assert a['amplitude_strictly_greater_than'] == 1 and a['stability_strictly_greater_than'] == .8
    assert a['significant_genes'] == int(expected.sum()) and a['failed_genes'] == 0
    assert fit['gamma'] == 1.5 and fit['successful_genes'] == int(expected.sum()) and fit['failed_genes'] == 0
    assert tree['n_cells'] == N_CELLS and tree['root'] == 159
    assert {key: tuple(value) for key, value in tree['edges'].items()} == SEGMENT_EDGES
    assert tree['paths'] == FIXED_PATHS and tree['segment_counts'] == EXPECTED_COUNTS
    assert stats.loc[stats.signi, 'fit_status'].eq('ok').all()


def validate_tree_cells(tree, cells):
    assert cells.index.equals(tree['obs'].index) and cells.index.is_unique
    for column in tree['obs']:
        a, b = cells[column], tree['obs'][column]
        if pd.api.types.is_numeric_dtype(b.dtype):
            np.testing.assert_array_equal(a.to_numpy(), b.to_numpy())
        else:
            np.testing.assert_array_equal(a.astype(object).fillna('').astype(str),
                                          b.astype(object).fillna('').astype(str))
    np.testing.assert_array_equal(cells.t.to_numpy(), tree['projection']['t'])
    names = {tuple(pair): name for name, pair in SEGMENT_EDGES.items()}
    edge_names = np.full(len(tree['projection']['edges']), '', dtype=object)
    covered = np.zeros(len(edge_names), dtype=int)
    for branch in tree['publication_display']['branches']:
        name = names[(int(branch['start']), int(branch['end']))]
        edge_names[branch['edge_ids']] = name
        covered[branch['edge_ids']] += 1
    assert np.all(covered == 1)
    np.testing.assert_array_equal(cells.seg.to_numpy(), edge_names[tree['projection']['edge_id']])
    assert cells.seg.value_counts().to_dict() == EXPECTED_COUNTS
    if 'candidate_call' in cells:
        expected = tree['local_candidate_calls'].reindex(cells.index).fillna('Not assessed in local 478')
        np.testing.assert_array_equal(cells.candidate_call.astype(str), expected.astype(str))


def collect_completed(workdir, requested):
    workdir = Path(workdir)
    requested = list(requested)
    assert requested and len(requested) == len(set(requested))
    cells = pd.read_csv(workdir / 'cells.tsv', sep='\t', index_col='cell_id', keep_default_na=False, float_precision='round_trip')
    genes = pd.read_csv(workdir / 'genes.tsv', sep='\t', keep_default_na=False, float_precision='round_trip')
    stats = pd.read_csv(workdir / 'association.tsv', sep='\t', keep_default_na=False, float_precision='round_trip')
    inputmeta = json.loads((workdir / 'input_meta.json').read_text(encoding='utf-8'))
    fitmeta = json.loads((workdir / 'fitmeta.json').read_text(encoding='utf-8'))
    validate_association(stats, genes, fitmeta)
    assert runtime_file_hash(workdir / 'expression.h5', 'md5') == fitmeta['input']['expression_md5']
    assert runtime_file_hash(workdir / 'cells.tsv', 'md5') == fitmeta['input']['cells_md5']
    for filename, key in [('expression.h5', 'expression_sha256'), ('cells.tsv', 'cells_sha256'), ('genes.tsv', 'genes_sha256')]:
        assert runtime_file_hash(workdir / filename) == inputmeta[key]
    tree = read_fixed_tree()
    validate_tree_cells(tree, cells)
    inputmeta['tree_core_schema'] = TREE_CORE_SCHEMA
    inputmeta['tree_core_sha256'] = fixed_tree_core_hash(tree)
    inputmeta['tree_bundle_sha256_at_collection'] = runtime_file_hash(BUNDLE)
    assert inputmeta['tree_bundle_sha256_at_collection'] == inputmeta['tree_bundle_sha256']
    assert np.isfinite(cells.library_total).all() and (cells.library_total >= cells.common_axis_total).all()
    assert (cells.library_total > 0).all() and genes.eligible.all()
    assert set(TF_SET) == set(pd.read_csv(io.BytesIO(TF_BYTES), sep='\t').Symbol.astype(str))
    significant = stats.loc[stats.signi].reset_index(drop=True)
    gene_index = significant.gene_index.to_numpy(dtype=np.int64)
    symbols, ids = significant.gene.to_numpy(dtype=str), significant.gene_id.to_numpy(dtype=str)
    requested_index = pd.Index(genes.gene_symbol).get_indexer(requested)
    fitted_index = pd.Index(symbols).get_indexer(requested)
    assert np.all(requested_index >= 0) and np.all(fitted_index >= 0)
    grouping = fixed_grouping(cells)
    with h5py.File(workdir / 'expression.h5', 'r') as expression, h5py.File(workdir / 'fitted.h5', 'r') as fitted:
        assert np.array_equal(expression['cell_ids'].asstr()[:], cells.index.astype(str))
        assert np.array_equal(fitted['cell_ids'].asstr()[:], cells.index.astype(str))
        assert np.array_equal(expression['gene_symbols'].asstr()[:], genes.gene_symbol.astype(str))
        assert np.array_equal(expression['gene_ids'].asstr()[:], genes.gene_id.astype(str))
        assert np.array_equal(expression['library_total'][:], cells.library_total)
        assert np.array_equal(fitted['gene_symbols'].asstr()[:], symbols)
        assert np.array_equal(fitted['gene_ids'].asstr()[:], ids)
        assert np.array_equal(fitted['gene_index'][:], gene_index) and np.all(fitted['fit_ok'][:] == 1)
        assert expression['expression'].shape == (N_CELLS, N_GENES)
        assert fitted['fitted'].shape == (N_CELLS, len(significant))
        means, mean_source = all_group_means(workdir, expression['expression'], genes.gene_symbol.to_numpy(dtype=str), grouping)
        unique_rows, inverse = np.unique(grouping['extremes'].ravel(), return_inverse=True)
        endpoints = fitted['fitted'][unique_rows, :][inverse].reshape(len(SEGMENTS), 2, len(significant))
        starts, ends = endpoints[:, 0, :].T, endpoints[:, 1, :].T
        amplitude = np.empty(len(significant), dtype=np.float64)
        for start in range(0, len(significant), 128):
            stop = min(start + 128, len(significant))
            block = fitted['fitted'][:, start:stop]
            assert np.isfinite(block).all()
            amplitude[start:stop] = np.ptp(block, axis=0)
        measured_selected = read_h5_columns(expression['expression'], requested_index)
        fitted_selected = read_h5_columns(fitted['fitted'], fitted_index)
    changes = ends - starts
    up, down = automatic_orders(symbols, changes)
    candidate = dict(gene_symbols=symbols, gene_ids=ids, gene_index=gene_index,
        is_tf=np.isin(symbols, list(TF_SET)), fitted_start=starts, fitted_end=ends, fitted_change=changes,
        mean_expression=means[gene_index], fit_amplitude=amplitude,
        segment_order=np.asarray(SEGMENTS, dtype=str), group_names=grouping['names'],
        group_cell_counts=grouping['counts'], extreme_cell_index=grouping['extremes'],
        segment_midpoints=grouping['midpoints'], segment_t_limits=grouping['limits'],
        up_order=up, down_order=down, mean_source=mean_source)
    engine_path = workdir / 'engine.R'
    if engine_path.exists():
        assert runtime_file_hash(engine_path, 'md5') == fitmeta['runtime']['script_md5']
        actual_r = engine_path.read_text(encoding='utf-8').replace('\r\n', '\n')
        assert actual_r == R_SOURCE.replace('\r\n', '\n')
    fitmeta['completed_file_fingerprints'] = {
        filename: runtime_file_hash(workdir / filename)
        for filename in ('expression.h5', 'cells.tsv', 'genes.tsv', 'association.tsv', 'fitted.h5')}
    fitmeta['R_source_sha256_lf'] = hashlib.sha256(R_SOURCE.replace('\r\n', '\n').encode('utf-8')).hexdigest()
    data = dict(cells=cells, genes=genes, stats=stats, requested=requested,
                expression=measured_selected, fitted=fitted_selected,
                inputmeta=inputmeta, fitmeta=fitmeta, candidate=candidate)
    validate_compact_data(data)
    return data


#3. =========== Complete summaries and selected-only per-cell HDF5 storage
def validate_compact_data(data):
    cells, genes, stats, candidate = (data[key] for key in ('cells', 'genes', 'stats', 'candidate'))
    validate_association(stats, genes, data['fitmeta'])
    assert len(cells) == N_CELLS and cells.seg.value_counts().to_dict() == EXPECTED_COUNTS
    requested = list(data['requested'])
    assert len(requested) == len(set(requested)) and len(requested) > 0
    assert data['expression'].shape == data['fitted'].shape == (N_CELLS, len(requested))
    assert np.isfinite(data['expression']).all() and np.all(data['expression'] >= 0)
    assert np.isfinite(data['fitted']).all()
    accepted = stats.loc[stats.signi]
    count = len(accepted)
    assert np.array_equal(candidate['gene_symbols'], accepted.gene.astype(str))
    assert np.array_equal(candidate['gene_ids'], accepted.gene_id.astype(str))
    assert np.array_equal(candidate['gene_index'], accepted.gene_index)
    assert np.array_equal(candidate['is_tf'], np.isin(candidate['gene_symbols'], list(TF_SET)))
    assert set(requested).issubset(set(candidate['gene_symbols']))
    for key in ('fitted_start', 'fitted_end', 'fitted_change'):
        assert candidate[key].shape == (count, len(SEGMENTS)) and np.isfinite(candidate[key]).all()
    np.testing.assert_array_equal(candidate['fitted_change'], candidate['fitted_end'] - candidate['fitted_start'])
    assert candidate['mean_expression'].shape == (count, 2 * len(SEGMENTS))
    assert np.isfinite(candidate['mean_expression']).all() and np.all(candidate['mean_expression'] >= 0)
    assert candidate['fit_amplitude'].shape == (count,) and np.isfinite(candidate['fit_amplitude']).all()
    assert np.all(candidate['fit_amplitude'] >= 0)
    assert np.array_equal(candidate['segment_order'], SEGMENTS)
    grouping = fixed_grouping(cells)
    for key, reference in [('group_names', 'names'), ('group_cell_counts', 'counts'),
                           ('extreme_cell_index', 'extremes'), ('segment_midpoints', 'midpoints'),
                           ('segment_t_limits', 'limits')]:
        np.testing.assert_array_equal(candidate[key], grouping[reference])
    up, down = automatic_orders(candidate['gene_symbols'], candidate['fitted_change'])
    for direction, expected in [('up_order', up), ('down_order', down)]:
        for segment in SEGMENTS:
            np.testing.assert_array_equal(candidate[direction][segment], expected[segment])
    columns = pd.Index(candidate['gene_symbols']).get_indexer(requested)
    for segment_index, (first, last) in enumerate(grouping['extremes']):
        np.testing.assert_array_equal(data['fitted'][first], candidate['fitted_start'][columns, segment_index])
        np.testing.assert_array_equal(data['fitted'][last], candidate['fitted_end'][columns, segment_index])
    np.testing.assert_array_equal(np.ptp(data['fitted'], axis=0), candidate['fit_amplitude'][columns])
    expected_means = np.asarray([data['expression'][group].mean(axis=0) for group in grouping['indices']]).T
    np.testing.assert_allclose(expected_means, candidate['mean_expression'][columns], rtol=0, atol=1e-14)


def write_candidate(group, candidate):
    for key, value in candidate.items():
        if key in ('up_order', 'down_order'):
            child = group.create_group(key)
            for segment in SEGMENTS:
                child.create_dataset(segment, data=np.asarray(value[segment], dtype=np.int64))
        elif isinstance(value, str):
            group.attrs[key] = value
        else:
            value = np.asarray(value)
            if value.dtype.kind in 'OSU':
                group.create_dataset(key, data=value.astype(object), dtype=h5py.string_dtype('utf-8'), compression='gzip')
            else:
                group.create_dataset(key, data=value, compression='gzip', shuffle=True)


def read_candidate(group):
    result = dict(group.attrs)
    for key, value in group.items():
        if isinstance(value, h5py.Group):
            result[key] = {segment: value[segment][:] for segment in SEGMENTS}
        elif h5py.check_string_dtype(value.dtype):
            result[key] = value.asstr()[:]
        else:
            result[key] = value[:]
    return result


def write_results(data):
    validate_compact_data(data)
    tree = read_fixed_tree()
    validate_tree_cells(tree, data['cells'])
    core = fixed_tree_core_hash(tree)
    assert core == data['inputmeta']['tree_core_sha256']
    target, scratch = Path(RESULT).resolve(), Path(SCRATCH).resolve()
    assert target.parent == Path(DATA_DIR).resolve() and target.name.endswith('results-2.h5')
    target.parent.mkdir(parents=True, exist_ok=True)
    scratch.mkdir(parents=True, exist_ok=True)
    # The only persistent output of this function is RESULT; staging stays in SCRATCH.
    with tempfile.TemporaryDirectory(prefix='trajectory-result-', dir=scratch) as staging:
        staging = Path(staging).resolve()
        assert staging.is_relative_to(scratch)
        temporary = staging / target.name
        with h5py.File(temporary, 'w') as store:
            candidate = data['candidate']
            store.attrs.update(schema=RESULT_SCHEMA, cells=N_CELLS, common_genes=N_GENES,
                natural_segments=len(SEGMENTS), root_to_tip_paths=len(FIXED_PATHS),
                stored_fitted_genes=len(data['requested']), significant_genes=len(candidate['gene_symbols']),
                significant_tf_genes=int(candidate['is_tf'].sum()), calculation=DEFINITION,
                tree_sha256=data['inputmeta']['tree_bundle_sha256'], tree_core_schema=TREE_CORE_SCHEMA,
                tree_core_sha256=core, tf_sha256=hashlib.sha256(TF_BYTES).hexdigest())
            for key, frame in [('cells', data['cells']), ('genes', data['genes']), ('association', data['stats'])]:
                write_table(store.create_group(key), frame)
            selected = store.create_group('selected')
            selected.create_dataset('gene_symbols', data=np.asarray(data['requested'], dtype=object), dtype=h5py.string_dtype('utf-8'))
            selected.create_dataset('gene_index', data=pd.Index(data['genes'].gene_symbol).get_indexer(data['requested']))
            for key in ('expression', 'fitted'):
                selected.create_dataset(key, data=data[key], compression='gzip', compression_opts=4,
                    shuffle=True, chunks=(min(512, N_CELLS), min(64, len(data['requested']))))
            selected.create_dataset('fit_ok', data=np.ones(len(data['requested']), dtype='int8'))
            normalization = store.create_group('normalization')
            for key in ('library_total', 'common_axis_total'):
                normalization.create_dataset(key, data=data['cells'][key].to_numpy())
            normalization.attrs['denominator_scope'] = 'Full available source endogenous features; 73 E10 totals embedded in notebook'
            reference = store.create_group('reference')
            reference.create_dataset('tf_snapshot_tsv', data=np.frombuffer(TF_BYTES, dtype='uint8'), compression='gzip')
            write_candidate(store.create_group('candidate'), candidate)
            metadata = store.create_group('meta')
            for key, value in [('input', data['inputmeta']), ('fit', data['fitmeta']), ('plot', data.get('plotmeta', {}))]:
                metadata.create_dataset(key, data=json.dumps(value, ensure_ascii=False), dtype=h5py.string_dtype('utf-8'))
            scope = (f'All {N_GENES} association records and all {len(candidate["gene_symbols"])} significant-candidate summaries; '
                     f'per-cell measured RNA and fitted values for only {len(data["requested"])} displayed unique genes. '
                     'The full significant-gene per-cell fitted matrix is not retained.')
            metadata.create_dataset('storage_scope', data=scope, dtype=h5py.string_dtype('utf-8'))
            store.attrs['complete'] = True
        os.replace(temporary, target)
    return target


def load_results():
    with h5py.File(RESULT, 'r') as store:
        assert store.attrs['schema'] == RESULT_SCHEMA and bool(store.attrs['complete'])
        assert store.attrs['cells'] == N_CELLS and store.attrs['common_genes'] == N_GENES
        assert store.attrs['natural_segments'] == 9 and store.attrs['root_to_tip_paths'] == 5
        assert store.attrs['tree_core_schema'] == TREE_CORE_SCHEMA
        assert store.attrs['tf_sha256'] == hashlib.sha256(TF_BYTES).hexdigest()
        assert store['reference/tf_snapshot_tsv'][:].tobytes() == TF_BYTES
        data = {key: read_table(store[key]) for key in ('cells', 'genes', 'association')}
        data['stats'] = data.pop('association').reset_index(drop=True)
        data['requested'] = list(store['selected/gene_symbols'].asstr()[:])
        expected_indices = pd.Index(data['genes'].gene_symbol).get_indexer(data['requested'])
        np.testing.assert_array_equal(store['selected/gene_index'][:], expected_indices)
        data['expression'], data['fitted'] = store['selected/expression'][:], store['selected/fitted'][:]
        assert np.all(store['selected/fit_ok'][:] == 1)
        for key in ('library_total', 'common_axis_total'):
            np.testing.assert_array_equal(store['normalization/' + key][:], data['cells'][key])
        for key in ('input', 'fit', 'plot'):
            data[key + 'meta'] = json.loads(store['meta/' + key].asstr()[()])
        data['candidate'] = read_candidate(store['candidate'])
        assert store.attrs['stored_fitted_genes'] == len(data['requested'])
        assert store.attrs['significant_genes'] == len(data['candidate']['gene_symbols'])
        assert store.attrs['significant_tf_genes'] == int(data['candidate']['is_tf'].sum())
        assert store.attrs['tree_core_sha256'] == data['inputmeta']['tree_core_sha256']
        stored_core, stored_container = str(store.attrs['tree_core_sha256']), str(store.attrs['tree_sha256'])
    validate_compact_data(data)
    tree = read_fixed_tree()
    assert fixed_tree_core_hash(tree) == stored_core, 'Accepted model, projection, annotations or natural branches changed.'
    validate_tree_cells(tree, data['cells'])
    assert data['inputmeta']['tree_core_schema'] == TREE_CORE_SCHEMA
    assert sequence_hash(data['cells'].index) == data['inputmeta']['cell_ids_sha256']
    assert sequence_hash(data['genes'].gene_id) == data['inputmeta']['gene_ids_sha256']
    assert sequence_hash(data['genes'].gene_symbol) == data['inputmeta']['gene_symbols_sha256']
    assert data['inputmeta']['source_h5ad_sha256'] == RNA_SHA256
    expected_r_hash = hashlib.sha256(R_SOURCE.replace('\r\n', '\n').encode('utf-8')).hexdigest()
    assert data['fitmeta']['R_source_sha256_lf'] == expected_r_hash, 'Embedded R engine changed; rebuild explicitly.'
    assert np.isfinite(data['cells'].library_total).all() and (data['cells'].library_total > 0).all()
    assert (data['cells'].library_total >= data['cells'].common_axis_total).all()
    if runtime_file_hash(BUNDLE) != stored_container:
        print('Tree container metadata changed; the explicitly frozen analysis core remains identical.')
    missing = set(requested_genes()) - set(data['requested'])
    assert not missing, 'Displayed genes missing from compact per-cell cache; use REBUILD_ANALYSIS=True: ' + ', '.join(sorted(missing))
    return data


#4. =========== Fresh full normalization, association and five-path GAM rebuild
def rebuild_analysis():
    scratch = Path(SCRATCH).resolve()
    scratch.mkdir(parents=True, exist_ok=True)
    assert Path(R_SCRIPT).exists(), 'Set R_SCRIPT to an installed Rscript executable.'
    with tempfile.TemporaryDirectory(prefix='trajectory-dot-plot-2-', dir=scratch) as directory:
        workdir = Path(directory).resolve()
        assert workdir.is_relative_to(scratch)
        prepare_analysis_inputs(workdir, project_root=ROOT)
        runtime = workdir / 'r-runtime'
        runtime.mkdir()
        engine = workdir / 'engine.R'
        engine.write_text(R_SOURCE, encoding='utf-8')
        environment = os.environ.copy()
        environment.update(TMP=str(runtime), TEMP=str(runtime), LC_ALL='C',
                           OPENBLAS_NUM_THREADS='1', OMP_NUM_THREADS='1', MKL_NUM_THREADS='1')
        command = [str(R_SCRIPT), '--vanilla', str(engine), '--workdir', str(workdir), '--workers', str(WORKERS)]
        subprocess.run(command, check=True, cwd=ROOT, env=environment)
        data = collect_completed(workdir, requested_genes())
        write_results(data)
    return data
